In [1]:
import pandas as pd
import pickle
import numpy as np

import json

feature_list_path = r'D:\home loan credit risk\artifact\binning\prebin\selected_features.json'

with open(feature_list_path, "r") as f:
    features = json.load(f)
    
# cat binning df
cat_df = pd.read_csv(r'D:\home loan credit risk\artifact\binning\prebin\cat_bin_df.csv')

In [2]:
import gc
# numerical feature bins
with open(r'D:\home loan credit risk\artifact\binning\prebin\numerical_feature_bins.pkl', 'rb') as f:
   feature_bins_num = pickle.load(f)


In [3]:
num_bins = {}
for key,value in feature_bins_num.items():
    if key in features:
        num_bins[key] = value
del feature_bins_num
gc.collect()

0

In [4]:
iv_df = pd.read_csv(r'D:\home loan credit risk\artifact\binning\prebin\iv_df.csv')

# CODE TO MERGE BINS NUMERICAL 

In [ ]:
import numpy as np
import pandas as pd
def merge_bins_and_create_mapping(
    feature: str,
    bins_index: list,
    num_bins: dict
):

    bins_index = sorted(bins_index)
    df = num_bins[feature]
    df_calc = df.copy()
    bin_filter = df_calc.index.isin(bins_index)

    count_bin = df_calc.loc[bin_filter, 'Count'].sum()
    count_non_event = df_calc.loc[bin_filter, 'Non-event'].sum()
    count_event = df_calc.loc[bin_filter, 'Event'].sum()

    total_event = df_calc['Event'].sum()
    total_non_event = df_calc['Non-event'].sum()

    dist_non_event = count_non_event / total_non_event
    dist_event = count_event / total_event

    merged_woe = round(np.log(dist_non_event / dist_event), 6)
    
    old_woe_values = (
        pd.to_numeric(df_calc.loc[bin_filter, 'WoE'], errors='coerce')
        .round(6)
        .dropna()
        .unique()
        .tolist()
    )

    mapping = {old: merged_woe for old in old_woe_values}

    lower = df.loc[bins_index[0], 'Bin'].split(',')[0].replace('(', '')
    upper = df.loc[bins_index[-1], 'Bin'].split(',')[1].replace(')', '').replace(']', '').strip()
    new_bin = f'({lower}, {upper}]'

    df.drop(index=bins_index[1:], inplace=True)

    total_count = df_calc.loc[df_calc.index != 'Totals', 'Count'].sum()
    count_pct = count_bin / total_count

    updates = {
        'Bin': new_bin,
        'Count': count_bin,
        'Count (%)': count_pct,
        'Non-event': count_non_event,
        'Event': count_event,
        'Event rate': count_event / count_bin,
        'WoE': merged_woe,
        'feature': feature
    }

    keep_idx = bins_index[0]
    df.loc[keep_idx, updates.keys()] = updates.values()

    return mapping
def run_merge_plan_and_collect_mappings(
    feature: str,
    merge_plan: list,
    num_bins: dict
):
    """
    Returns:
    woe_mapping = {feature: {old_woe: new_woe}}
    """

    feature_mapping = {}

    for bins_index in merge_plan:
        mapping = merge_bins_and_create_mapping(
            feature=feature,
            bins_index=bins_index,
            num_bins=num_bins
        )
        feature_mapping.update(mapping)

    # --- recompute IV after all merges ---
    df = num_bins[feature].copy()   
    df_valid = df[df.index != 'Totals'].copy()

    total_event = df_valid['Event'].sum()
    total_non_event = df_valid['Non-event'].sum()

    df_valid['dist_event'] = df_valid['Event'] / total_event
    df_valid['dist_non_event'] = df_valid['Non-event'] / total_non_event

    df_valid['IV'] = (
        (df_valid['dist_non_event'] - df_valid['dist_event']) *
        df_valid['WoE']
    )

    df.loc[df_valid.index, 'IV'] = df_valid['IV'].values  
    # write back clean dataframe
    num_bins[feature] = df

    return {feature : feature_mapping}

## MANUAL BINNING NUMERICAL


In [6]:
rows = []
# the feature are seleceted as drop those woe mappings isnt updated
all_woe_mapping_num = {}

##### EXT_SOURCE_3

In [10]:
temp = num_bins['EXT_SOURCE_3']
temp[['feature','Bin','WoE','Event rate','Count (%)','IV']]

,feature,Bin,WoE,Event rate,Count (%),IV
0,EXT_SOURCE_3,"(-0.000473, 0.16]",-1.214084,0.228219,0.042783,0.103231
1,EXT_SOURCE_3,"(0.16, 0.234]",-0.802008,0.163765,0.041799,0.037510
2,EXT_SOURCE_3,"(0.234, 0.291]",-0.562067,0.133494,0.042234,0.016878
3,EXT_SOURCE_3,"(0.291, 0.339]",-0.401090,0.115946,0.042596,0.008108
4,EXT_SOURCE_3,"(0.339, 0.381]",-0.202126,0.097057,0.042133,0.001874
5,EXT_SOURCE_3,"(0.381, 0.419]",-0.151927,0.092746,0.041637,0.001024
6,EXT_SOURCE_3,"(0.419, 0.454]",0.015118,0.079614,0.042990,0.000010
7,EXT_SOURCE_3,"(0.454, 0.488]",0.111069,0.072861,0.042568,0.000501
8,EXT_SOURCE_3,"(0.488, 0.52]",0.267028,0.063002,0.040970,0.002613
9,EXT_SOURCE_3,"(0.52, 0.55]",0.340922,0.058778,0.042393,0.004275


In [8]:
merge_plan = [
    [0,1,2],           
    [3, 4,5], 
    [6,7, 8],  
    [9, 10, 11],
    [12,13, 14,],
    [15, 16,17,18]
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="EXT_SOURCE_3",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [9]:
all_woe_mapping_num.update(woe_mappings)

In [10]:
num_bins['EXT_SOURCE_3']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,EXT_SOURCE_3,"(-0.000473, 0.291]]",31198,5473,25725,0.126817,0.175428,-0.884846,0.143191
3,EXT_SOURCE_3,"(0.291, 0.419]]",31087,3171,27916,0.126366,0.102004,-0.257329,0.009322
6,EXT_SOURCE_3,"(0.419, 0.52]]",31127,2240,28887,0.126528,0.071963,0.124434,0.001860
9,EXT_SOURCE_3,"(0.52, 0.608]]",31502,1704,29798,0.128053,0.054092,0.428981,0.019717
12,EXT_SOURCE_3,"(0.608, 0.688]]",30803,1331,29472,0.125211,0.043210,0.665028,0.042098
15,EXT_SOURCE_3,"(0.688, 0.896]]",41486,1413,40073,0.168637,0.034060,0.912506,0.096771
19,EXT_SOURCE_3,SPECIAL_-99999,48805,4528,44277,0.198388,0.092777,-0.152297,0.004905
Totals,EXT_SOURCE_3,None,246008,19860,226148,1.000000,0.080729,NaN,0.336897


In [11]:
rows.append({
    "feature":"EXT_SOURCE_3",
    "n_bins":6,
    "n_special_bins":1,
    "special_bins_pct":0.198388	,
    "bad_rate_trend":"decreasing",
    "iv":0.336897,
    "keep":"Yes",
    "comment":"clear decreasing monotonic trend",
})

##### REMOVED EXT_SOURCE_2

In [12]:
''' merge_plan = [
    [0,1,2],        
    [3,4,5],        
    [6,7,8,9],      
    [10,11],        
    [12,13,14],     
    [15,16,17,18,19]
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="EXT_SOURCE_2",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature':'EXT_SOURCE_2',
    'n_bins':6,
    'n_special_bins':1,
    'special_bins_pct':0.002158,
    'bad_rate_trend':'decreasing',
    'iv':0.315593,
    'keep':'Yes',
    'comment':'clear monotonic decreasing bad rate trend',
})'''

' merge_plan = [\n    [0,1,2],        \n    [3,4,5],        \n    [6,7,8,9],      \n    [10,11],        \n    [12,13,14],     \n    [15,16,17,18,19]\n]\nwoe_mappings = run_merge_plan_and_collect_mappings(\n    feature="EXT_SOURCE_2",\n    merge_plan=merge_plan,\n    num_bins=num_bins\n)\nall_woe_mapping_num.update(woe_mappings)\nrows.append({\n    \'feature\':\'EXT_SOURCE_2\',\n    \'n_bins\':6,\n    \'n_special_bins\':1,\n    \'special_bins_pct\':0.002158,\n    \'bad_rate_trend\':\'decreasing\',\n    \'iv\':0.315593,\n    \'keep\':\'Yes\',\n    \'comment\':\'clear monotonic decreasing bad rate trend\',\n})'

#####  EXT_SOURCE_1

In [13]:
num_bins['EXT_SOURCE_1']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,EXT_SOURCE_1,"(0.013600000000000001, 0.16]",5654,1152,4502,0.022983,0.203750,-1.069460,0.040745
1,EXT_SOURCE_1,"(0.16, 0.217]",5653,790,4863,0.022979,0.139749,-0.615104,0.011241
2,EXT_SOURCE_1,"(0.217, 0.262]",5653,703,4950,0.022979,0.124359,-0.480696,0.006494
3,EXT_SOURCE_1,"(0.262, 0.304]",5655,592,5063,0.022987,0.104686,-0.286274,0.002124
4,EXT_SOURCE_1,"(0.304, 0.342]",5652,525,5127,0.022975,0.092887,-0.153604,0.000578
5,EXT_SOURCE_1,"(0.342, 0.379]",5653,494,5159,0.022979,0.087387,-0.086519,0.000178
6,EXT_SOURCE_1,"(0.379, 0.415]",5653,507,5146,0.022979,0.089687,-0.115018,0.000319
7,EXT_SOURCE_1,"(0.415, 0.451]",5654,407,5247,0.022983,0.071984,0.124117,0.000336
8,EXT_SOURCE_1,"(0.451, 0.487]",5653,374,5279,0.022979,0.066160,0.214754,0.000969
9,EXT_SOURCE_1,"(0.487, 0.523]",5655,372,5283,0.022987,0.065782,0.220874,0.001023


In [14]:
merge_plan = [
    [0, 1, 2],       
    [3, 4, 5],       
    [6, 7, 8],       
    [9, 10, 11],     
    [12, 13, 14],
    [15, 16,17,18],  
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="EXT_SOURCE_1",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature':'EXT_SOURCE_1',
    'n_bins':6,
    'n_special_bins':1,
    'special_bins_pct':0.563376,
    'bad_rate_trend':'decreasing',
    'iv':0.156080,
    'keep':'Yes',
    'comment':'clear monotonic trend decreasing bad rate',
})

In [15]:
num_bins['EXT_SOURCE_1']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,EXT_SOURCE_1,"(0.013600000000000001, 0.262]]",16960,2645,14315,0.068941,0.155955,-0.743845,0.051982
3,EXT_SOURCE_1,"(0.262, 0.379]]",16960,1611,15349,0.068941,0.094988,-0.178287,0.002362
6,EXT_SOURCE_1,"(0.379, 0.487]]",16960,1288,15672,0.068941,0.075943,0.066303,0.000295
9,EXT_SOURCE_1,"(0.487, 0.594]]",16960,1007,15953,0.068941,0.059375,0.330189,0.006550
12,EXT_SOURCE_1,"(0.594, 0.703]]",16959,809,16150,0.068937,0.047703,0.561394,0.017223
15,EXT_SOURCE_1,"(0.703, 0.952]]",22614,685,21929,0.091924,0.030291,1.033664,0.064579
19,EXT_SOURCE_1,SPECIAL_-99999,138595,11815,126780,0.563376,0.085248,-0.059399,0.002038
Totals,EXT_SOURCE_1,None,246008,19860,226148,1.000000,0.080729,NaN,0.156080


#####  ANNUITY_CREDIT_RATIO

In [16]:
num_bins['ANNUITY_CREDIT_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,ANNUITY_CREDIT_RATIO,"(0.0211, 0.0292]",12950,702,12248,0.052641,0.054208,0.426703,0.008027
1,ANNUITY_CREDIT_RATIO,"(0.0292, 0.0294]",13012,829,12183,0.052893,0.063710,0.255095,0.003094
2,ANNUITY_CREDIT_RATIO,"(0.0294, 0.0323]",13140,784,12356,0.053413,0.059665,0.325006,0.004927
3,ANNUITY_CREDIT_RATIO,"(0.0323, 0.0342]",12692,960,11732,0.051592,0.075638,0.070660,0.000250
4,ANNUITY_CREDIT_RATIO,"(0.0342, 0.0379]",12997,937,12060,0.052832,0.072094,0.122484,0.000753
5,ANNUITY_CREDIT_RATIO,"(0.0379, 0.0398]",12987,902,12085,0.052791,0.069454,0.162624,0.001304
6,ANNUITY_CREDIT_RATIO,"(0.0398, 0.0426]",12923,705,12218,0.052531,0.054554,0.419986,0.007782
7,ANNUITY_CREDIT_RATIO,"(0.0426, 0.047]",12886,1148,11738,0.052380,0.089089,-0.107672,0.000635
8,ANNUITY_CREDIT_RATIO,"(0.047, 0.0489]",13216,1680,11536,0.053722,0.127119,-0.505803,0.016986
9,ANNUITY_CREDIT_RATIO,"(0.0489, 0.05]",27146,2008,25138,0.110346,0.073970,0.094759,0.000952


In [17]:
merge_plan = [
    [0,1,2],           
    [3,4,5,6,7,8,9],
    [10,11,12,13,14,15,16,17]          
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="ANNUITY_CREDIT_RATIO",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature':'ANNUITY_CREDIT_RATIO',
    'n_bins':3,
    'n_special_bins':1,
    'special_bins_pct':0.000041,
    'bad_rate_trend':'increasing',
    'iv':0.145948,
    'keep':'Yes',
    'comment':'clear monotonic increasing trend feature special values SPECIAL_-88888',
})

In [18]:
num_bins['ANNUITY_CREDIT_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,ANNUITY_CREDIT_RATIO,"(0.0211, 0.0323]]",39102,2315,36787,0.158946,0.059204,0.333253,0.015364
3,ANNUITY_CREDIT_RATIO,"(0.0323, 0.05]]",104847,8340,96507,0.426193,0.079544,0.016070,0.000109
10,ANNUITY_CREDIT_RATIO,"(0.05, 0.124]]",102049,9205,92844,0.414820,0.090202,-0.121308,0.006423
18,ANNUITY_CREDIT_RATIO,SPECIAL_-88888,10,0,10,0.000041,0.000000,0.000000,0.000000
Totals,ANNUITY_CREDIT_RATIO,None,246008,19860,226148,1.000000,0.080729,NaN,0.145948


##### YEARS_EMPLOYED

In [19]:
num_bins['YEARS_EMPLOYED']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,YEARS_EMPLOYED,"(-0.001, 0.56]",10107,1081,9026,0.041084,0.106956,-0.310259,0.004505
1,YEARS_EMPLOYED,"(0.56, 0.91]",10106,1151,8955,0.041080,0.113893,-0.380901,0.006992
2,YEARS_EMPLOYED,"(0.91, 1.28]",10281,1185,9096,0.041791,0.115261,-0.394390,0.007669
3,YEARS_EMPLOYED,"(1.28, 1.69]",10069,1171,8898,0.040930,0.116298,-0.404514,0.007935
4,YEARS_EMPLOYED,"(1.69, 2.1]",10057,1114,8943,0.040881,0.110769,-0.349568,0.005785
5,YEARS_EMPLOYED,"(2.1, 2.52]",10182,1119,9063,0.041389,0.109900,-0.340717,0.005543
6,YEARS_EMPLOYED,"(2.52, 2.96]",10087,1024,9063,0.041003,0.101517,-0.251998,0.002894
7,YEARS_EMPLOYED,"(2.96, 3.42]",9922,1024,8898,0.040332,0.103205,-0.270372,0.003303
8,YEARS_EMPLOYED,"(3.42, 3.95]",10070,1012,9058,0.040934,0.100497,-0.240762,0.002625
9,YEARS_EMPLOYED,"(3.95, 4.52]",10245,912,9333,0.041645,0.089019,-0.106810,0.000497


In [20]:
merge_plan = [
    [0, 1, 2, 3, 4],  
    [5, 6, 7, 8,9,10],
    [11, 12], 
    [13, 14, 15, 16],
    [17, 18, 19],    
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="YEARS_EMPLOYED",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [21]:
all_woe_mapping_num.update(woe_mappings)

In [22]:
num_bins['YEARS_EMPLOYED']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,YEARS_EMPLOYED,"(-0.001, 2.1]]",50620,5702,44918,0.205766,0.112643,-0.368460,0.032604
5,YEARS_EMPLOYED,"(2.1, 5.12]]",60430,5986,54444,0.245642,0.099057,-0.224733,0.013633
11,YEARS_EMPLOYED,"(5.12, 6.73]]",20202,1609,18593,0.082119,0.079646,0.014690,0.000018
13,YEARS_EMPLOYED,"(6.73, 12.0]]",40347,2730,37617,0.164007,0.067663,0.190672,0.005506
17,YEARS_EMPLOYED,"(12.0, 49.07]]",30266,1463,28803,0.123029,0.048338,0.547508,0.029400
20,YEARS_EMPLOYED,SPECIAL_-99999,44143,2370,41773,0.179437,0.053689,0.436878,0.028563
Totals,YEARS_EMPLOYED,None,246008,19860,226148,1.000000,0.080729,NaN,0.114435


In [23]:
rows.append({
    'feature':'YEARS_EMPLOYED',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.179437,
    'bad_rate_trend':'decreasing',
    'iv':0.114435,
    'keep':'Yes',
    'comment':'clear monotonic trend SC_-99999 special values',
})

##### AMT_GOODS_PRICE


In [24]:
num_bins["AMT_GOODS_PRICE"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,AMT_GOODS_PRICE,"(40499.999, 135000.0]",17422,1124,16298,0.070819,0.064516,0.241667,3.738988e-03
1,AMT_GOODS_PRICE,"(135000.0, 180000.0]",14158,1198,12960,0.057551,0.084616,-0.051268,1.545546e-04
2,AMT_GOODS_PRICE,"(180000.0, 220500.0]",5346,430,4916,0.021731,0.080434,0.003983,3.442180e-07
3,AMT_GOODS_PRICE,"(220500.0, 225000.0]",20218,1797,18421,0.082184,0.088881,-0.105109,9.489136e-04
4,AMT_GOODS_PRICE,"(225000.0, 238500.0]",6748,529,6219,0.027430,0.078394,0.031894,2.753201e-05
5,AMT_GOODS_PRICE,"(238500.0, 270000.0]",15681,1234,14447,0.063742,0.078694,0.027744,4.849625e-05
6,AMT_GOODS_PRICE,"(270000.0, 315000.0]",9094,861,8233,0.036966,0.094678,-0.174671,1.213632e-03
7,AMT_GOODS_PRICE,"(315000.0, 378000.0]",9801,1066,8735,0.039840,0.108764,-0.329057,4.952505e-03
8,AMT_GOODS_PRICE,"(378000.0, 450000.0]",27768,3687,24081,0.112874,0.132779,-0.555872,4.400626e-02
9,AMT_GOODS_PRICE,"(450000.0, 463500.0]",11334,668,10666,0.046072,0.058938,0.338046,4.573210e-03


In [25]:
merge_plan = [

    [0,1,2,3,4,5,6,7,8],
    [9,10,11,12],
    [13,14,15],
    [16,17,18],      # 819k – inf

]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="AMT_GOODS_PRICE",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [26]:
all_woe_mapping_num.update(woe_mappings)

In [27]:
num_bins["AMT_GOODS_PRICE"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,AMT_GOODS_PRICE,"(40499.999, 450000.0]]",126236,11926,114310,0.513138,0.094474,-0.172289,0.016374
9,AMT_GOODS_PRICE,"(450000.0, 675000.0]]",55194,4265,50929,0.224359,0.077273,0.047508,0.000496
13,AMT_GOODS_PRICE,"(675000.0, 900000.0]]",30764,1934,28830,0.125053,0.062866,0.269344,0.008108
16,AMT_GOODS_PRICE,"(900000.0, 4050000.0]]",33593,1718,31875,0.136552,0.051142,0.488179,0.026577
19,AMT_GOODS_PRICE,SPECIAL_-99999,221,17,204,0.000898,0.076923,0.052425,0.000002
Totals,AMT_GOODS_PRICE,None,246008,19860,226148,1.000000,0.080729,NaN,0.102669


In [28]:
rows.append({
    'feature':'AMT_GOODS_PRICE',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.000898,
    'bad_rate_trend':'decreasing',
    'iv':0.102669,
    'keep':'Yes',
    'comment':'only clear monotonic trend',
})

##### IP_WORST_DPD_720D

In [29]:
num_bins["IP_WORST_DPD_720D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_WORST_DPD_720D,"(-0.001, 1.0]",52171,3948,48223,0.212070,0.075674,0.070145,0.001013
1,IP_WORST_DPD_720D,"(1.0, 2.0]",10001,897,9104,0.040653,0.089691,-0.115069,0.000565
2,IP_WORST_DPD_720D,"(2.0, 3.0]",8096,784,7312,0.032909,0.096838,-0.199619,0.001426
3,IP_WORST_DPD_720D,"(3.0, 4.0]",7493,733,6760,0.030458,0.097825,-0.210850,0.001479
4,IP_WORST_DPD_720D,"(4.0, 5.0]",5950,607,5343,0.024186,0.102017,-0.257468,0.001786
5,IP_WORST_DPD_720D,"(5.0, 6.0]",5112,550,4562,0.020780,0.107590,-0.316884,0.002383
6,IP_WORST_DPD_720D,"(6.0, 7.0]",4000,454,3546,0.016260,0.113500,-0.377004,0.002707
7,IP_WORST_DPD_720D,"(7.0, 8.0]",3084,385,2699,0.012536,0.124838,-0.485089,0.003614
8,IP_WORST_DPD_720D,"(8.0, 11.0]",5844,827,5017,0.023755,0.141513,-0.629699,0.012252
9,IP_WORST_DPD_720D,"(11.0, 20.0]",4792,775,4017,0.019479,0.161728,-0.787054,0.016733


In [30]:
merge_plan = [
    [0],        
    [1,2],    
    [3,4,5,6],
    [7,8,9],     
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_WORST_DPD_720D",
    merge_plan=merge_plan,
    num_bins=num_bins
)


In [31]:
all_woe_mapping_num.update(woe_mappings)

In [32]:
num_bins["IP_WORST_DPD_720D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_WORST_DPD_720D,"(-0.001, 1.0]]",52171,3948,48223,0.212070,0.075674,0.070145,0.001013
1,IP_WORST_DPD_720D,"(1.0, 3.0]]",18097,1681,16416,0.073563,0.092888,-0.153614,0.001851
3,IP_WORST_DPD_720D,"(3.0, 7.0]]",22555,2344,20211,0.091684,0.103924,-0.278114,0.007969
7,IP_WORST_DPD_720D,"(7.0, 20.0]]",13720,1987,11733,0.055771,0.144825,-0.656703,0.031632
10,IP_WORST_DPD_720D,"(20.0, 705.0]",5479,968,4511,0.022272,0.176675,-0.893440,0.025726
11,IP_WORST_DPD_720D,SPECIAL_-66666,46999,3207,43792,0.191047,0.068235,0.181633,0.005842
12,IP_WORST_DPD_720D,SPECIAL_-99999,86987,5725,81262,0.353594,0.065814,0.220354,0.015659
Totals,IP_WORST_DPD_720D,None,246008,19860,226148,1.000000,0.080729,NaN,0.091186


In [33]:
rows.append({
    'feature':'IP_WORST_DPD_720D',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.544641,
    'bad_rate_trend':'increasing',
    'iv':0.091014,
    'keep':'Yes',
    'comment':'clear monotonic trend',
})

##### YEARS_AGE

In [34]:
num_bins["YEARS_AGE"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,YEARS_AGE,"(20.519, 25.95]",12959,1542,11417,0.052677,0.118991,-0.430459,0.011691
1,YEARS_AGE,"(25.95, 28.35]",12980,1478,11502,0.052763,0.113867,-0.380651,0.008968
2,YEARS_AGE,"(28.35, 30.44]",12925,1438,11487,0.052539,0.111257,-0.354519,0.007662
3,YEARS_AGE,"(30.44, 32.42]",12954,1368,11586,0.052657,0.105604,-0.296034,0.005225
4,YEARS_AGE,"(32.42, 34.51]",12939,1309,11630,0.052596,0.101167,-0.248158,0.003595
5,YEARS_AGE,"(34.51, 36.56]",12986,1250,11736,0.052787,0.096258,-0.192965,0.002131
6,YEARS_AGE,"(36.56, 38.41]",12945,1159,11786,0.052620,0.089533,-0.113127,0.000706
7,YEARS_AGE,"(38.41, 40.24]",12910,1056,11854,0.052478,0.081797,-0.014305,0.000011
8,YEARS_AGE,"(40.24, 42.14]",12963,1065,11898,0.052693,0.082157,-0.019086,0.000019
9,YEARS_AGE,"(42.14, 44.1]",12973,987,11986,0.052734,0.076081,0.064343,0.000213


In [35]:
merge_plan = [
    [0,1,2],
    [3,4,5],
    [6,7,8],
    [9,10,11],
    [12,13,14],
    [15,16,17],
    [18]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="YEARS_AGE",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [36]:
all_woe_mapping_num.update(woe_mappings)

In [37]:
num_bins["YEARS_AGE"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,YEARS_AGE,"(20.519, 30.44]]",38864,4458,34406,0.157979,0.114708,-0.388951,0.028134
3,YEARS_AGE,"(30.44, 36.56]]",38879,3927,34952,0.158040,0.101006,-0.246382,0.010639
6,YEARS_AGE,"(36.56, 42.14]]",38818,3280,35538,0.157792,0.084497,-0.049723,0.000398
9,YEARS_AGE,"(42.14, 48.56]]",38822,2914,35908,0.157808,0.075061,0.078951,0.000952
12,YEARS_AGE,"(48.56, 55.58]]",38879,2604,36275,0.158040,0.066977,0.201598,0.005904
15,YEARS_AGE,"(55.58, 63.4]]",38804,2107,36697,0.157735,0.054299,0.424948,0.023872
18,YEARS_AGE,"(63.4, 69.12]]",12942,570,12372,0.052608,0.044043,0.645073,0.016776
Totals,YEARS_AGE,None,246008,19860,226148,1.000000,0.080729,NaN,0.088899


In [38]:
rows.append({
    'feature':'YEARS_AGE',
    'n_bins':7,
    'n_special_bins':0,
    'special_bins_pct':0.0,
    'bad_rate_trend':'decreasing',
    'iv':0.088899,
    'keep':'Yes',
    'comment':'clear monotonic trend;'
})

##### CB_MAX_MONTHLY_UTIL_6M

In [39]:
num_bins["CB_MAX_MONTHLY_UTIL_6M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_MONTHLY_UTIL_6M,"(-0.001, 0.0819]",28689,1615,27074,0.116618,0.056293,0.386757,0.014851
1,CB_MAX_MONTHLY_UTIL_6M,"(0.0819, 0.424]",2869,175,2694,0.011662,0.060997,0.301514,0.000935
2,CB_MAX_MONTHLY_UTIL_6M,"(0.424, 0.712]",2869,155,2714,0.011662,0.054026,0.430272,0.001806
3,CB_MAX_MONTHLY_UTIL_6M,"(0.712, 0.904]",2868,243,2625,0.011658,0.084728,-0.052707,0.000033
4,CB_MAX_MONTHLY_UTIL_6M,"(0.904, 0.97]",2869,305,2564,0.011662,0.106309,-0.303470,0.001220
5,CB_MAX_MONTHLY_UTIL_6M,"(0.97, 1.004]",2869,375,2494,0.011662,0.130708,-0.537765,0.004224
6,CB_MAX_MONTHLY_UTIL_6M,"(1.004, 1.025]",2869,443,2426,0.011662,0.154409,-0.732053,0.008476
7,CB_MAX_MONTHLY_UTIL_6M,"(1.025, 1.041]",2869,453,2416,0.011662,0.157895,-0.758506,0.009198
8,CB_MAX_MONTHLY_UTIL_6M,"(1.041, 1.058]",2869,538,2331,0.011662,0.187522,-0.966288,0.016216
9,CB_MAX_MONTHLY_UTIL_6M,"(1.058, 3.927]",2869,665,2204,0.011662,0.231788,-1.234240,0.029299


In [40]:
merge_plan = [

    [0],
    [1,2,3,4,5,6,7,8,9],          
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_MAX_MONTHLY_UTIL_6M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [41]:
all_woe_mapping_num.update(woe_mappings)

In [42]:
num_bins["CB_MAX_MONTHLY_UTIL_6M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_MONTHLY_UTIL_6M,"(-0.001, 0.0819]]",28689,1615,27074,0.116618,0.056293,0.386757,0.014851
1,CB_MAX_MONTHLY_UTIL_6M,"(0.0819, 3.927]]",25820,3352,22468,0.104956,0.129822,-0.529947,0.036795
10,CB_MAX_MONTHLY_UTIL_6M,SPECIAL_-99999,191499,14893,176606,0.778426,0.077771,0.040548,0.001258
Totals,CB_MAX_MONTHLY_UTIL_6M,None,246008,19860,226148,1.000000,0.080729,NaN,0.087516


In [43]:
rows.append({
    'feature':'CB_MAX_MONTHLY_UTIL_6M',
    'n_bins':2,
    'n_special_bins':2,
    'special_bins_pct':0.7784,
    'bad_rate_trend':'increasing',
    'iv':0.088130,
    'keep':'Review',
    'comment':'high missingness, remaining population split into two monotonic utilization bins'
})

##### GOODS_CREDIT_RATIO

In [44]:
num_bins["GOODS_CREDIT_RATIO"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,GOODS_CREDIT_RATIO,"(0.166, 0.716]",14635,1937,12698,0.059490,0.132354,-0.552178,2.285114e-02
1,GOODS_CREDIT_RATIO,"(0.716, 0.783]",10187,1168,9019,0.041409,0.114656,-0.388441,7.353475e-03
2,GOODS_CREDIT_RATIO,"(0.783, 0.808]",14088,1385,12703,0.057266,0.098311,-0.216344,2.935136e-03
3,GOODS_CREDIT_RATIO,"(0.808, 0.826]",11163,1357,9806,0.045377,0.121562,-0.454764,1.135423e-02
4,GOODS_CREDIT_RATIO,"(0.826, 0.835]",22163,1957,20206,0.090091,0.088300,-0.097915,8.999582e-04
5,GOODS_CREDIT_RATIO,"(0.835, 0.856]",1565,170,1395,0.006362,0.108626,-0.327631,7.834937e-04
6,GOODS_CREDIT_RATIO,"(0.856, 0.863]",14124,1333,12791,0.057413,0.094378,-0.171172,1.807499e-03
7,GOODS_CREDIT_RATIO,"(0.863, 0.873]",11582,854,10728,0.047080,0.073735,0.098199,4.357055e-04
8,GOODS_CREDIT_RATIO,"(0.873, 0.883]",13578,961,12617,0.055193,0.070776,0.142344,1.053655e-03
9,GOODS_CREDIT_RATIO,"(0.883, 0.894]",12950,819,12131,0.052641,0.063243,0.262953,3.261461e-03


In [45]:
merge_plan = [
    [0,1],        # ≤0.783
    [2,3,4],      # 0.783–0.835
    [5,6,7],      # 0.835–0.873
    [8,9,10],     # 0.873–0.904
    [11,12,13]    # >0.904
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="GOODS_CREDIT_RATIO",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [46]:
all_woe_mapping_num.update(woe_mappings)

In [47]:
num_bins["GOODS_CREDIT_RATIO"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,GOODS_CREDIT_RATIO,"(0.166, 0.783]]",24822,3105,21717,0.100899,0.125091,-0.487400,0.029397
2,GOODS_CREDIT_RATIO,"(0.783, 0.835]]",47414,4699,42715,0.192734,0.099106,-0.225282,0.010752
5,GOODS_CREDIT_RATIO,"(0.835, 0.873]]",27271,2357,24914,0.110854,0.086429,-0.074442,0.000634
8,GOODS_CREDIT_RATIO,"(0.873, 0.904]]",36860,2613,34247,0.149833,0.070890,0.140618,0.002793
11,GOODS_CREDIT_RATIO,"(0.904, 6.667]]",109420,7069,102351,0.444782,0.064604,0.240207,0.023214
14,GOODS_CREDIT_RATIO,SPECIAL_-88888,221,17,204,0.000898,0.076923,0.052425,0.000002
Totals,GOODS_CREDIT_RATIO,None,246008,19860,226148,1.000000,0.080729,NaN,0.076527


In [48]:
rows.append({
    'feature':'GOODS_CREDIT_RATIO',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.000898,
    'bad_rate_trend':'decreasing',
    'iv':0.076527,
    'keep':'Yes',
    'comment':'clear monotonic trend SPECIAL_-88888 ',
})

##### IP_RATIO_EARLY_PAYMENTS_1080D

In [49]:
num_bins["IP_RATIO_EARLY_PAYMENTS_1080D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_RATIO_EARLY_PAYMENTS_1080D,"(-0.001, 0.33]",11153,964,10189,0.045336,0.086434,-0.074509,0.000260
1,IP_RATIO_EARLY_PAYMENTS_1080D,"(0.33, 0.448]",11160,1252,9908,0.045364,0.112186,-0.363882,0.006997
2,IP_RATIO_EARLY_PAYMENTS_1080D,"(0.448, 0.524]",11170,1338,9832,0.045405,0.119785,-0.438016,0.010467
3,IP_RATIO_EARLY_PAYMENTS_1080D,"(0.524, 0.596]",11148,1258,9890,0.045316,0.112845,-0.370481,0.007265
4,IP_RATIO_EARLY_PAYMENTS_1080D,"(0.596, 0.667]",13069,1450,11619,0.053124,0.110950,-0.351404,0.007602
5,IP_RATIO_EARLY_PAYMENTS_1080D,"(0.667, 0.731]",9279,967,8312,0.037718,0.104214,-0.281225,0.003357
6,IP_RATIO_EARLY_PAYMENTS_1080D,"(0.731, 0.8]",13110,1271,11839,0.053291,0.096949,-0.200887,0.002340
7,IP_RATIO_EARLY_PAYMENTS_1080D,"(0.8, 0.85]",9204,815,8389,0.037413,0.088548,-0.100994,0.000398
8,IP_RATIO_EARLY_PAYMENTS_1080D,"(0.85, 0.9]",12241,1047,11194,0.049759,0.085532,-0.063033,0.000203
9,IP_RATIO_EARLY_PAYMENTS_1080D,"(0.9, 0.935]",10062,787,9275,0.040901,0.078215,0.034368,0.000048


In [50]:

merge_plan = [
    [0, 1, 2,3],
    [4,5, 6],           
    [7, 8, 9, 10],      
    [11],               
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_RATIO_EARLY_PAYMENTS_1080D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [51]:
num_bins["IP_RATIO_EARLY_PAYMENTS_1080D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_RATIO_EARLY_PAYMENTS_1080D,"(-0.001, 0.596]]",44631,4812,39819,0.181421,0.107817,-0.319251,0.021141
4,IP_RATIO_EARLY_PAYMENTS_1080D,"(0.596, 0.8]]",35458,3688,31770,0.144134,0.104010,-0.279044,0.012617
7,IP_RATIO_EARLY_PAYMENTS_1080D,"(0.8, 0.973]]",42616,3293,39323,0.173230,0.077271,0.047529,0.000384
11,IP_RATIO_EARLY_PAYMENTS_1080D,"(0.973, 1.0]]",89181,5507,83674,0.362513,0.061751,0.288426,0.026739
12,IP_RATIO_EARLY_PAYMENTS_1080D,SPECIAL_-99999,34122,2560,31562,0.138703,0.075025,0.079465,0.000847
Totals,IP_RATIO_EARLY_PAYMENTS_1080D,None,246008,19860,226148,1.000000,0.080729,NaN,0.071447


In [52]:
all_woe_mapping_num.update(woe_mappings)

In [53]:
rows.append({
    'feature':'IP_RATIO_EARLY_PAYMENTS_1080D',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.138703,
    'bad_rate_trend':'decreasing',
    'iv':0.071447,
    'keep':'Yes',
    'comment':'SUPERWISE the trend is clear but slow at start and after faster',
})

##### IP_WORST_DPD_270D

In [54]:
num_bins["IP_WORST_DPD_270D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_WORST_DPD_270D,"(-0.001, 1.0]",50627,4152,46475,0.205794,0.082012,-0.017158,0.000061
1,IP_WORST_DPD_270D,"(1.0, 2.0]",6483,666,5817,0.026353,0.102730,-0.265232,0.002072
2,IP_WORST_DPD_270D,"(2.0, 3.0]",5002,576,4426,0.020333,0.115154,-0.393338,0.003710
3,IP_WORST_DPD_270D,"(3.0, 4.0]",4446,482,3964,0.018073,0.108412,-0.325417,0.002194
4,IP_WORST_DPD_270D,"(4.0, 5.0]",3490,405,3085,0.014187,0.116046,-0.402062,0.002714
5,IP_WORST_DPD_270D,"(5.0, 7.0]",5047,669,4378,0.020516,0.132554,-0.553919,0.007936
6,IP_WORST_DPD_270D,"(7.0, 10.0]",3670,628,3042,0.014918,0.171117,-0.854752,0.015531
7,IP_WORST_DPD_270D,"(10.0, 248.0]",3471,643,2828,0.014109,0.185249,-0.951302,0.018904
8,IP_WORST_DPD_270D,SPECIAL_-66666,76785,5914,70871,0.312124,0.077020,0.051057,0.000796
9,IP_WORST_DPD_270D,SPECIAL_-99999,86987,5725,81262,0.353594,0.065814,0.220354,0.015659


In [55]:
merge_plan = [
    [0],
    [1,2,3],
    [4,5],
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_WORST_DPD_270D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [56]:
all_woe_mapping_num.update(woe_mappings)

In [57]:
num_bins["IP_WORST_DPD_270D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_WORST_DPD_270D,"(-0.001, 1.0]]",50627,4152,46475,0.205794,0.082012,-0.017158,0.000061
1,IP_WORST_DPD_270D,"(1.0, 4.0]]",15931,1724,14207,0.064758,0.108217,-0.323394,0.007757
4,IP_WORST_DPD_270D,"(4.0, 7.0]]",8537,1074,7463,0.034702,0.125805,-0.493915,0.010411
6,IP_WORST_DPD_270D,"(7.0, 10.0]",3670,628,3042,0.014918,0.171117,-0.854752,0.015531
7,IP_WORST_DPD_270D,"(10.0, 248.0]",3471,643,2828,0.014109,0.185249,-0.951302,0.018904
8,IP_WORST_DPD_270D,SPECIAL_-66666,76785,5914,70871,0.312124,0.077020,0.051057,0.000796
9,IP_WORST_DPD_270D,SPECIAL_-99999,86987,5725,81262,0.353594,0.065814,0.220354,0.015659
Totals,IP_WORST_DPD_270D,None,246008,19860,226148,1.000000,0.080729,NaN,0.069577


In [58]:
rows.append({
    'feature':'IP_WORST_DPD_270D',
    'n_bins':3,
    'n_special_bins':1,
    'special_bins_pct':0.665718,
    'bad_rate_trend':'increasing',
    'iv':0.065074,
    'keep':'Yes',
    'comment':'clear monotonic trend SC_-99999 special values',
})

##### CB_MAX_RATIO_PAYMENT_BALANCE_3M

In [59]:
num_bins["CB_MAX_RATIO_PAYMENT_BALANCE_3M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_PAYMENT_BALANCE_3M,"(-0.001, 0.0306]",1362,147,1215,0.005536,0.107930,-0.320415,0.000650
1,CB_MAX_RATIO_PAYMENT_BALANCE_3M,"(0.0306, 0.0503]",1362,256,1106,0.005536,0.187959,-0.969154,0.007753
2,CB_MAX_RATIO_PAYMENT_BALANCE_3M,"(0.0503, 0.0515]",1361,255,1106,0.005532,0.187362,-0.965240,0.007673
3,CB_MAX_RATIO_PAYMENT_BALANCE_3M,"(0.0515, 0.0525]",1362,232,1130,0.005536,0.170338,-0.849246,0.005677
4,CB_MAX_RATIO_PAYMENT_BALANCE_3M,"(0.0525, 0.0539]",1361,233,1128,0.005532,0.171198,-0.855319,0.005768
5,CB_MAX_RATIO_PAYMENT_BALANCE_3M,"(0.0539, 0.0553]",1362,215,1147,0.005536,0.157856,-0.758215,0.004363
6,CB_MAX_RATIO_PAYMENT_BALANCE_3M,"(0.0553, 0.0571]",1362,213,1149,0.005536,0.156388,-0.747127,0.004217
7,CB_MAX_RATIO_PAYMENT_BALANCE_3M,"(0.0571, 0.06]",1361,186,1175,0.005532,0.136664,-0.589205,0.002457
8,CB_MAX_RATIO_PAYMENT_BALANCE_3M,"(0.06, 0.065]",1362,184,1178,0.005536,0.135095,-0.575844,0.002336
9,CB_MAX_RATIO_PAYMENT_BALANCE_3M,"(0.065, 0.0732]",1361,149,1212,0.005532,0.109478,-0.336401,0.000721


In [60]:
merge_plan = [
    [0,1,2,3,4,5,6,7,8],           # 0.057–0.092
    [9,10,11,12,13,14,15,16,17,18]      # >0.149
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_MAX_RATIO_PAYMENT_BALANCE_3M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [61]:
all_woe_mapping_num.update(woe_mappings)

In [62]:
num_bins["CB_MAX_RATIO_PAYMENT_BALANCE_3M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_PAYMENT_BALANCE_3M,"(-0.001, 0.065]]",12255,1921,10334,0.049815,0.156752,-0.749888,0.038268
9,CB_MAX_RATIO_PAYMENT_BALANCE_3M,"(0.065, 234394.89]]",13616,1397,12219,0.055348,0.102600,-0.263817,0.004303
19,CB_MAX_RATIO_PAYMENT_BALANCE_3M,SPECIAL_-99999,220137,16542,203595,0.894837,0.075144,0.077748,0.005236
Totals,CB_MAX_RATIO_PAYMENT_BALANCE_3M,None,246008,19860,226148,1.000000,0.080729,NaN,0.053877


In [63]:
rows.append({
    'feature':'CB_MAX_RATIO_PAYMENT_BALANCE_3M',
    'n_bins':2,
    'n_special_bins':2,
    'special_bins_pct':0.894837,
    'bad_rate_trend':'decreasing',
    'iv':0.061602,
    'keep':'Yes',
    'comment':'clear monotonic decreasing trend; high missingness handled via SPECIAL_-77777 and SPECIAL_-99999 bins'
})

##### B_RATIO_DEBT_TO_LOAN

In [64]:
num_bins["B_DEBT_TO_CREDIT_RATIO"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_DEBT_TO_CREDIT_RATIO,"(-175.29, 0.0]",55904,3068,52836,0.227245,0.054880,0.413685,0.032745
1,B_DEBT_TO_CREDIT_RATIO,"(0.0, 0.0242]",5322,298,5024,0.021633,0.055994,0.392406,0.002829
2,B_DEBT_TO_CREDIT_RATIO,"(0.0242, 0.0733]",10204,635,9569,0.041478,0.062230,0.280177,0.002897
3,B_DEBT_TO_CREDIT_RATIO,"(0.0733, 0.123]",10204,611,9593,0.041478,0.059878,0.321210,0.003743
4,B_DEBT_TO_CREDIT_RATIO,"(0.123, 0.173]",10204,644,9560,0.041478,0.063113,0.265162,0.002611
5,B_DEBT_TO_CREDIT_RATIO,"(0.173, 0.224]",10205,686,9519,0.041482,0.067222,0.197685,0.001493
6,B_DEBT_TO_CREDIT_RATIO,"(0.224, 0.275]",10204,710,9494,0.041478,0.069581,0.160668,0.001001
7,B_DEBT_TO_CREDIT_RATIO,"(0.275, 0.326]",10204,738,9466,0.041478,0.072325,0.119036,0.000559
8,B_DEBT_TO_CREDIT_RATIO,"(0.326, 0.378]",10204,798,9406,0.041478,0.078205,0.034512,0.000049
9,B_DEBT_TO_CREDIT_RATIO,"(0.378, 0.433]",10204,838,9366,0.041478,0.082125,-0.018659,0.000015


In [65]:
merge_plan = [
    [0],                
    [1,2,3,4,5],        
    [6,7,8],
    [9,10,11],       
    [12,13],
    [14,15],   
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_DEBT_TO_CREDIT_RATIO",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)

In [66]:
num_bins["B_DEBT_TO_CREDIT_RATIO"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_DEBT_TO_CREDIT_RATIO,"(-175.29, 0.0]]",55904,3068,52836,0.227245,0.054880,0.413685,0.032745
1,B_DEBT_TO_CREDIT_RATIO,"(0.0, 0.224]]",46139,2874,43265,0.187551,0.062290,0.279157,0.013009
6,B_DEBT_TO_CREDIT_RATIO,"(0.224, 0.378]]",30612,2246,28366,0.124435,0.073370,0.103558,0.001278
9,B_DEBT_TO_CREDIT_RATIO,"(0.378, 0.558]]",30613,2738,27875,0.124439,0.089439,-0.111979,0.001635
12,B_DEBT_TO_CREDIT_RATIO,"(0.558, 0.733]]",20408,2204,18204,0.082957,0.107997,-0.321115,0.009788
14,B_DEBT_TO_CREDIT_RATIO,"(0.733, 5.843]]",20409,2709,17700,0.082961,0.132736,-0.555497,0.032295
16,B_DEBT_TO_CREDIT_RATIO,SPECIAL_-77777,1,0,1,0.000004,0.000000,0.000000,0.000000
17,B_DEBT_TO_CREDIT_RATIO,SPECIAL_-88888,5908,391,5517,0.024015,0.066181,0.214400,0.001009
18,B_DEBT_TO_CREDIT_RATIO,SPECIAL_-99999,36014,3630,32384,0.146394,0.100794,-0.244050,0.009660
Totals,B_DEBT_TO_CREDIT_RATIO,None,246008,19860,226148,1.000000,0.080729,NaN,0.103549


In [67]:
rows.append({
    'feature':'B_DEBT_TO_CREDIT_RATIO',
    'n_bins':5,
    'n_special_bins':3,
    'special_bins_pct':0.33800,
    'bad_rate_trend':'increasing',
    'iv':0.061182,
    'keep':'Yes',
    'comment':'clear monotonic trend SC_-99999,77777,-88888 special values',
})

##### B_NUM_ACTIVE_CREDIT_720D

In [68]:
num_bins["B_NUM_ACTIVE_CREDIT_720D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_NUM_ACTIVE_CREDIT_720D,"(-0.001, 1.0]",108729,7017,101712,0.441973,0.064537,0.241328,0.023272
1,B_NUM_ACTIVE_CREDIT_720D,"(1.0, 2.0]",41559,3512,38047,0.168934,0.084506,-0.049845,0.000429
2,B_NUM_ACTIVE_CREDIT_720D,"(2.0, 3.0]",20343,2090,18253,0.082692,0.102738,-0.265317,0.006507
3,B_NUM_ACTIVE_CREDIT_720D,"(3.0, 4.0]",8950,1099,7851,0.036381,0.122793,-0.466242,0.009614
4,B_NUM_ACTIVE_CREDIT_720D,"(4.0, 22.0]",6690,1144,5546,0.027194,0.171001,-0.853936,0.028248
5,B_NUM_ACTIVE_CREDIT_720D,SPECIAL_-99999,59737,4998,54739,0.242825,0.083667,-0.038943,0.000374
Totals,B_NUM_ACTIVE_CREDIT_720D,None,246008,19860,226148,1.000000,0.080729,NaN,0.068444


In [69]:
merge_plan = [
    [3,4],
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_NUM_ACTIVE_CREDIT_720D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [70]:
all_woe_mapping_num.update(woe_mappings)

In [71]:
num_bins["B_NUM_ACTIVE_CREDIT_720D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_NUM_ACTIVE_CREDIT_720D,"(-0.001, 1.0]",108729,7017,101712,0.441973,0.064537,0.241328,0.023272
1,B_NUM_ACTIVE_CREDIT_720D,"(1.0, 2.0]",41559,3512,38047,0.168934,0.084506,-0.049845,0.000429
2,B_NUM_ACTIVE_CREDIT_720D,"(2.0, 3.0]",20343,2090,18253,0.082692,0.102738,-0.265317,0.006507
3,B_NUM_ACTIVE_CREDIT_720D,"(3.0, 22.0]]",15640,2243,13397,0.063575,0.143414,-0.645265,0.034651
5,B_NUM_ACTIVE_CREDIT_720D,SPECIAL_-99999,59737,4998,54739,0.242825,0.083667,-0.038943,0.000374
Totals,B_NUM_ACTIVE_CREDIT_720D,None,246008,19860,226148,1.000000,0.080729,NaN,0.068444


In [72]:
rows.append({
    'feature':'B_NUM_ACTIVE_CREDIT_720D',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.322193,
    'bad_rate_trend':'increasing',
    'iv':0.059220,
    'keep':'Yes',
    'comment':'clear monotonic increasing trend; SPECIAL_-99999 represents'
})

##### REMOVED IP_AVG_RATIO_PAYMENT_INSTALMENT

In [73]:
'''merge_plan = [
    [0,1],        # ≤0.857
    [2,3],        # 0.857–0.93
    [4,5],        # 0.93–0.97
    [6,7],        # 0.97–1.0
    [8,9,10],     # >1.0
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_AVG_RATIO_PAYMENT_INSTALMENT",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature':'IP_AVG_RATIO_PAYMENT_INSTALMENT',
    'n_bins':5,
    'n_special_bins':3,
    'special_bins_pct':0.051595,
    'bad_rate_trend':'decreasing',
    'iv':0.056823,
    'keep':'Yes',
    'comment':'SUPERWISE clear monotonic decreasing trend; SPECIAL_-99999, SPECIAL_-77777 and SPECIAL_-88888 represent special values'
})'''

'merge_plan = [\n    [0,1],        # ≤0.857\n    [2,3],        # 0.857–0.93\n    [4,5],        # 0.93–0.97\n    [6,7],        # 0.97–1.0\n    [8,9,10],     # >1.0\n]\n\nwoe_mappings = run_merge_plan_and_collect_mappings(\n    feature="IP_AVG_RATIO_PAYMENT_INSTALMENT",\n    merge_plan=merge_plan,\n    num_bins=num_bins\n)\nall_woe_mapping_num.update(woe_mappings)\nrows.append({\n    \'feature\':\'IP_AVG_RATIO_PAYMENT_INSTALMENT\',\n    \'n_bins\':5,\n    \'n_special_bins\':3,\n    \'special_bins_pct\':0.051595,\n    \'bad_rate_trend\':\'decreasing\',\n    \'iv\':0.056823,\n    \'keep\':\'Yes\',\n    \'comment\':\'SUPERWISE clear monotonic decreasing trend; SPECIAL_-99999, SPECIAL_-77777 and SPECIAL_-88888 represent special values\'\n})'

##### AMT_CREDIT

In [74]:
num_bins["AMT_CREDIT"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,AMT_CREDIT,"(44999.999, 142200.0]",12954,793,12161,0.052657,0.061217,0.297684,0.004121
1,AMT_CREDIT,"(142200.0, 180000.0]",13647,997,12650,0.055474,0.073056,0.108180,0.000620
2,AMT_CREDIT,"(180000.0, 225000.0]",13575,1073,12502,0.055181,0.079042,0.022948,0.000029
3,AMT_CREDIT,"(225000.0, 254700.0]",11725,905,10820,0.047661,0.077186,0.048735,0.000111
4,AMT_CREDIT,"(254700.0, 277969.5]",12888,959,11929,0.052389,0.074410,0.088355,0.000394
5,AMT_CREDIT,"(277969.5, 315000.0]",13828,1325,12503,0.056210,0.095820,-0.187926,0.002148
6,AMT_CREDIT,"(315000.0, 380533.5]",12136,1187,10949,0.049332,0.097808,-0.210663,0.002392
7,AMT_CREDIT,"(380533.5, 450000.0]",17612,1957,15655,0.071591,0.111117,-0.353104,0.010351
8,AMT_CREDIT,"(450000.0, 494550.0]",8224,678,7546,0.033430,0.082442,-0.022856,0.000018
9,AMT_CREDIT,"(494550.0, 536917.5]",13335,1268,12067,0.054206,0.095088,-0.179448,0.001882


In [75]:
merge_plan = [
    [0],
    [1],
    [2,3,4],              # 277k–380k
    [5,6,7,8,9,10,11,12,13,14,15,16,17,18],         # >1.0M
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="AMT_CREDIT",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [76]:
all_woe_mapping_num.update(woe_mappings)

In [77]:
num_bins["AMT_CREDIT"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,AMT_CREDIT,"(44999.999, 142200.0]]",12954,793,12161,0.052657,0.061217,0.297684,0.004121
1,AMT_CREDIT,"(142200.0, 180000.0]]",13647,997,12650,0.055474,0.073056,0.108180,0.000620
2,AMT_CREDIT,"(180000.0, 277969.5]]",38188,2937,35251,0.155231,0.076909,0.052623,0.000420
5,AMT_CREDIT,"(277969.5, 4050000.0]]",181219,15133,166086,0.736639,0.083507,-0.036854,0.001016
Totals,AMT_CREDIT,None,246008,19860,226148,1.000000,0.080729,NaN,0.055306


In [78]:
rows.append({
    'feature':'AMT_CREDIT',
    'n_bins':4,
    'n_special_bins':0,
    'special_bins_pct':0.0,
    'bad_rate_trend':'increasing',
    'iv':0.055306,
    'keep':'Yes',
    'comment':'SUPERWISE binning resulted in weak separation'
})

##### CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M

In [79]:
num_bins["CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,"(-0.001, 1.061]",1690,214,1476,0.006870,0.126627,-0.501367,0.002130
1,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,"(1.061, 1.244]",1689,285,1404,0.006866,0.168739,-0.837891,0.006822
2,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,"(1.244, 1.659]",1690,270,1420,0.006870,0.159763,-0.772492,0.005652
3,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,"(1.659, 2.013]",1689,197,1492,0.006866,0.116637,-0.407813,0.001355
4,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,"(2.013, 2.216]",1690,280,1410,0.006870,0.165680,-0.815927,0.006416
5,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,"(2.216, 2.72]",1689,236,1453,0.006866,0.139728,-0.614928,0.003356
6,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,"(2.72, 3.609]",1690,217,1473,0.006870,0.128402,-0.517323,0.002283
7,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,"(3.609, 5.023]",1689,205,1484,0.006866,0.121374,-0.452996,0.001703
8,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,"(5.023, 7.706]",1690,166,1524,0.006870,0.098225,-0.215376,0.000349
9,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,"(7.706, 11.348]",1689,155,1534,0.006866,0.091770,-0.140273,0.000143


In [80]:
merge_plan = [
    [0,1,2,3,4,5,6,7],        # ≤5.023
    [8,9,10,11,12,13,14,15,16,17,18],            # >20.815
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [81]:
all_woe_mapping_num.update(woe_mappings)

In [82]:
num_bins["CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,"(-0.001, 5.023]]",13516,1904,11612,0.054941,0.140870,-0.624400,0.027801
8,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,"(5.023, 2975900.0]]",18585,1921,16664,0.075546,0.103363,-0.272077,0.006269
19,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,SPECIAL_-99999,213907,16035,197872,0.869512,0.074962,0.080365,0.005430
Totals,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M,None,246008,19860,226148,1.000000,0.080729,NaN,0.044600


In [83]:
rows.append({
    'feature':'CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M',
    'n_bins':2,
    'n_special_bins':2,
    'special_bins_pct':0.869512,
    'bad_rate_trend':'decreasing',
    'iv':0.054918,
    'keep':'Review',
    'comment':'limited non-special observations; majority of data in SPECIAL_-99999 and SPECIAL_-77777 bins, making stable binning difficult'
})

##### REMOVED IP_WORST_DPD_2160D

In [84]:
'''merge_plan = [
    [0],                 # ≤1
    [1,2],               # 1–3
    [3,4],               # 3–5
    [5,6],               # 5–8
    [7,8,9,10,11,12,13]        # >16
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_WORST_DPD_2160D",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature':'IP_WORST_DPD_2160D',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.409470,
    'bad_rate_trend':'increasing',
    'iv':0.054219,
    'keep':'Yes',
    'comment':'clear monotonic increase in event rate with delinquency '
})'''

'merge_plan = [\n    [0],                 # ≤1\n    [1,2],               # 1–3\n    [3,4],               # 3–5\n    [5,6],               # 5–8\n    [7,8,9,10,11,12,13]        # >16\n]\nwoe_mappings = run_merge_plan_and_collect_mappings(\n    feature="IP_WORST_DPD_2160D",\n    merge_plan=merge_plan,\n    num_bins=num_bins\n)\nall_woe_mapping_num.update(woe_mappings)\nrows.append({\n    \'feature\':\'IP_WORST_DPD_2160D\',\n    \'n_bins\':5,\n    \'n_special_bins\':1,\n    \'special_bins_pct\':0.409470,\n    \'bad_rate_trend\':\'increasing\',\n    \'iv\':0.054219,\n    \'keep\':\'Yes\',\n    \'comment\':\'clear monotonic increase in event rate with delinquency \'\n})'

##### REMOVED CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_12M

In [85]:
'''merge_plan = [
    [0,1,2,3,4,5,6,7,8,9],       # higher risk region
    [10,11,12,13,14,15,16,17,18,19]  # lower risk region
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_12M",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature':'CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_12M',
    'n_bins':2,
    'n_special_bins':2,
    'special_bins_pct':0.877813,
    'bad_rate_trend':'decreasing',
    'iv':0.053286,
    'keep':'Yes',
    'comment':'clear trend with SC_-77777, SC_-99999'
})'''

'merge_plan = [\n    [0,1,2,3,4,5,6,7,8,9],       # higher risk region\n    [10,11,12,13,14,15,16,17,18,19]  # lower risk region\n]\nwoe_mappings = run_merge_plan_and_collect_mappings(\n    feature="CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_12M",\n    merge_plan=merge_plan,\n    num_bins=num_bins\n)\nall_woe_mapping_num.update(woe_mappings)\nrows.append({\n    \'feature\':\'CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_12M\',\n    \'n_bins\':2,\n    \'n_special_bins\':2,\n    \'special_bins_pct\':0.877813,\n    \'bad_rate_trend\':\'decreasing\',\n    \'iv\':0.053286,\n    \'keep\':\'Yes\',\n    \'comment\':\'clear trend with SC_-77777, SC_-99999\'\n})'

##### REGION_RATING_CLIENT_W_CITY

In [86]:
num_bins['REGION_RATING_CLIENT_W_CITY']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,REGION_RATING_CLIENT_W_CITY,1,27323,1330,25993,0.111065,0.048677,0.540166,0.025911
1,REGION_RATING_CLIENT_W_CITY,2,183622,14511,169111,0.746407,0.079026,0.023166,0.000397
2,REGION_RATING_CLIENT_W_CITY,3,35063,4019,31044,0.142528,0.114622,-0.388110,0.025263
Totals,REGION_RATING_CLIENT_W_CITY,None,246008,19860,226148,1.000000,0.080729,NaN,0.051572


In [87]:
rows.append({
    'feature':'REGION_RATING_CLIENT_W_CITY',
    'n_bins':3,
    'n_special_bins':0,
    'special_bins_pct':0,
    'bad_rate_trend':'increasing',
    'iv':0.051572,
    'keep':'Yes',
    'comment':'clear monotonic trend'
})

##### IP_NUM_UNDERPAID_INSTALLMENTS_720D

In [88]:
num_bins["IP_NUM_UNDERPAID_INSTALLMENTS_720D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_NUM_UNDERPAID_INSTALLMENTS_720D,"(-0.001, 1.0]",156148,12061,144087,0.634727,0.077241,0.047958,0.001431
1,IP_NUM_UNDERPAID_INSTALLMENTS_720D,"(1.0, 2.0]",10070,1215,8855,0.040934,0.120655,-0.446244,0.009827
2,IP_NUM_UNDERPAID_INSTALLMENTS_720D,"(2.0, 3.0]",5891,714,5177,0.023946,0.121202,-0.451384,0.005895
3,IP_NUM_UNDERPAID_INSTALLMENTS_720D,"(3.0, 5.0]",6407,845,5562,0.026044,0.131887,-0.548106,0.009840
4,IP_NUM_UNDERPAID_INSTALLMENTS_720D,"(5.0, 51.0]",7807,1059,6748,0.031735,0.135647,-0.580561,0.013634
5,IP_NUM_UNDERPAID_INSTALLMENTS_720D,SPECIAL_-99999,59685,3966,55719,0.242614,0.066449,0.210081,0.009808
Totals,IP_NUM_UNDERPAID_INSTALLMENTS_720D,None,246008,19860,226148,1.000000,0.080729,NaN,0.050435


In [89]:
merge_plan = [
    [0],
    [1,2],
    [3,4]
    ]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_NUM_UNDERPAID_INSTALLMENTS_720D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [90]:
num_bins["IP_NUM_UNDERPAID_INSTALLMENTS_720D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_NUM_UNDERPAID_INSTALLMENTS_720D,"(-0.001, 1.0]]",156148,12061,144087,0.634727,0.077241,0.047958,0.001431
1,IP_NUM_UNDERPAID_INSTALLMENTS_720D,"(1.0, 3.0]]",15961,1929,14032,0.064880,0.120857,-0.448143,0.015722
3,IP_NUM_UNDERPAID_INSTALLMENTS_720D,"(3.0, 51.0]]",14214,1904,12310,0.057779,0.133952,-0.566027,0.023455
5,IP_NUM_UNDERPAID_INSTALLMENTS_720D,SPECIAL_-99999,59685,3966,55719,0.242614,0.066449,0.210081,0.009808
Totals,IP_NUM_UNDERPAID_INSTALLMENTS_720D,None,246008,19860,226148,1.000000,0.080729,NaN,0.050435


In [91]:
all_woe_mapping_num.update(woe_mappings)

In [92]:
rows.append({
    'feature':'IP_NUM_UNDERPAID_INSTALLMENTS_720D',
    'n_bins':3,
    'n_special_bins':1,
    'special_bins_pct':0.242614,
    'bad_rate_trend':'increasing',
    'iv':0.050435,
    'keep':'Yes',
    'comment':'clear trend with SC_-99999'
})

##### CB_TREND_CREDIT_DRAWING_3M_12M

In [93]:
num_bins["CB_TREND_CREDIT_DRAWING_3M_12M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_TREND_CREDIT_DRAWING_3M_12M,"(-1.285, -0.151]",2763,523,2240,0.011231,0.189287,-0.977832,0.016065
1,CB_TREND_CREDIT_DRAWING_3M_12M,"(-0.151, -0.111]",2762,459,2303,0.011227,0.166184,-0.819564,0.010595
2,CB_TREND_CREDIT_DRAWING_3M_12M,"(-0.111, -0.0833]",2859,393,2466,0.011622,0.137461,-0.595939,0.005294
3,CB_TREND_CREDIT_DRAWING_3M_12M,"(-0.0833, -0.0536]",2666,327,2339,0.010837,0.122656,-0.464963,0.002847
4,CB_TREND_CREDIT_DRAWING_3M_12M,"(-0.0536, -0.0268]",2761,276,2485,0.011223,0.099964,-0.234855,0.000683
5,CB_TREND_CREDIT_DRAWING_3M_12M,"(-0.0268, -0.00894]",2762,272,2490,0.011227,0.098479,-0.218246,0.000586
6,CB_TREND_CREDIT_DRAWING_3M_12M,"(-0.00894, 0.0]",28427,1742,26685,0.115553,0.061280,0.296586,0.008982
7,CB_TREND_CREDIT_DRAWING_3M_12M,"(0.0, 0.0164]",1954,205,1749,0.007943,0.104913,-0.288692,0.000747
8,CB_TREND_CREDIT_DRAWING_3M_12M,"(0.0164, 0.0774]",2763,274,2489,0.011231,0.099168,-0.225974,0.000631
9,CB_TREND_CREDIT_DRAWING_3M_12M,"(0.0774, 2.932]",2762,350,2412,0.011227,0.126720,-0.502204,0.003494


In [94]:
merge_plan = [
    [0,1,2,3],
    [4,5,6,7,8,9],
    
]
 
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_TREND_CREDIT_DRAWING_3M_12M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [95]:
all_woe_mapping_num.update(woe_mappings)

In [96]:
num_bins["CB_TREND_CREDIT_DRAWING_3M_12M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_TREND_CREDIT_DRAWING_3M_12M,"(-1.285, -0.0536]]",11050,1702,9348,0.044917,0.154027,-0.729124,0.032347
4,CB_TREND_CREDIT_DRAWING_3M_12M,"(-0.0536, 2.932]]",41429,3119,38310,0.168405,0.075285,0.075717,0.000935
10,CB_TREND_CREDIT_DRAWING_3M_12M,SPECIAL_-99999,193529,15039,178490,0.786678,0.077709,0.041404,0.001325
Totals,CB_TREND_CREDIT_DRAWING_3M_12M,None,246008,19860,226148,1.000000,0.080729,NaN,0.051250


In [97]:
rows.append({
    'feature':'CB_TREND_CREDIT_DRAWING_3M_12M',
    'n_bins':2,
    'n_special_bins':1,
    'special_bins_pct':0.721932,
    'bad_rate_trend':'decreasing',
    'iv':0.050210,
    'keep':'Review',
    'comment':'clear monotonic trend with SC_-99999 too much special values'
})

##### IP_MEAN_INSTALMENT_PAYMENT_DIFF

In [98]:
num_bins["IP_MEAN_INSTALMENT_PAYMENT_DIFF"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(-562494.701, -5059.065]",11667,644,11023,0.047425,0.055198,0.407559,0.006649
1,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(-5059.065, -608.123]",11666,731,10935,0.047421,0.062661,0.272828,0.003150
2,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(-608.123, 0.0]",120929,8852,112077,0.491565,0.073200,0.106061,0.005289
3,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(0.0, 97.498]",7397,538,6859,0.030068,0.072732,0.112976,0.000366
4,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(97.498, 269.073]",11666,996,10670,0.047421,0.085376,-0.061038,0.000181
5,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(269.073, 482.11]",11666,1069,10597,0.047421,0.091634,-0.138635,0.000966
6,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(482.11, 764.871]",11666,1140,10526,0.047421,0.097720,-0.209662,0.002276
7,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(764.871, 1195.124]",11666,1203,10463,0.047421,0.103120,-0.269455,0.003855
8,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(1195.124, 1923.975]",11666,1244,10422,0.047421,0.106635,-0.306895,0.005080
9,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(1923.975, 3475.574]",11666,1332,10334,0.047421,0.114178,-0.383724,0.008202


In [99]:
merge_plan = [
    [0,1],
    [2],
    [3,4,5],
    [6,7],
    [8,9,10],
    ]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_MEAN_INSTALMENT_PAYMENT_DIFF",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [100]:
all_woe_mapping_num.update(woe_mappings)

In [101]:
num_bins["IP_MEAN_INSTALMENT_PAYMENT_DIFF"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(-562494.701, -608.123]]",23333,1375,21958,0.094847,0.058929,0.338196,0.009423
2,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(-608.123, 0.0]]",120929,8852,112077,0.491565,0.073200,0.106061,0.005289
3,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(0.0, 482.11]]",30729,2603,28126,0.124911,0.084708,-0.052452,0.000351
6,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(482.11, 1195.124]]",23332,2343,20989,0.094842,0.100420,-0.239916,0.006037
8,IP_MEAN_INSTALMENT_PAYMENT_DIFF,"(1195.124, 438437.7]]",34999,3928,31071,0.142268,0.112232,-0.364337,0.022003
11,IP_MEAN_INSTALMENT_PAYMENT_DIFF,SPECIAL_-99999,12686,759,11927,0.051567,0.059830,0.322076,0.004677
Totals,IP_MEAN_INSTALMENT_PAYMENT_DIFF,None,246008,19860,226148,1.000000,0.080729,NaN,0.049689


In [102]:
rows.append({
    'feature':'IP_MEAN_INSTALMENT_PAYMENT_DIFF',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.051567,
    'bad_rate_trend':'increasing',
    'iv':0.049689,
    'keep':'Yes',
    'comment':'clear trend with SC_-99999 values'
})

##### B_NUM_ACTIVE_CREDIT_270D

In [103]:
num_bins["B_NUM_ACTIVE_CREDIT_270D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_NUM_ACTIVE_CREDIT_270D,"(-0.001, 1.0]",157507,11192,146315,0.640252,0.071057,0.138081,0.011522
1,B_NUM_ACTIVE_CREDIT_270D,"(1.0, 2.0]",20787,2396,18391,0.084497,0.115264,-0.394421,0.015509
2,B_NUM_ACTIVE_CREDIT_270D,"(2.0, 12.0]",7977,1274,6703,0.032426,0.159709,-0.772088,0.026644
3,B_NUM_ACTIVE_CREDIT_270D,SPECIAL_-99999,59737,4998,54739,0.242825,0.083667,-0.038943,0.000374
Totals,B_NUM_ACTIVE_CREDIT_270D,None,246008,19860,226148,1.000000,0.080729,NaN,0.054050


In [104]:
merge_plan = [
    [1,2],
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_NUM_ACTIVE_CREDIT_270D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [105]:
num_bins["B_NUM_ACTIVE_CREDIT_270D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_NUM_ACTIVE_CREDIT_270D,"(-0.001, 1.0]",157507,11192,146315,0.640252,0.071057,0.138081,0.011522
1,B_NUM_ACTIVE_CREDIT_270D,"(1.0, 12.0]]",28764,3670,25094,0.116923,0.127590,-0.510045,0.037657
3,B_NUM_ACTIVE_CREDIT_270D,SPECIAL_-99999,59737,4998,54739,0.242825,0.083667,-0.038943,0.000374
Totals,B_NUM_ACTIVE_CREDIT_270D,None,246008,19860,226148,1.000000,0.080729,NaN,0.054050


In [106]:
rows.append({
    'feature':'B_NUM_ACTIVE_CREDIT_270D',
    'n_bins':2,
    'n_special_bins':1,
    'special_bins_pct':0.322193,
    'bad_rate_trend':'increasing',
    'iv':0.048756,
    'keep':'Yes',
    'comment':'clear monotonic trend with SC_-99999'
})

##### IP_RATIO_EARLY_PAYMENTS_180D

In [107]:
num_bins["IP_RATIO_EARLY_PAYMENTS_180D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_RATIO_EARLY_PAYMENTS_180D,"(-0.001, 0.273]",14883,1448,13435,0.060498,0.097292,-0.204802,2.765314e-03
1,IP_RATIO_EARLY_PAYMENTS_180D,"(0.273, 0.375]",7471,945,6526,0.030369,0.126489,-0.500117,9.365136e-03
2,IP_RATIO_EARLY_PAYMENTS_180D,"(0.375, 0.455]",7906,917,6989,0.032137,0.115988,-0.401497,6.130321e-03
3,IP_RATIO_EARLY_PAYMENTS_180D,"(0.455, 0.5]",9281,919,8362,0.037726,0.099020,-0.224315,2.085712e-03
4,IP_RATIO_EARLY_PAYMENTS_180D,"(0.5, 0.571]",5453,601,4852,0.022166,0.110215,-0.343931,3.028948e-03
5,IP_RATIO_EARLY_PAYMENTS_180D,"(0.571, 0.667]",10186,1003,9183,0.041405,0.098468,-0.218124,2.158850e-03
6,IP_RATIO_EARLY_PAYMENTS_180D,"(0.667, 0.75]",5138,488,4650,0.020885,0.094979,-0.178175,7.145255e-04
7,IP_RATIO_EARLY_PAYMENTS_180D,"(0.75, 0.833]",7262,555,6707,0.029519,0.076425,0.059457,1.017873e-04
8,IP_RATIO_EARLY_PAYMENTS_180D,"(0.833, 1.0]",78148,4880,73268,0.317664,0.062446,0.276497,2.163932e-02
9,IP_RATIO_EARLY_PAYMENTS_180D,SPECIAL_-99999,100280,8104,92176,0.407629,0.080814,-0.001140,5.299778e-07


In [108]:
merge_plan = [
    [0,1,2],
    [3,4],
    [5,6],
    [7,8]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_RATIO_EARLY_PAYMENTS_180D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [109]:
all_woe_mapping_num.update(woe_mappings)

In [110]:
num_bins["IP_RATIO_EARLY_PAYMENTS_180D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_RATIO_EARLY_PAYMENTS_180D,"(-0.001, 0.455]]",30260,3310,26950,0.123004,0.109385,-0.335447,1.593270e-02
3,IP_RATIO_EARLY_PAYMENTS_180D,"(0.455, 0.571]]",14734,1520,13214,0.059892,0.103163,-0.269915,4.886808e-03
5,IP_RATIO_EARLY_PAYMENTS_180D,"(0.571, 0.75]]",15324,1491,13833,0.062291,0.097298,-0.204872,2.849282e-03
7,IP_RATIO_EARLY_PAYMENTS_180D,"(0.75, 1.0]]",85410,5435,79975,0.347184,0.063634,0.256373,2.050329e-02
9,IP_RATIO_EARLY_PAYMENTS_180D,SPECIAL_-99999,100280,8104,92176,0.407629,0.080814,-0.001140,5.299778e-07
Totals,IP_RATIO_EARLY_PAYMENTS_180D,None,246008,19860,226148,1.000000,0.080729,NaN,4.799045e-02


In [111]:
rows.append({
    'feature':'IP_RATIO_EARLY_PAYMENTS_180D',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.407629,  # combined special bins (0.158222 + 0.716745)
    'bad_rate_trend':'increasing',
    'iv':0.04799045,
    'keep':'Review',
    'comment':'good feature still review',
})

##### REMOVED CB_MAX_RATIO_POS_TO_TOTAL_DRAW_12M

In [112]:
'''rows.append({
    'feature':'CB_MAX_RATIO_POS_TO_TOTAL_DRAW_12M',
    'n_bins':4,  
    'n_special_bins':2,  
    'special_bins_pct':0.874966, 
    'bad_rate_trend':'non monotonic trend',  
    'iv':0.047982, 
    'keep':'Drop',
    'comment':'clear trend with SC_-99999',
})'''

"rows.append({\n    'feature':'CB_MAX_RATIO_POS_TO_TOTAL_DRAW_12M',\n    'n_bins':4,  \n    'n_special_bins':2,  \n    'special_bins_pct':0.874966, \n    'bad_rate_trend':'non monotonic trend',  \n    'iv':0.047982, \n    'keep':'Drop',\n    'comment':'clear trend with SC_-99999',\n})"

##### CB_AVG_ATM_WITHDRAWAL_FREQ_6M

In [113]:
num_bins["CB_AVG_ATM_WITHDRAWAL_FREQ_6M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,"(-0.001, 0.167]",33883,2679,31204,0.137731,0.079066,0.022621,0.000070
1,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,"(0.167, 0.333]",2644,339,2305,0.010748,0.128215,-0.515646,0.003546
2,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,"(0.333, 0.5]",2183,289,1894,0.008874,0.132387,-0.552462,0.003412
3,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,"(0.5, 0.833]",3289,449,2840,0.013369,0.136516,-0.587946,0.005909
4,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,"(0.833, 1.167]",1945,285,1660,0.007906,0.146530,-0.670398,0.004700
5,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,"(1.167, 1.833]",2517,419,2098,0.010231,0.166468,-0.821613,0.009712
6,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,"(1.833, 35.0]",2558,494,2064,0.010398,0.193120,-1.002616,0.015789
7,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,SPECIAL_-99999,196989,14906,182083,0.800742,0.075669,0.070217,0.003834
Totals,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,None,246008,19860,226148,1.000000,0.080729,NaN,0.046971


In [114]:
merge_plan = [
    [0],          
    [1, 2, 3, 4, 5,6]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_AVG_ATM_WITHDRAWAL_FREQ_6M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [115]:
all_woe_mapping_num.update(woe_mappings)

In [116]:
num_bins["CB_AVG_ATM_WITHDRAWAL_FREQ_6M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,"(-0.001, 0.167]]",33883,2679,31204,0.137731,0.079066,0.022621,0.000070
1,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,"(0.167, 35.0]]",15136,2275,12861,0.061526,0.150304,-0.700263,0.040393
7,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,SPECIAL_-99999,196989,14906,182083,0.800742,0.075669,0.070217,0.003834
Totals,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,None,246008,19860,226148,1.000000,0.080729,NaN,0.046971


In [117]:
rows.append({
    'feature': 'CB_AVG_ATM_WITHDRAWAL_FREQ_6M',
    'n_bins': 2,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.800742,      
    'bad_rate_trend': 'increasing',    
    'iv':0.046971,                    
    'keep': 'Yes',
    'comment': 'clear strong trend but have too much special bins values',
})

##### REMOVED IP_WORST_DPD_90D

In [118]:
'''merge_plan = [
    [ 1, 2,3,4,5],          
  
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_WORST_DPD_90D",
    merge_plan=merge_plan,
    num_bins=num_bins
)all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature':'IP_WORST_DPD_90D',
    'n_bins':2,
    'n_special_bins':1,
    'special_bins_pct':0.752638,
    'bad_rate_trend':'increasing',
    'iv':0.045835,
    'keep':'Yes',
    'comment':'clear trend with  high SC_-99999 values ' ,
})'''

'merge_plan = [\n    [ 1, 2,3,4,5],          \n\n]\nwoe_mappings = run_merge_plan_and_collect_mappings(\n    feature="IP_WORST_DPD_90D",\n    merge_plan=merge_plan,\n    num_bins=num_bins\n)all_woe_mapping_num.update(woe_mappings)\nrows.append({\n    \'feature\':\'IP_WORST_DPD_90D\',\n    \'n_bins\':2,\n    \'n_special_bins\':1,\n    \'special_bins_pct\':0.752638,\n    \'bad_rate_trend\':\'increasing\',\n    \'iv\':0.045835,\n    \'keep\':\'Yes\',\n    \'comment\':\'clear trend with  high SC_-99999 values \' ,\n})'

##### DAYS_LAST_PHONE_CHANGE

In [119]:
num_bins["DAYS_LAST_PHONE_CHANGE"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,DAYS_LAST_PHONE_CHANGE,"(-4292.001, -2523.0]",12317,637,11680,0.050067,0.051717,0.476382,0.009324
1,DAYS_LAST_PHONE_CHANGE,"(-2523.0, -2159.0]",12296,680,11616,0.049982,0.055303,0.405564,0.006945
2,DAYS_LAST_PHONE_CHANGE,"(-2159.0, -1886.0]",12335,763,11572,0.050141,0.061857,0.286604,0.003655
3,DAYS_LAST_PHONE_CHANGE,"(-1886.0, -1718.0]",12333,764,11569,0.050133,0.061948,0.285035,0.003616
4,DAYS_LAST_PHONE_CHANGE,"(-1718.0, -1569.0]",12230,782,11448,0.049714,0.063941,0.251234,0.002825
5,DAYS_LAST_PHONE_CHANGE,"(-1569.0, -1424.0]",12346,839,11507,0.050185,0.067957,0.186018,0.001607
6,DAYS_LAST_PHONE_CHANGE,"(-1424.0, -1228.0]",12265,865,11400,0.049856,0.070526,0.146157,0.001002
7,DAYS_LAST_PHONE_CHANGE,"(-1228.0, -1058.0]",12350,925,11425,0.050202,0.074899,0.081283,0.000321
8,DAYS_LAST_PHONE_CHANGE,"(-1058.0, -889.0]",12308,1032,11276,0.050031,0.083848,-0.041304,0.000087
9,DAYS_LAST_PHONE_CHANGE,"(-889.0, -757.0]",12301,1013,11288,0.050002,0.082351,-0.021658,0.000024


In [120]:
merge_plan = [
    [0, 1, 2, 3],
    [4, 5, 6, 7],
    [8, 9, 10, 11],
    [12, 13, 14],
    [15, 16,17]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="DAYS_LAST_PHONE_CHANGE",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [121]:
all_woe_mapping_num.update(woe_mappings)

In [122]:
num_bins["DAYS_LAST_PHONE_CHANGE"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,DAYS_LAST_PHONE_CHANGE,"(-4292.001, -1718.0]]",49281,2844,46437,0.200323,0.057710,0.360403,0.022394
4,DAYS_LAST_PHONE_CHANGE,"(-1718.0, -1058.0]]",49191,3411,45780,0.199957,0.069342,0.164360,0.005043
8,DAYS_LAST_PHONE_CHANGE,"(-1058.0, -547.0]]",49237,4335,44902,0.200144,0.088044,-0.094721,0.001869
12,DAYS_LAST_PHONE_CHANGE,"(-547.0, -273.0]]",36934,3473,33461,0.150133,0.094033,-0.167120,0.004498
15,DAYS_LAST_PHONE_CHANGE,"(-273.0, 0.0]]",61364,5797,55567,0.249439,0.094469,-0.172233,0.007954
18,DAYS_LAST_PHONE_CHANGE,SPECIAL_-99999,1,0,1,0.000004,0.000000,0.000000,0.000000
Totals,DAYS_LAST_PHONE_CHANGE,None,246008,19860,226148,1.000000,0.080729,NaN,0.045783


In [123]:
rows.append({
    'feature':'DAYS_LAST_PHONE_CHANGE',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.000004,
    'bad_rate_trend':'increasing',
    'iv':0.045783,
    'keep':'Yes',
    'comment':'clear trend and good feature' ,
})

##### CB_MAX_RATIO_PAYMENT_BALANCE_1M


In [124]:
num_bins['CB_MAX_RATIO_PAYMENT_BALANCE_1M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_PAYMENT_BALANCE_1M,"(-0.001, 0.0199]",1862,246,1616,0.007569,0.132116,-0.550104,0.002883
1,CB_MAX_RATIO_PAYMENT_BALANCE_1M,"(0.0199, 0.0455]",930,149,781,0.003780,0.160215,-0.775853,0.003141
2,CB_MAX_RATIO_PAYMENT_BALANCE_1M,"(0.0455, 0.0492]",931,139,792,0.003784,0.149302,-0.692395,0.002421
3,CB_MAX_RATIO_PAYMENT_BALANCE_1M,"(0.0492, 0.0503]",930,176,754,0.003780,0.189247,-0.977574,0.005404
4,CB_MAX_RATIO_PAYMENT_BALANCE_1M,"(0.0503, 0.0511]",931,165,766,0.003784,0.177229,-0.897245,0.004415
5,CB_MAX_RATIO_PAYMENT_BALANCE_1M,"(0.0511, 0.0518]",930,168,762,0.003780,0.180645,-0.920499,0.004685
6,CB_MAX_RATIO_PAYMENT_BALANCE_1M,"(0.0518, 0.0527]",931,147,784,0.003784,0.157895,-0.758506,0.002985
7,CB_MAX_RATIO_PAYMENT_BALANCE_1M,"(0.0527, 0.0537]",930,136,794,0.003780,0.146237,-0.668053,0.002229
8,CB_MAX_RATIO_PAYMENT_BALANCE_1M,"(0.0537, 0.055]",931,146,785,0.003784,0.156821,-0.750405,0.002912
9,CB_MAX_RATIO_PAYMENT_BALANCE_1M,"(0.055, 0.0566]",930,126,804,0.003780,0.135484,-0.579165,0.001615


In [125]:
0.094038+0.830310

0.924348

In [126]:
# DROP THIS FEATURE

In [127]:
rows.append({
    'feature': 'CB_MAX_RATIO_PAYMENT_BALANCE_1M',
    'n_bins': 4,                        
    'n_special_bins': 2,                
    'special_bins_pct': 0.924348,            
    'bad_rate_trend': 'non monotonic ',     
    'iv': 0.044303,                     
    'keep': 'Drop',
    'comment': 'too much noise and special values',
})

##### REGION_POPULATION_RELATIVE

In [128]:
num_bins["REGION_POPULATION_RELATIVE"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,REGION_POPULATION_RELATIVE,"(-0.00071, 0.005]",13507,1258,12249,0.054905,0.093137,-0.156561,1.437191e-03
1,REGION_POPULATION_RELATIVE,"(0.005, 0.00667]",12663,984,11679,0.051474,0.077707,0.041440,8.687196e-05
2,REGION_POPULATION_RELATIVE,"(0.00667, 0.00733]",14245,1332,12913,0.057905,0.093506,-0.160929,1.604416e-03
3,REGION_POPULATION_RELATIVE,"(0.00733, 0.00917]",12370,1091,11279,0.050283,0.088197,-0.096634,4.889805e-04
4,REGION_POPULATION_RELATIVE,"(0.00917, 0.01]",12622,994,11628,0.051307,0.078751,0.026952,3.685164e-05
5,REGION_POPULATION_RELATIVE,"(0.01, 0.011]",13333,1064,12269,0.054197,0.079802,0.012558,8.502740e-06
6,REGION_POPULATION_RELATIVE,"(0.011, 0.0152]",17144,1397,15747,0.069689,0.081486,-0.010159,7.223244e-06
7,REGION_POPULATION_RELATIVE,"(0.0152, 0.018]",8480,867,7613,0.034470,0.102241,-0.259908,2.596951e-03
8,REGION_POPULATION_RELATIVE,"(0.018, 0.0188]",15353,1495,13858,0.062409,0.097375,-0.205746,2.880125e-03
9,REGION_POPULATION_RELATIVE,"(0.0188, 0.0191]",13095,907,12188,0.053230,0.069263,0.165583,1.361789e-03


In [129]:
merge_plan = [
    [0, 1, 2, 3, 4, 5, 6,7,8,9,10],
    [11,12],
    [13,14, 15],                 # Transition → improving risk
    [16, 17, 18]                  # High population → strong low risk
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="REGION_POPULATION_RELATIVE",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [130]:
all_woe_mapping_num.update(woe_mappings)

In [131]:
num_bins["REGION_POPULATION_RELATIVE"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,REGION_POPULATION_RELATIVE,"(-0.00071, 0.0202]]",143476,12539,130937,0.583217,0.087394,-0.086609,0.004537
11,REGION_POPULATION_RELATIVE,"(0.0202, 0.0252]]",29658,2523,27135,0.120557,0.085070,-0.057106,0.000403
13,REGION_POPULATION_RELATIVE,"(0.0252, 0.0313]]",37035,2827,34208,0.150544,0.076333,0.060761,0.000542
16,REGION_POPULATION_RELATIVE,"(0.0313, 0.0725]]",35839,1971,33868,0.145682,0.054996,0.411448,0.020785
Totals,REGION_POPULATION_RELATIVE,None,246008,19860,226148,1.000000,0.080729,NaN,0.044303


In [132]:
rows.append({
    'feature': 'REGION_POPULATION_RELATIVE',
    'n_bins': 4,  
    'n_special_bins': 0,
    'special_bins_pct': 0, 
    'bad_rate_trend': 'decreasing',
    'iv': 0.044303, 
    'keep': 'Yes', 
    'comment': 'original trend was noisy forced too much monotonicity'
})

##### REMOVED B_AVG_REPAYMENT_DAYS_DIFF

In [133]:
'''merge_plan = [
    [0, 1, 2, 3,4, 5, 6, 7,8, 9, 10, 11, 12],  # near on-time
    [13, 14, 15],        # mild late
    [16, 17],            # strong late
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_AVG_REPAYMENT_DAYS_DIFF",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature': 'B_AVG_REPAYMENT_DAYS_DIFF',
    'n_bins': 3,
    'n_special_bins': 1,
    'special_bins_pct': 0.256406,
    'bad_rate_trend': 'non monotonic',
    'iv':0.040527,
    'keep': 'Drop',
    'comment': 'non monotonic trend'
})'''

'merge_plan = [\n    [0, 1, 2, 3,4, 5, 6, 7,8, 9, 10, 11, 12],  # near on-time\n    [13, 14, 15],        # mild late\n    [16, 17],            # strong late\n]\n\nwoe_mappings = run_merge_plan_and_collect_mappings(\n    feature="B_AVG_REPAYMENT_DAYS_DIFF",\n    merge_plan=merge_plan,\n    num_bins=num_bins\n)\nall_woe_mapping_num.update(woe_mappings)\nrows.append({\n    \'feature\': \'B_AVG_REPAYMENT_DAYS_DIFF\',\n    \'n_bins\': 3,\n    \'n_special_bins\': 1,\n    \'special_bins_pct\': 0.256406,\n    \'bad_rate_trend\': \'non monotonic\',\n    \'iv\':0.040527,\n    \'keep\': \'Drop\',\n    \'comment\': \'non monotonic trend\'\n})'

##### CB_MAX_RATIO_POS_TO_TOTAL_DRAW_3M

In [134]:
num_bins["CB_MAX_RATIO_POS_TO_TOTAL_DRAW_3M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_3M,"(-0.001, 0.0704]",7198,948,6250,0.029259,0.131703,-0.546500,0.010983
1,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_3M,"(0.0704, 0.32]",1029,194,835,0.004183,0.188533,-0.972908,0.005911
2,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_3M,"(0.32, 0.662]",1028,140,888,0.004179,0.136187,-0.585153,0.001827
3,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_3M,"(0.662, 1.0]",11311,1425,9886,0.045978,0.125984,-0.495534,0.013894
4,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_3M,SPECIAL_-99999,225442,17153,208289,0.916401,0.076086,0.064271,0.003685
Totals,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_3M,None,246008,19860,226148,1.000000,0.080729,NaN,0.036300


In [135]:
0.194469+0.721932

0.916401

In [136]:
rows.append({
    'feature':'CB_MAX_RATIO_POS_TO_TOTAL_DRAW_3M',
    'n_bins':4,
    'n_special_bins':2,
    'special_bins_pct':0.916401,
    'bad_rate_trend':'non monotonic trend',
    'iv':0.039557,
    'keep':'Drop',
    'comment':'too much special values dont have proper bins with sp valyes -77777 and -99999'})

##### B_MAX_REPAYMENT_DAYS_DIFF

In [137]:
num_bins["B_MAX_REPAYMENT_DAYS_DIFF"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_MAX_REPAYMENT_DAYS_DIFF,"(-33357.001, -362.0]",9148,696,8452,0.037186,0.076082,0.064327,0.000150
1,B_MAX_REPAYMENT_DAYS_DIFF,"(-362.0, -90.0]",9229,700,8529,0.037515,0.075848,0.067665,0.000167
2,B_MAX_REPAYMENT_DAYS_DIFF,"(-90.0, -29.0]",9806,630,9176,0.039860,0.064246,0.246145,0.002179
3,B_MAX_REPAYMENT_DAYS_DIFF,"(-29.0, -2.0]",8742,693,8049,0.035535,0.079272,0.019791,0.000014
4,B_MAX_REPAYMENT_DAYS_DIFF,"(-2.0, 0.0]",52090,3974,48116,0.211741,0.076291,0.061360,0.000777
5,B_MAX_REPAYMENT_DAYS_DIFF,"(0.0, 1.0]",9273,599,8674,0.037694,0.064596,0.240342,0.001969
6,B_MAX_REPAYMENT_DAYS_DIFF,"(1.0, 2.0]",3828,283,3545,0.015560,0.073929,0.095365,0.000136
7,B_MAX_REPAYMENT_DAYS_DIFF,"(2.0, 5.0]",8592,639,7953,0.034926,0.074372,0.088918,0.000266
8,B_MAX_REPAYMENT_DAYS_DIFF,"(5.0, 12.0]",8677,655,8022,0.035271,0.075487,0.072826,0.000181
9,B_MAX_REPAYMENT_DAYS_DIFF,"(12.0, 21.0]",9245,615,8630,0.037580,0.066522,0.208896,0.001503


In [138]:
merge_plan = [
    [0, 1, 2,3, 4],           # around repayment date
    [5, 6, 7,8],        # slight delay
    [9, 10,11,12],       # moderate delay
    [ 13,14, 15]          # extreme delay
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_MAX_REPAYMENT_DAYS_DIFF",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [139]:
all_woe_mapping_num.update(woe_mappings)

In [140]:
num_bins['B_MAX_REPAYMENT_DAYS_DIFF']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_MAX_REPAYMENT_DAYS_DIFF,"(-33357.001, 0.0]]",89015,6693,82322,0.361838,0.075190,0.077094,0.002082
5,B_MAX_REPAYMENT_DAYS_DIFF,"(0.0, 12.0]]",30370,2176,28194,0.123451,0.071650,0.129139,0.001950
9,B_MAX_REPAYMENT_DAYS_DIFF,"(12.0, 153.0]]",36129,2556,33573,0.146861,0.070746,0.142797,0.002821
13,B_MAX_REPAYMENT_DAYS_DIFF,"(153.0, 41692.0]]",27416,1800,25616,0.111444,0.065655,0.222948,0.005047
16,B_MAX_REPAYMENT_DAYS_DIFF,SPECIAL_-99999,63078,6635,56443,0.256406,0.105187,-0.291609,0.024642
Totals,B_MAX_REPAYMENT_DAYS_DIFF,None,246008,19860,226148,1.000000,0.080729,NaN,0.039511


In [141]:
rows.append({
    'feature':'B_MAX_REPAYMENT_DAYS_DIFF',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.256406,
    'bad_rate_trend':'decreasing',
    'iv':0.039511,
    'keep':'Yes',
    'comment':'monotonic decreasing trend with SC_-99999'
})

##### FLOORSMAX_MEDI

In [142]:
num_bins["FLOORSMAX_MEDI"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,FLOORSMAX_MEDI,"(-0.001, 0.0417]",14385,1338,13047,0.058474,0.093014,-0.155100,0.001501
1,FLOORSMAX_MEDI,"(0.0417, 0.0833]",5434,444,4990,0.022089,0.081708,-0.013115,0.000004
2,FLOORSMAX_MEDI,"(0.0833, 0.125]",5989,478,5511,0.024345,0.079813,0.012409,0.000004
3,FLOORSMAX_MEDI,"(0.125, 0.167]",51220,3683,47537,0.208205,0.071906,0.125299,0.003102
4,FLOORSMAX_MEDI,"(0.167, 0.292]",3608,209,3399,0.014666,0.057927,0.356420,0.001606
5,FLOORSMAX_MEDI,"(0.292, 0.333]",26761,1649,25112,0.108781,0.061620,0.290695,0.008143
6,FLOORSMAX_MEDI,"(0.333, 0.375]",6953,370,6583,0.028263,0.053214,0.446261,0.004676
7,FLOORSMAX_MEDI,"(0.375, 0.479]",3283,182,3101,0.013345,0.055437,0.402991,0.001833
8,FLOORSMAX_MEDI,"(0.479, 1.0]",6078,279,5799,0.024707,0.045903,0.601747,0.006977
9,FLOORSMAX_MEDI,SPECIAL_-99999,122297,11228,111069,0.497126,0.091809,-0.140741,0.010446


In [143]:
merge_plan = [
    [0,1,2],      # very low buildings
    [3],          # low-mid buildings
    [4,5],        # mid buildings
    [6,7,8]       # tall buildings
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="FLOORSMAX_MEDI",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [144]:
all_woe_mapping_num.update(woe_mappings)

In [145]:
num_bins["FLOORSMAX_MEDI"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,FLOORSMAX_MEDI,"(-0.001, 0.125]]",25808,2260,23548,0.104907,0.087570,-0.088806,0.000859
3,FLOORSMAX_MEDI,"(0.125, 0.167]]",51220,3683,47537,0.208205,0.071906,0.125299,0.003102
4,FLOORSMAX_MEDI,"(0.167, 0.333]]",30369,1858,28511,0.123447,0.061181,0.298307,0.009700
6,FLOORSMAX_MEDI,"(0.333, 1.0]]",16314,831,15483,0.066315,0.050938,0.492386,0.013108
9,FLOORSMAX_MEDI,SPECIAL_-99999,122297,11228,111069,0.497126,0.091809,-0.140741,0.010446
Totals,FLOORSMAX_MEDI,None,246008,19860,226148,1.000000,0.080729,NaN,0.038292


In [146]:
rows.append({
    'feature':'FLOORSMAX_MEDI',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.497126,
    'bad_rate_trend':'decreasing',
    'iv':0.038292,
    'keep':'Yes',
    'comment':'clear monotonic decreasing trend'
})

##### YEARS_ID_PUBLISH

In [147]:
num_bins["YEARS_ID_PUBLISH"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,YEARS_ID_PUBLISH,"(-0.001, 1.08]",12967,1350,11617,0.052710,0.104110,-0.280117,4.651852e-03
1,YEARS_ID_PUBLISH,"(1.08, 2.1]",13076,1280,11796,0.053153,0.097889,-0.211582,2.600470e-03
2,YEARS_ID_PUBLISH,"(2.1, 3.02]",12802,1244,11558,0.052039,0.097172,-0.203436,2.345689e-03
3,YEARS_ID_PUBLISH,"(3.02, 3.98]",12989,1257,11732,0.052799,0.096774,-0.198890,2.270430e-03
4,YEARS_ID_PUBLISH,"(3.98, 4.95]",12938,1197,11741,0.052592,0.092518,-0.149213,1.246615e-03
5,YEARS_ID_PUBLISH,"(4.95, 5.89]",12974,1153,11821,0.052738,0.088870,-0.104972,6.072940e-04
6,YEARS_ID_PUBLISH,"(5.89, 6.79]",13050,1184,11866,0.053047,0.090728,-0.127703,9.127288e-04
7,YEARS_ID_PUBLISH,"(6.79, 7.61]",12807,1142,11665,0.052059,0.089170,-0.108670,6.434637e-04
8,YEARS_ID_PUBLISH,"(7.61, 8.48]",13058,1104,11954,0.053080,0.084546,-0.050356,1.374685e-04
9,YEARS_ID_PUBLISH,"(8.48, 9.34]",12928,1047,11881,0.052551,0.080987,-0.003470,6.338451e-07


In [148]:
merge_plan = [
    [0,1,2,3],     
    [4,5,6,7],     
    [8,9,10],      
    [11,12,13],    
    [14,15],       
    [16,17,18]     
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="YEARS_ID_PUBLISH",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [149]:
all_woe_mapping_num.update(woe_mappings)

In [150]:
num_bins["YEARS_ID_PUBLISH"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,YEARS_ID_PUBLISH,"(-0.001, 3.98]]",51834,5131,46703,0.210700,0.098989,-0.223974,0.011612
4,YEARS_ID_PUBLISH,"(3.98, 7.61]]",51769,4676,47093,0.210436,0.090324,-0.122801,0.003341
8,YEARS_ID_PUBLISH,"(7.61, 10.22]]",38907,3110,35797,0.158153,0.079934,0.010759,0.000018
11,YEARS_ID_PUBLISH,"(10.22, 11.68]]",38823,2950,35873,0.157812,0.075986,0.065698,0.000663
14,YEARS_ID_PUBLISH,"(11.68, 12.45]]",26034,1654,24380,0.105826,0.063532,0.258085,0.006329
16,YEARS_ID_PUBLISH,"(12.45, 19.72]]",38641,2339,36302,0.157072,0.060532,0.309667,0.013238
Totals,YEARS_ID_PUBLISH,None,246008,19860,226148,1.000000,0.080729,NaN,0.036918


In [151]:
rows.append({
    'feature':'YEARS_ID_PUBLISH',
    'n_bins':6,
    'n_special_bins':0,
    'special_bins_pct':0,
    'bad_rate_trend':'decreasing',
    'iv':0.036918,
    'keep':'Yes',
    'comment':' clear trend decreasing with no special values' ,
})

##### REG_CITY_NOT_WORK_CITY

In [152]:
num_bins['REG_CITY_NOT_WORK_CITY']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,REG_CITY_NOT_WORK_CITY,0,189169,13812,175357,0.768955,0.073014,0.108804,0.008698
1,REG_CITY_NOT_WORK_CITY,1,56839,6048,50791,0.231045,0.106406,-0.304490,0.024341
Totals,REG_CITY_NOT_WORK_CITY,None,246008,19860,226148,1.000000,0.080729,NaN,0.033039


In [153]:
rows.append({
    'feature':'REG_CITY_NOT_WORK_CITY',
    'n_bins':2,
    'n_special_bins':0,
    'special_bins_pct':0,
    'bad_rate_trend':'increasing',
    'iv':0.033039,
    'keep':'Yes',
    'comment':' monotonic trend with SC_-99999' ,
})

##### AMT_ANNUITY

In [154]:
num_bins["AMT_ANNUITY"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,AMT_ANNUITY,"(1615.499, 9000.0]",15843,1122,14721,0.064400,0.070820,0.141680,0.001218
1,AMT_ANNUITY,"(9000.0, 11250.0]",10661,731,9930,0.043336,0.068568,0.176420,0.001253
2,AMT_ANNUITY,"(11250.0, 13500.0]",16020,1071,14949,0.065120,0.066854,0.203570,0.002479
3,AMT_ANNUITY,"(13500.0, 15129.0]",9283,720,8563,0.037735,0.077561,0.043473,0.000070
4,AMT_ANNUITY,"(15129.0, 16875.0]",13284,1021,12263,0.053998,0.076859,0.053322,0.000150
5,AMT_ANNUITY,"(16875.0, 18900.0]",12614,1202,11412,0.051275,0.095291,-0.181803,0.001829
6,AMT_ANNUITY,"(18900.0, 20749.5]",12927,1039,11888,0.052547,0.080374,0.004789,0.000001
7,AMT_ANNUITY,"(20749.5, 22365.0]",12955,1290,11665,0.052661,0.099575,-0.230531,0.003083
8,AMT_ANNUITY,"(22365.0, 24021.0]",12947,973,11974,0.052628,0.075153,0.077627,0.000307
9,AMT_ANNUITY,"(24021.0, 25785.0]",12963,1204,11759,0.052693,0.092880,-0.153512,0.001324


In [155]:
merge_plan = [
    [0,1,2,3,4,5], 
    [6,7,8,9,10,11],
    [12,13,14,15,16,17,18]
]#amt annuity
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="AMT_ANNUITY",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [156]:
all_woe_mapping_num.update(woe_mappings)

In [157]:
num_bins["AMT_ANNUITY"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,AMT_ANNUITY,"(1615.499, 18900.0]]",77705,5867,71838,0.315864,0.075504,0.072588,0.001614
6,AMT_ANNUITY,"(18900.0, 29407.5]]",77684,6838,70846,0.315778,0.088023,-0.094469,0.002932
12,AMT_ANNUITY,"(29407.5, 258025.5]]",90609,7155,83454,0.368317,0.078966,0.024002,0.000210
19,AMT_ANNUITY,SPECIAL_-99999,10,0,10,0.000041,0.000000,0.000000,0.000000
Totals,AMT_ANNUITY,None,246008,19860,226148,1.000000,0.080729,NaN,0.031323


In [158]:
rows.append({
    'feature':'AMT_ANNUITY',
    'n_bins':3,
    'n_special_bins':1,
    'special_bins_pct':0.000041,
    'bad_rate_trend':'non monotonic',
    'iv':0.025758,
    'keep':'Drop',
    'comment':' monotonic trend with SC_-99999 and SC_-77777' ,
})

##### ELEVATORS_MEDI

In [159]:
num_bins["ELEVATORS_MEDI"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,ELEVATORS_MEDI,"(-0.001, 0.08]",82808,6191,76617,0.336607,0.074763,0.083240,0.002252
1,ELEVATORS_MEDI,"(0.08, 0.12]",5221,292,4929,0.021223,0.055928,0.393656,0.002792
2,ELEVATORS_MEDI,"(0.12, 0.16]",7729,457,7272,0.031418,0.059128,0.334621,0.003060
3,ELEVATORS_MEDI,"(0.16, 0.2]",3548,205,3343,0.014422,0.057779,0.359132,0.001602
4,ELEVATORS_MEDI,"(0.2, 0.24]",5333,286,5047,0.021678,0.053628,0.438075,0.003468
5,ELEVATORS_MEDI,"(0.24, 0.36]",5593,314,5279,0.022735,0.056142,0.389617,0.002935
6,ELEVATORS_MEDI,"(0.36, 1.0]",4759,233,4526,0.019345,0.048960,0.534073,0.004423
7,ELEVATORS_MEDI,SPECIAL_-99999,131017,11882,119135,0.532572,0.090691,-0.127249,0.009097
Totals,ELEVATORS_MEDI,None,246008,19860,226148,1.000000,0.080729,NaN,0.029629


In [160]:
merge_plan = [
    [1,2,3],
    [4,5,6]]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="ELEVATORS_MEDI",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [161]:
all_woe_mapping_num.update(woe_mappings)

In [162]:
num_bins["ELEVATORS_MEDI"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,ELEVATORS_MEDI,"(-0.001, 0.08]",82808,6191,76617,0.336607,0.074763,0.083240,0.002252
1,ELEVATORS_MEDI,"(0.08, 0.2]]",16498,954,15544,0.067063,0.057825,0.358284,0.007416
4,ELEVATORS_MEDI,"(0.2, 1.0]]",15685,833,14852,0.063758,0.053108,0.448374,0.010640
7,ELEVATORS_MEDI,SPECIAL_-99999,131017,11882,119135,0.532572,0.090691,-0.127249,0.009097
Totals,ELEVATORS_MEDI,None,246008,19860,226148,1.000000,0.080729,NaN,0.029629


In [163]:
rows.append({
    'feature':'ELEVATORS_MEDI',
    'n_bins':3,
    'n_special_bins':1,
    'special_bins_pct':0.532572,
    'bad_rate_trend':'decreasing',
    'iv':0.029629,
    'keep':'Yes',
    'comment':' monotonic trend with SC_-99999' ,
})

##### FLAG_DOCUMENT_3

In [164]:
num_bins['FLAG_DOCUMENT_3']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,FLAG_DOCUMENT_3,0,71232,4404,66828,0.289552,0.061826,0.287127,0.021177
1,FLAG_DOCUMENT_3,1,174776,15456,159320,0.710448,0.088433,-0.099565,0.007343
Totals,FLAG_DOCUMENT_3,None,246008,19860,226148,1.000000,0.080729,NaN,0.028520


In [165]:
rows.append({
    'feature':'FLAG_DOCUMENT_3',
    'n_bins':2,
    'n_special_bins':0,
    'special_bins_pct':0,
    'bad_rate_trend':'increasing',
    'iv':0.028520,
    'keep':'Yes',
    'comment':' monotonic trend with SC_-99999' ,
})

##### CB_RATIO_MAX_POS_SPEND_9M

In [166]:
'''merge_plan = [
    [1,2, 3,4,5,6],    
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_RATIO_MAX_POS_SPEND_9M",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature':'CB_RATIO_MAX_POS_SPEND_9M',
    'n_bins':2,
    'n_special_bins':2,
    'special_bins_pct':0.772268,
    'bad_rate_trend':'increasing',
    'iv':0.028363,
    'keep':'Review',
    'comment':' monotonic trend with SC_-99999 SC_-77777' ,
})'''

'merge_plan = [\n    [1,2, 3,4,5,6],    \n]\nwoe_mappings = run_merge_plan_and_collect_mappings(\n    feature="CB_RATIO_MAX_POS_SPEND_9M",\n    merge_plan=merge_plan,\n    num_bins=num_bins\n)\nall_woe_mapping_num.update(woe_mappings)\nrows.append({\n    \'feature\':\'CB_RATIO_MAX_POS_SPEND_9M\',\n    \'n_bins\':2,\n    \'n_special_bins\':2,\n    \'special_bins_pct\':0.772268,\n    \'bad_rate_trend\':\'increasing\',\n    \'iv\':0.028363,\n    \'keep\':\'Review\',\n    \'comment\':\' monotonic trend with SC_-99999 SC_-77777\' ,\n})'

##### YEARS_REGISTRATION

In [167]:
num_bins["YEARS_REGISTRATION"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,YEARS_REGISTRATION,"(-0.001, 0.96]",13028,1293,11735,0.052958,0.099248,-0.226871,0.002998
1,YEARS_REGISTRATION,"(0.96, 2.01]",12999,1250,11749,0.052840,0.096161,-0.191857,0.002108
2,YEARS_REGISTRATION,"(2.01, 3.09]",12864,1159,11705,0.052291,0.090096,-0.120023,0.000792
3,YEARS_REGISTRATION,"(3.09, 4.39]",12980,1122,11858,0.052763,0.086441,-0.074592,0.000303
4,YEARS_REGISTRATION,"(4.39, 5.94]",12906,1083,11823,0.052462,0.083914,-0.042170,0.000095
5,YEARS_REGISTRATION,"(5.94, 7.38]",12924,1132,11792,0.052535,0.087589,-0.089047,0.000432
6,YEARS_REGISTRATION,"(7.38, 8.79]",13017,1137,11880,0.052913,0.087347,-0.086019,0.000406
7,YEARS_REGISTRATION,"(8.79, 10.33]",12920,1148,11772,0.052519,0.088854,-0.104779,0.000603
8,YEARS_REGISTRATION,"(10.33, 11.69]",12914,1104,11810,0.052494,0.085489,-0.062475,0.000210
9,YEARS_REGISTRATION,"(11.69, 12.95]",12939,1155,11784,0.052596,0.089265,-0.109840,0.000664


In [168]:
merge_plan = [
    [0,1,2],
    [3,4,5,6,7],
    [8,9,10,11],
    [12,13,14],
    [15,16],
    [17,18]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="YEARS_REGISTRATION",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [169]:
all_woe_mapping_num.update(woe_mappings)

In [170]:
num_bins["YEARS_REGISTRATION"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,YEARS_REGISTRATION,"(-0.001, 3.09]]",38891,3702,35189,0.158088,0.095189,-0.180622,0.005564
3,YEARS_REGISTRATION,"(3.09, 10.33]]",64747,5622,59125,0.263191,0.086830,-0.079516,0.001721
8,YEARS_REGISTRATION,"(10.33, 16.09]]",51782,4458,47324,0.210489,0.086092,-0.070165,0.001067
12,YEARS_REGISTRATION,"(16.09, 22.14]]",38807,2873,35934,0.157747,0.074033,0.093845,0.001336
15,YEARS_REGISTRATION,"(22.14, 26.93]]",25910,1748,24162,0.105322,0.067464,0.193827,0.003649
17,YEARS_REGISTRATION,"(26.93, 67.59]]",25871,1457,24414,0.105163,0.056318,0.386295,0.013363
Totals,YEARS_REGISTRATION,None,246008,19860,226148,1.000000,0.080729,NaN,0.027729


In [171]:
rows.append({
    'feature':'YEARS_REGISTRATION',
    'n_bins':5,
    'n_special_bins':0,
    'special_bins_pct':0,
    'bad_rate_trend':'decreasing',
    'iv':0.027729,
    'keep':'Yes',
    'comment':'clear monotonic trend ' ,
})

##### ENTRANCES_AVG

In [172]:
num_bins["ENTRANCES_AVG"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,ENTRANCES_AVG,"(-0.001, 0.0345]",12538,982,11556,0.050966,0.078322,0.032887,5.436699e-05
1,ENTRANCES_AVG,"(0.0345, 0.0517]",642,51,591,0.002610,0.079439,0.017508,7.941286e-07
2,ENTRANCES_AVG,"(0.0517, 0.069]",18736,1521,17215,0.076160,0.081181,-0.006069,2.812252e-06
3,ENTRANCES_AVG,"(0.069, 0.0803]",413,27,386,0.001679,0.065375,0.227519,7.902406e-05
4,ENTRANCES_AVG,"(0.0803, 0.103]",17290,1243,16047,0.070282,0.071891,0.125512,1.050516e-03
5,ENTRANCES_AVG,"(0.103, 0.126]",1944,105,1839,0.007902,0.054012,0.430535,1.224800e-03
6,ENTRANCES_AVG,"(0.126, 0.138]",27380,1856,25524,0.111297,0.067787,0.188714,3.662923e-03
7,ENTRANCES_AVG,"(0.138, 0.172]",9045,551,8494,0.036767,0.060918,0.302899,2.973029e-03
8,ENTRANCES_AVG,"(0.172, 0.207]",16289,1064,15225,0.066213,0.065320,0.228421,3.140371e-03
9,ENTRANCES_AVG,"(0.207, 0.276]",10403,657,9746,0.042287,0.063155,0.264446,2.648191e-03


In [173]:
merge_plan = [
    [0, 1,2],
    [3, 4, 5],           # spike region (higher WoE)
    [6],
    [7, 8],
    [9, 10, 11] # stable higher range
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="ENTRANCES_AVG",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [174]:
all_woe_mapping_num.update(woe_mappings)

In [175]:
num_bins["ENTRANCES_AVG"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,ENTRANCES_AVG,"(-0.001, 0.069]]",31916,2554,29362,0.129736,0.080023,0.009559,0.000012
3,ENTRANCES_AVG,"(0.069, 0.126]]",19647,1375,18272,0.079863,0.069985,0.154434,0.001786
6,ENTRANCES_AVG,"(0.126, 0.138]]",27380,1856,25524,0.111297,0.067787,0.188714,0.003663
7,ENTRANCES_AVG,"(0.138, 0.207]]",25334,1615,23719,0.102980,0.063748,0.254459,0.005996
9,ENTRANCES_AVG,"(0.207, 1.0]]",17956,1118,16838,0.072989,0.062263,0.279615,0.005078
12,ENTRANCES_AVG,SPECIAL_-99999,123775,11342,112433,0.503134,0.091634,-0.138637,0.010250
Totals,ENTRANCES_AVG,None,246008,19860,226148,1.000000,0.080729,NaN,0.027555


In [176]:
rows.append({
    'feature':'ENTRANCES_AVG',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.503134,
    'bad_rate_trend':'decreasing',
    'iv':0.027555,
    'keep':'Yes',
    'comment':' clear monotonic trend with SC_-99999' ,
})

##### IP_NUM_UNDERPAID_INSTALLMENTS_2880D

In [177]:
num_bins["IP_NUM_UNDERPAID_INSTALLMENTS_2880D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,"(-0.001, 1.0]",169150,12648,156502,0.687579,0.074774,0.083088,0.004584
1,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,"(1.0, 2.0]",15661,1539,14122,0.063661,0.098270,-0.215881,0.003248
2,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,"(2.0, 3.0]",10384,1045,9339,0.042210,0.100636,-0.242300,0.002743
3,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,"(3.0, 4.0]",7554,808,6746,0.030706,0.106963,-0.310339,0.003369
4,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,"(4.0, 6.0]",10423,1024,9399,0.042369,0.098244,-0.215595,0.002156
5,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,"(6.0, 9.0]",8952,931,8021,0.036389,0.103999,-0.278923,0.003183
6,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,"(9.0, 73.0]",11198,1106,10092,0.045519,0.098768,-0.221489,0.002451
7,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,SPECIAL_-99999,12686,759,11927,0.051567,0.059830,0.322076,0.004677
Totals,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,None,246008,19860,226148,1.000000,0.080729,NaN,0.026411


In [178]:
merge_plan = [
[1],
[2,3,4,5,6]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_NUM_UNDERPAID_INSTALLMENTS_2880D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [179]:
all_woe_mapping_num.update(woe_mappings)

In [180]:
num_bins["IP_NUM_UNDERPAID_INSTALLMENTS_2880D"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,"(-0.001, 1.0]",169150,12648,156502,0.687579,0.074774,0.083088,0.004584
1,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,"(1.0, 2.0]]",15661,1539,14122,0.063661,0.098270,-0.215881,0.003248
2,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,"(2.0, 73.0]]",48511,4914,43597,0.197193,0.101297,-0.249582,0.013640
7,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,SPECIAL_-99999,12686,759,11927,0.051567,0.059830,0.322076,0.004677
Totals,IP_NUM_UNDERPAID_INSTALLMENTS_2880D,None,246008,19860,226148,1.000000,0.080729,NaN,0.026411


In [181]:
rows.append({
    'feature':'IP_NUM_UNDERPAID_INSTALLMENTS_2880D',
    'n_bins':3,
    'n_special_bins':1,
    'special_bins_pct':0.051567,
    'bad_rate_trend':'increasing',
    'iv':0.026411,
    'keep':'Yes',
    'comment':' clear monotonic trend with SC_-99999' ,
})

##### REMOVED B_AVG_AMT_CREDIT_SUM

In [182]:
'''merge_plan = [
    [0],                                      # lowest credit (worst)
    [1,2,3,4,5,6,7,8,9,10,11,12,13],              # noisy middle (force smooth)
    [14,15,16],                             # transition region
    [17,18]                                       # highest credit (best)
 
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_AVG_AMT_CREDIT_SUM",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature':'B_AVG_AMT_CREDIT_SUM',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.143268,
    'bad_rate_trend':'decreasing',
    'iv':0.025847,
    'keep':'Review',
    'comment':'clear monotonic trend with SC_-99999' ,
})'''

'merge_plan = [\n    [0],                                      # lowest credit (worst)\n    [1,2,3,4,5,6,7,8,9,10,11,12,13],              # noisy middle (force smooth)\n    [14,15,16],                             # transition region\n    [17,18]                                       # highest credit (best)\n\n]\n\nwoe_mappings = run_merge_plan_and_collect_mappings(\n    feature="B_AVG_AMT_CREDIT_SUM",\n    merge_plan=merge_plan,\n    num_bins=num_bins\n)\nall_woe_mapping_num.update(woe_mappings)\nrows.append({\n    \'feature\':\'B_AVG_AMT_CREDIT_SUM\',\n    \'n_bins\':4,\n    \'n_special_bins\':1,\n    \'special_bins_pct\':0.143268,\n    \'bad_rate_trend\':\'decreasing\',\n    \'iv\':0.025847,\n    \'keep\':\'Review\',\n    \'comment\':\'clear monotonic trend with SC_-99999\' ,\n})'

##### OWN_CAR_AGE

In [183]:
num_bins["OWN_CAR_AGE"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,OWN_CAR_AGE,"(-0.001, 1.0]",5896,388,5508,0.023967,0.065807,0.220470,1.062437e-03
1,OWN_CAR_AGE,"(1.0, 2.0]",4683,278,4405,0.019036,0.059364,0.330392,1.810686e-03
2,OWN_CAR_AGE,"(2.0, 3.0]",5109,257,4852,0.020768,0.050303,0.505588,4.304776e-03
3,OWN_CAR_AGE,"(3.0, 4.0]",4457,243,4214,0.018117,0.054521,0.420624,2.691223e-03
4,OWN_CAR_AGE,"(4.0, 5.0]",2864,145,2719,0.011642,0.050628,0.498804,2.355347e-03
5,OWN_CAR_AGE,"(5.0, 6.0]",5058,272,4786,0.020560,0.053776,0.435166,3.249500e-03
6,OWN_CAR_AGE,"(6.0, 7.0]",5940,357,5583,0.024146,0.060101,0.317264,2.129329e-03
7,OWN_CAR_AGE,"(7.0, 8.0]",4714,291,4423,0.019162,0.061731,0.288768,1.416530e-03
8,OWN_CAR_AGE,"(8.0, 9.0]",3993,291,3702,0.016231,0.072878,0.110823,1.903104e-04
9,OWN_CAR_AGE,"(9.0, 10.0]",3846,292,3554,0.015634,0.075923,0.066593,6.742242e-05


In [184]:
merge_plan = [
    [0, 1,2],                
    [3, 4,5, 6,7,8],            
    [9, 10,11, 12, 13,14],         
    [ 15, 16, 17, 18], 
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="OWN_CAR_AGE",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [185]:
all_woe_mapping_num.update(woe_mappings)

In [186]:
num_bins["OWN_CAR_AGE"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,OWN_CAR_AGE,"(-0.001, 3.0]]",15688,923,14765,0.063770,0.058835,0.339904,0.006395
3,OWN_CAR_AGE,"(3.0, 9.0]]",27026,1599,25427,0.109858,0.059165,0.333951,0.010660
9,OWN_CAR_AGE,"(9.0, 17.0]]",25758,2151,23607,0.104704,0.083508,-0.036872,0.000145
15,OWN_CAR_AGE,"(17.0, 91.0]]",15118,1408,13710,0.061453,0.093134,-0.156527,0.001608
19,OWN_CAR_AGE,SPECIAL_-99999,162418,13779,148639,0.660214,0.084837,-0.054107,0.001977
Totals,OWN_CAR_AGE,None,246008,19860,226148,1.000000,0.080729,NaN,0.023939


In [187]:
rows.append({
    'feature':'OWN_CAR_AGE',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.660214,
    'bad_rate_trend':'increasing',
    'iv':0.023939,
    'keep':'Yes',
    'comment':'clear monotonic trend with SC_-99999' ,
})

##### REG_CITY_NOT_LIVE_CITY

In [188]:
num_bins['REG_CITY_NOT_LIVE_CITY']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,REG_CITY_NOT_LIVE_CITY,0,226845,17493,209352,0.922104,0.077114,0.049734,0.002234
1,REG_CITY_NOT_LIVE_CITY,1,19163,2367,16796,0.077896,0.123519,-0.472965,0.021243
Totals,REG_CITY_NOT_LIVE_CITY,None,246008,19860,226148,1.000000,0.080729,NaN,0.023477


In [189]:
rows.append({
    'feature':'REG_CITY_NOT_LIVE_CITY',
    'n_bins':2,
    'n_special_bins':0,
    'special_bins_pct':0,
    'bad_rate_trend':'increasing',
    'iv':0.023477,
    'keep':'Yes',
    'comment':'clear monotonic trend ' ,
})

##### REMOVED B_MAX_AMT_OVERDUE

In [190]:
'''merge_plan = [
    [0,1,2],
    [3,4,5],          # moderate overdue
    [6,7],                    # very high overdue (worst)]
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_MAX_AMT_OVERDUE",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature':'B_MAX_AMT_OVERDUE',
    'n_bins':3,
    'n_special_bins':1,
    'special_bins_pct':0.402702,
    'bad_rate_trend':'increasing',
    'iv':0.023322,
    'keep':'Review',
    'comment':'clear monotonic trend with special values -99999 ' ,
})'''

'merge_plan = [\n    [0,1,2],\n    [3,4,5],          # moderate overdue\n    [6,7],                    # very high overdue (worst)]\n]\nwoe_mappings = run_merge_plan_and_collect_mappings(\n    feature="B_MAX_AMT_OVERDUE",\n    merge_plan=merge_plan,\n    num_bins=num_bins\n)\nall_woe_mapping_num.update(woe_mappings)\nrows.append({\n    \'feature\':\'B_MAX_AMT_OVERDUE\',\n    \'n_bins\':3,\n    \'n_special_bins\':1,\n    \'special_bins_pct\':0.402702,\n    \'bad_rate_trend\':\'increasing\',\n    \'iv\':0.023322,\n    \'keep\':\'Review\',\n    \'comment\':\'clear monotonic trend with special values -99999 \' ,\n})'

##### NONLIVINGAREA_AVG

In [191]:
num_bins["NONLIVINGAREA_AVG"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,NONLIVINGAREA_AVG,"(-0.001, 0.0025]",52442,3760,48682,0.213172,0.071698,0.128408,0.003331
1,NONLIVINGAREA_AVG,"(0.0025, 0.0046]",5803,414,5389,0.023589,0.071342,0.133767,0.000399
2,NONLIVINGAREA_AVG,"(0.0046, 0.0075]",5793,377,5416,0.023548,0.065079,0.232386,0.001154
3,NONLIVINGAREA_AVG,"(0.0075, 0.0115]",5723,370,5353,0.023263,0.064651,0.239427,0.001207
4,NONLIVINGAREA_AVG,"(0.0115, 0.0173]",5767,414,5353,0.023442,0.071788,0.127064,0.000359
5,NONLIVINGAREA_AVG,"(0.0173, 0.0254]",5822,398,5424,0.023666,0.068361,0.179655,0.000709
6,NONLIVINGAREA_AVG,"(0.0254, 0.0369]",5796,429,5367,0.023560,0.074017,0.094085,0.000200
7,NONLIVINGAREA_AVG,"(0.0369, 0.0526]",5829,401,5428,0.023694,0.068794,0.172883,0.000659
8,NONLIVINGAREA_AVG,"(0.0526, 0.0778]",5791,373,5418,0.023540,0.064410,0.243422,0.001260
9,NONLIVINGAREA_AVG,"(0.0778, 0.124]",5808,337,5471,0.023609,0.058023,0.354652,0.002562


In [192]:
merge_plan = [
    [0],
    [1,2,3],
    [4,5, 6,7,8,9,10],          

]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="NONLIVINGAREA_AVG",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [193]:
all_woe_mapping_num.update(woe_mappings)

In [194]:
num_bins["NONLIVINGAREA_AVG"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,NONLIVINGAREA_AVG,"(-0.001, 0.0025]]",52442,3760,48682,0.213172,0.071698,0.128408,0.003331
1,NONLIVINGAREA_AVG,"(0.0025, 0.0115]]",17319,1161,16158,0.070400,0.067036,0.200652,0.002606
4,NONLIVINGAREA_AVG,"(0.0115, 1.0]]",40623,2699,37924,0.165129,0.066440,0.210221,0.006684
11,NONLIVINGAREA_AVG,SPECIAL_-99999,135624,12240,123384,0.551299,0.090250,-0.121890,0.008621
Totals,NONLIVINGAREA_AVG,None,246008,19860,226148,1.000000,0.080729,NaN,0.022625


In [195]:
rows.append({
    'feature':'NONLIVINGAREA_AVG',
    'n_bins':3,
    'n_special_bins':1,
    'special_bins_pct':0.551299,
    'bad_rate_trend':'increasing',
    'iv':0.022625,
    'keep':'Review',
    'comment':' required hard binning clear monotonic trend with special values -99999 ' ,
})

##### BASEMENTAREA_AVG

In [196]:
num_bins["BASEMENTAREA_AVG"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,BASEMENTAREA_AVG,"(-0.001, 0.0209]",16155,1245,14910,0.065669,0.077066,0.050415,0.000163
1,BASEMENTAREA_AVG,"(0.0209, 0.0365]",5399,412,4987,0.021946,0.076310,0.061084,0.000080
2,BASEMENTAREA_AVG,"(0.0365, 0.0463]",5349,396,4953,0.021743,0.074033,0.093853,0.000184
3,BASEMENTAREA_AVG,"(0.0463, 0.0541]",5381,370,5011,0.021873,0.068760,0.173406,0.000612
4,BASEMENTAREA_AVG,"(0.0541, 0.0607]",5436,366,5070,0.022097,0.067329,0.195981,0.000782
5,BASEMENTAREA_AVG,"(0.0607, 0.066]",5334,393,4941,0.021682,0.073678,0.099031,0.000204
6,BASEMENTAREA_AVG,"(0.066, 0.0728]",5383,375,5008,0.021881,0.069664,0.159384,0.000520
7,BASEMENTAREA_AVG,"(0.0728, 0.079]",5390,417,4973,0.021910,0.077365,0.046210,0.000046
8,BASEMENTAREA_AVG,"(0.079, 0.0836]",5341,362,4979,0.021711,0.067778,0.188858,0.000716
9,BASEMENTAREA_AVG,"(0.0836, 0.0909]",5430,373,5057,0.022072,0.068692,0.174468,0.000625


In [197]:
merge_plan = [
    [0, 1, 2, 3, 4],       # small basement area → low risk
    [5, 6, 7,8, 9],
    [10,11, 12,13],  # medium area → moderate risk
    [14, 15,16]
    
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="BASEMENTAREA_AVG",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [198]:
all_woe_mapping_num.update(woe_mappings)

In [199]:
num_bins["BASEMENTAREA_AVG"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,BASEMENTAREA_AVG,"(-0.001, 0.0607]]",37720,2789,34931,0.153328,0.073940,0.095210,0.001336
5,BASEMENTAREA_AVG,"(0.0607, 0.0909]]",26878,1920,24958,0.109257,0.071434,0.132387,0.001812
10,BASEMENTAREA_AVG,"(0.0909, 0.139]]",21468,1435,20033,0.087265,0.066844,0.203734,0.003327
14,BASEMENTAREA_AVG,"(0.139, 1.0]]",16113,923,15190,0.065498,0.057283,0.368281,0.007621
17,BASEMENTAREA_AVG,SPECIAL_-99999,143829,12793,131036,0.584652,0.088946,-0.105908,0.006856
Totals,BASEMENTAREA_AVG,None,246008,19860,226148,1.000000,0.080729,NaN,0.022140


In [200]:
rows.append({
    'feature':'BASEMENTAREA_AVG',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.584652,
    'bad_rate_trend':'decreasing',
    'iv':0.022140,
    'keep':'Yes',
    'comment':' required hard binning clear monotonic trend with special values -99999 ' ,
})

##### CB_RATIO_UNDERPAYMENT_24M

In [201]:
num_bins["CB_RATIO_UNDERPAYMENT_24M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_RATIO_UNDERPAYMENT_24M,"(-0.001, 0.0455]",55891,4278,51613,0.227192,0.076542,0.057806,0.000741
1,CB_RATIO_UNDERPAYMENT_24M,"(0.0455, 0.087]",3411,341,3070,0.013865,0.099971,-0.234932,0.000845
2,CB_RATIO_UNDERPAYMENT_24M,"(0.087, 0.13]",3631,476,3155,0.014760,0.131093,-0.541156,0.005421
3,CB_RATIO_UNDERPAYMENT_24M,"(0.13, 0.2]",3446,484,2962,0.014008,0.140453,-0.620947,0.007000
4,CB_RATIO_UNDERPAYMENT_24M,"(0.2, 1.0]",3304,465,2839,0.013430,0.140738,-0.623312,0.006769
5,CB_RATIO_UNDERPAYMENT_24M,SPECIAL_-99999,176325,13816,162509,0.716745,0.078355,0.032424,0.000743
Totals,CB_RATIO_UNDERPAYMENT_24M,None,246008,19860,226148,1.000000,0.080729,NaN,0.021519


In [202]:
merge_plan = [
    [ 1, 2, 3, 4],       # small basement area → low risk
   
    
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_RATIO_UNDERPAYMENT_24M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

In [203]:
all_woe_mapping_num.update(woe_mappings)

In [204]:
num_bins["CB_RATIO_UNDERPAYMENT_24M"]

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_RATIO_UNDERPAYMENT_24M,"(-0.001, 0.0455]",55891,4278,51613,0.227192,0.076542,0.057806,0.000741
1,CB_RATIO_UNDERPAYMENT_24M,"(0.0455, 1.0]]",13792,1766,12026,0.056063,0.128045,-0.514128,0.018377
5,CB_RATIO_UNDERPAYMENT_24M,SPECIAL_-99999,176325,13816,162509,0.716745,0.078355,0.032424,0.000743
Totals,CB_RATIO_UNDERPAYMENT_24M,None,246008,19860,226148,1.000000,0.080729,NaN,0.021519


In [205]:
rows.append({
    'feature':'CB_RATIO_UNDERPAYMENT_24M',
    'n_bins':2,
    'n_special_bins':1,
    'special_bins_pct':0.716745,
    'bad_rate_trend':'increasing',
    'iv':0.021519,
    'keep':'Yes',
    'comment':' required hard binning clear monotonic trend with special values -99999 ' ,
})

##### B_HAS_CREDIT_DAYS_OVERDUE

In [206]:
num_bins['B_HAS_CREDIT_DAYS_OVERDUE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_HAS_CREDIT_DAYS_OVERDUE,0.0,208058,15873,192185,0.845737,0.076291,0.061357,0.003103
1,B_HAS_CREDIT_DAYS_OVERDUE,1.0,2706,420,2286,0.011000,0.155211,-0.738178,0.008149
2,B_HAS_CREDIT_DAYS_OVERDUE,SPECIAL_-99999,35244,3567,31677,0.143264,0.101209,-0.248616,0.009829
Totals,B_HAS_CREDIT_DAYS_OVERDUE,None,246008,19860,226148,1.000000,0.080729,NaN,0.021081


In [207]:
#DROP THE FEATURE
rows.append({
    'feature':'B_HAS_CREDIT_DAYS_OVERDUE',
    'n_bins':2,
    'n_special_bins':1,
    'special_bins_pct':0.143264,
    'bad_rate_trend':'increasing',
    'iv':0.021081,
    'keep':'Drop',
    'comment':'Dropeed due to dont have 5% data each bin',
})

## 2 ND UPDATE FEATURES

In [208]:
already_merge_planned_features = [
    "EXT_SOURCE_3", "YEARS_EMPLOYED", "AMT_GOODS_PRICE", "IP_WORST_DPD_720D", "YEARS_AGE", "CB_MAX_MONTHLY_UTIL_6M", "GOODS_CREDIT_RATIO", "IP_RATIO_EARLY_PAYMENTS_1080D", "IP_WORST_DPD_270D", "CB_MAX_RATIO_PAYMENT_BALANCE_3M", 
    "B_RATIO_DEBT_TO_LOAN", "B_NUM_ACTIVE_CREDIT_720D", "IP_AVG_RATIO_PAYMENT_INSTALMENT", "AMT_CREDIT", "CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_18M", "IP_WORST_DPD_2160D", "CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_12M", "REGION_RATING_CLIENT_W_CITY", "IP_NUM_UNDERPAID_INSTALLMENTS_720D", "CB_TREND_CREDIT_DRAWING_3M_12M",
    "IP_MEAN_INSTALMENT_PAYMENT_DIFF", "B_NUM_ACTIVE_CREDIT_270D", "IP_RATIO_EARLY_PAYMENTS_180D", "CB_MAX_RATIO_POS_TO_TOTAL_DRAW_12M", "CB_AVG_ATM_WITHDRAWAL_FREQ_6M", "IP_WORST_DPD_90D", "DAYS_LAST_PHONE_CHANGE", "CB_MAX_RATIO_PAYMENT_BALANCE_1M", "REGION_POPULATION_RELATIVE", "CB_MAX_RATIO_POS_TO_TOTAL_DRAW_3M", "B_MAX_REPAYMENT_DAYS_DIFF", "FLOORSMAX_MEDI", "YEARS_ID_PUBLISH", "AMT_ANNUITY", "ELEVATORS_MEDI", "FLAG_DOCUMENT_3", "CB_RATIO_MAX_POS_SPEND_9M", "YEARS_REGISTRATION", "ENTRANCES_AVG", "IP_NUM_UNDERPAID_INSTALLMENTS_2880D", "OWN_CAR_AGE", "REG_CITY_NOT_LIVE_CITY", "NONLIVINGAREA_AVG", "BASEMENTAREA_AVG", "CB_RATIO_UNDERPAYMENT_24M", "B_HAS_CREDIT_DAYS_OVERDUE", "OCCUPATION_GROUP", "ORG_GROUP", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "CODE_GENDER", "NAME_FAMILY_STATUS"
]

In [209]:
remaining_feature = set(features) - set(already_merge_planned_features) 
remaining_feature = list(remaining_feature)
filt = iv_df['feature'].isin(remaining_feature)
remaining_feature =iv_df.loc[filt].sort_values(by='IV',ascending=False)['feature'].tolist()

##### REMOVED EXT_SOURCE_MEAN

In [210]:
'''merge_plan = [

    [0, 1, 2],
    [3, 4, 5],
    [6, 7, 8, 9],
    [10, 11, 12, 13],
    [14, 15, 16, 17],
    [18, 19]

]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="EXT_SOURCE_MEAN",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature':'EXT_SOURCE_MEAN',
    'n_bins':6,
    'n_special_bins':1,
    'special_bins_pct':0.000565,
    'bad_rate_trend':'decreasing',
    'iv':0.624329,
    'keep':'Yes',
    'comment':'Strong monotonic decreasing bad rate with very high IV'
})'''

'merge_plan = [\n\n    [0, 1, 2],\n    [3, 4, 5],\n    [6, 7, 8, 9],\n    [10, 11, 12, 13],\n    [14, 15, 16, 17],\n    [18, 19]\n\n]\n\nwoe_mappings = run_merge_plan_and_collect_mappings(\n    feature="EXT_SOURCE_MEAN",\n    merge_plan=merge_plan,\n    num_bins=num_bins\n)\n\nall_woe_mapping_num.update(woe_mappings)\nrows.append({\n    \'feature\':\'EXT_SOURCE_MEAN\',\n    \'n_bins\':6,\n    \'n_special_bins\':1,\n    \'special_bins_pct\':0.000565,\n    \'bad_rate_trend\':\'decreasing\',\n    \'iv\':0.624329,\n    \'keep\':\'Yes\',\n    \'comment\':\'Strong monotonic decreasing bad rate with very high IV\'\n})'

##### EXT_SOURCE_X_INCOME

In [211]:
num_bins['EXT_SOURCE_X_INCOME']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,EXT_SOURCE_X_INCOME,"(1.879, 23956.09]",12294,2690,9604,0.049974,0.218806,-1.159844,0.107843
1,EXT_SOURCE_X_INCOME,"(23956.09, 32196.812]",12293,1732,10561,0.049970,0.140893,-0.624591,0.025303
2,EXT_SOURCE_X_INCOME,"(32196.812, 38372.547]",12294,1604,10690,0.049974,0.130470,-0.535674,0.017943
3,EXT_SOURCE_X_INCOME,"(38372.547, 43775.99]",12293,1362,10931,0.049970,0.110795,-0.349833,0.007082
4,EXT_SOURCE_X_INCOME,"(43775.99, 48778.758]",12294,1199,11095,0.049974,0.097527,-0.207475,0.002347
5,EXT_SOURCE_X_INCOME,"(48778.758, 53721.494]",12293,1119,11174,0.049970,0.091027,-0.131328,0.000911
6,EXT_SOURCE_X_INCOME,"(53721.494, 58550.092]",12293,1051,11242,0.049970,0.085496,-0.062567,0.000201
7,EXT_SOURCE_X_INCOME,"(58550.092, 63427.439]",12294,942,11352,0.049974,0.076623,0.056662,0.000157
8,EXT_SOURCE_X_INCOME,"(63427.439, 68453.926]",12293,910,11383,0.049970,0.074026,0.093950,0.000424
9,EXT_SOURCE_X_INCOME,"(68453.926, 73781.17]",12294,899,11395,0.049974,0.073125,0.107165,0.000549


In [212]:
merge_plan = [

    [0, 1],
    [2,3],
    [4, 5],
    [6, 7, 8, 9],
    [10, 11, 12, 13],
    [14, 15, 16, 17,18, 19]

]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="EXT_SOURCE_X_INCOME",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [213]:
num_bins['EXT_SOURCE_X_INCOME']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,EXT_SOURCE_X_INCOME,"(1.879, 32196.812]]",24587,4422,20165,0.099944,0.179851,-0.915126,0.122161
2,EXT_SOURCE_X_INCOME,"(32196.812, 43775.99]]",24587,2966,21621,0.099944,0.120633,-0.446031,0.023970
4,EXT_SOURCE_X_INCOME,"(43775.99, 53721.494]]",24587,2318,22269,0.099944,0.094277,-0.169991,0.003102
6,EXT_SOURCE_X_INCOME,"(53721.494, 73781.17]]",49174,3802,45372,0.199888,0.077317,0.046886,0.000431
10,EXT_SOURCE_X_INCOME,"(73781.17, 99851.896]]",49173,3048,46125,0.199884,0.061985,0.284387,0.014357
14,EXT_SOURCE_X_INCOME,"(99851.896, 25718078.0]]",73761,3292,70469,0.299832,0.044631,0.631196,0.092057
20,EXT_SOURCE_X_INCOME,SPECIAL_-99999,139,12,127,0.000565,0.086331,-0.073202,0.000003
Totals,EXT_SOURCE_X_INCOME,None,246008,19860,226148,1.000000,0.080729,NaN,0.278724


In [214]:
rows.append({
    'feature':'EXT_SOURCE_X_INCOME',
    'n_bins':6,
    'n_special_bins':1,
    'special_bins_pct':0.000565,
    'bad_rate_trend':'decreasing',
    'iv':0.276619,
    'keep':'Yes',
    'comment':'Strong monotonic decreasing trend'
})

##### EXT_SOURCE_WEIGHTED

In [215]:
num_bins['EXT_SOURCE_WEIGHTED']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,EXT_SOURCE_WEIGHTED,"(-0.0009885, 0.235]",12294,3283,9011,0.049974,0.267041,-1.422794,0.178506
1,EXT_SOURCE_WEIGHTED,"(0.235, 0.304]",12293,2337,9956,0.049970,0.190108,-0.983175,0.072410
2,EXT_SOURCE_WEIGHTED,"(0.304, 0.35]",12294,2005,10289,0.049974,0.163088,-0.797051,0.044204
3,EXT_SOURCE_WEIGHTED,"(0.35, 0.385]",12293,1581,10712,0.049970,0.128610,-0.519175,0.016738
4,EXT_SOURCE_WEIGHTED,"(0.385, 0.415]",12294,1417,10877,0.049974,0.115259,-0.394373,0.009170
5,EXT_SOURCE_WEIGHTED,"(0.415, 0.44]",12293,1198,11095,0.049970,0.097454,-0.206641,0.002327
6,EXT_SOURCE_WEIGHTED,"(0.44, 0.463]",12293,1029,11264,0.049970,0.083706,-0.039458,0.000079
7,EXT_SOURCE_WEIGHTED,"(0.463, 0.485]",12294,955,11339,0.049974,0.077680,0.041810,0.000086
8,EXT_SOURCE_WEIGHTED,"(0.485, 0.505]",12293,797,11496,0.049970,0.064834,0.236418,0.002530
9,EXT_SOURCE_WEIGHTED,"(0.505, 0.525]",12294,765,11529,0.049974,0.062225,0.280263,0.003492


In [216]:
merge_plan = [

    [0, 1, 2],             
    [3, 4, 5],             
    [6, 7, 8, 9],          
    [10, 11, 12],          
    [13, 14, 15],          
    [16, 17, 18, 19]       

]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="EXT_SOURCE_WEIGHTED",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [217]:
num_bins['EXT_SOURCE_WEIGHTED']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,EXT_SOURCE_WEIGHTED,"(-0.0009885, 0.35]]",36881,7625,29256,0.149918,0.206746,-1.087830,0.276930
3,EXT_SOURCE_WEIGHTED,"(0.35, 0.44]]",36880,4196,32684,0.149914,0.113774,-0.379728,0.025348
6,EXT_SOURCE_WEIGHTED,"(0.44, 0.525]]",49174,3546,45628,0.199888,0.072111,0.122219,0.002837
10,EXT_SOURCE_WEIGHTED,"(0.525, 0.583]]",36880,1868,35012,0.149914,0.050651,0.498341,0.030279
13,EXT_SOURCE_WEIGHTED,"(0.583, 0.644]]",36880,1385,35495,0.149914,0.037554,0.811210,0.070751
16,EXT_SOURCE_WEIGHTED,"(0.644, 0.879]]",49174,1228,47946,0.199888,0.024973,1.232207,0.185051
20,EXT_SOURCE_WEIGHTED,SPECIAL_-99999,139,12,127,0.000565,0.086331,-0.073202,0.000003
Totals,EXT_SOURCE_WEIGHTED,None,246008,19860,226148,1.000000,0.080729,NaN,0.626102


In [218]:
rows.append({
    'feature':'EXT_SOURCE_WEIGHTED',
    'n_bins':6,
    'n_special_bins':1,
    'special_bins_pct':0.643788,
    'bad_rate_trend':'decreasing',
    'iv':0.274544,
    'keep':'Yes',
    'comment':'Strong monotonic decreasing trend'
})

##### CREDIT_EXT_SOURCE_PRODUCT

In [219]:
num_bins['CREDIT_EXT_SOURCE_PRODUCT']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CREDIT_EXT_SOURCE_PRODUCT,"(4.25, 52309.32]",12294,2040,10254,0.049974,0.165935,-0.817764,0.046921
1,CREDIT_EXT_SOURCE_PRODUCT,"(52309.32, 74921.044]",12293,1524,10769,0.049970,0.123973,-0.477149,0.013894
2,CREDIT_EXT_SOURCE_PRODUCT,"(74921.044, 93615.676]",12294,1324,10970,0.049974,0.107695,-0.317975,0.005774
3,CREDIT_EXT_SOURCE_PRODUCT,"(93615.676, 111823.514]",12293,1244,11049,0.049970,0.101196,-0.248474,0.003424
4,CREDIT_EXT_SOURCE_PRODUCT,"(111823.514, 129773.24]",12294,1221,11073,0.049974,0.099317,-0.227642,0.002849
5,CREDIT_EXT_SOURCE_PRODUCT,"(129773.24, 148179.082]",12293,1092,11201,0.049970,0.088831,-0.104490,0.000570
6,CREDIT_EXT_SOURCE_PRODUCT,"(148179.082, 167125.878]",12293,1086,11207,0.049970,0.088343,-0.098445,0.000505
7,CREDIT_EXT_SOURCE_PRODUCT,"(167125.878, 188183.604]",12294,1075,11219,0.049974,0.087441,-0.087194,0.000394
8,CREDIT_EXT_SOURCE_PRODUCT,"(188183.604, 213486.538]",12293,1137,11156,0.049970,0.092492,-0.148898,0.001179
9,CREDIT_EXT_SOURCE_PRODUCT,"(213486.538, 242959.31]",12294,1165,11129,0.049974,0.094762,-0.175649,0.001660


In [220]:
merge_plan = [

    [0, 1],            
    [2,3, 4, 5],            
    [6, 7, 8, 9, 10],     
    [11, 12, 13],         
    [14, 15, 16],         
    [17, 18, 19]          

]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CREDIT_EXT_SOURCE_PRODUCT",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [221]:
num_bins['CREDIT_EXT_SOURCE_PRODUCT']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CREDIT_EXT_SOURCE_PRODUCT,"(4.25, 74921.044]]",24587,3564,21023,0.099944,0.144955,-0.657748,0.056892
2,CREDIT_EXT_SOURCE_PRODUCT,"(74921.044, 148179.082]]",49174,4881,44293,0.199888,0.099260,-0.227005,0.011330
6,CREDIT_EXT_SOURCE_PRODUCT,"(148179.082, 274041.892]]",61467,5516,55951,0.249858,0.089739,-0.115659,0.003509
11,CREDIT_EXT_SOURCE_PRODUCT,"(274041.892, 381604.556]]",36880,2526,34354,0.149914,0.068492,0.177599,0.004390
14,CREDIT_EXT_SOURCE_PRODUCT,"(381604.556, 556108.02]]",36880,1979,34901,0.149914,0.053661,0.437442,0.023920
17,CREDIT_EXT_SOURCE_PRODUCT,"(556108.02, 2995346.0]]",36881,1382,35499,0.149918,0.037472,0.813491,0.071087
20,CREDIT_EXT_SOURCE_PRODUCT,SPECIAL_-99999,139,12,127,0.000565,0.086331,-0.073202,0.000003
Totals,CREDIT_EXT_SOURCE_PRODUCT,None,246008,19860,226148,1.000000,0.080729,NaN,0.183983


In [222]:
rows.append({
    'feature':'CREDIT_EXT_SOURCE_PRODUCT',
    'n_bins':6,
    'n_special_bins':1,
    'special_bins_pct':0.000565,
    'bad_rate_trend':'decreasing',
    'iv':0.182650,
    'keep':'Yes',
    'comment':'Strong monotonic decreasing trend'
})

##### B_DAYS_CREDIT_MEAN

In [223]:
num_bins['B_DAYS_CREDIT_MEAN']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_DAYS_CREDIT_MEAN,"(-2922.001, -2103.642]",10539,583,9956,0.042840,0.055318,0.405261,0.005945
1,B_DAYS_CREDIT_MEAN,"(-2103.642, -1810.0]",10541,495,10046,0.042848,0.046959,0.577890,0.011268
2,B_DAYS_CREDIT_MEAN,"(-1810.0, -1648.143]",10536,524,10012,0.042828,0.049734,0.517566,0.009258
3,B_DAYS_CREDIT_MEAN,"(-1648.143, -1530.182]",10538,547,9991,0.042836,0.051907,0.472509,0.007861
4,B_DAYS_CREDIT_MEAN,"(-1530.182, -1434.5]",10540,585,9955,0.042844,0.055503,0.401736,0.005851
5,B_DAYS_CREDIT_MEAN,"(-1434.5, -1349.2]",10538,566,9972,0.042836,0.053710,0.436460,0.006807
6,B_DAYS_CREDIT_MEAN,"(-1349.2, -1270.0]",10564,623,9941,0.042942,0.058974,0.337394,0.004247
7,B_DAYS_CREDIT_MEAN,"(-1270.0, -1194.857]",10512,631,9881,0.042730,0.060027,0.318581,0.003798
8,B_DAYS_CREDIT_MEAN,"(-1194.857, -1122.906]",10536,695,9841,0.042828,0.065964,0.217919,0.001857
9,B_DAYS_CREDIT_MEAN,"(-1122.906, -1050.375]",10539,704,9835,0.042840,0.066800,0.204442,0.001644


In [224]:
merge_plan = [

    [0,1,2,3],            
    [4,5,6,7],            
    [8,9,10,11,12],       
    [13,14,15,16,17],
    [18,19],           
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_DAYS_CREDIT_MEAN",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [225]:
num_bins['B_DAYS_CREDIT_MEAN']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_DAYS_CREDIT_MEAN,"(-2922.001, -1530.182]]",42154,2149,40005,0.171352,0.050980,0.491520,0.033762
4,B_DAYS_CREDIT_MEAN,"(-1530.182, -1194.857]]",42154,2405,39749,0.171352,0.057053,0.372553,0.020367
8,B_DAYS_CREDIT_MEAN,"(-1194.857, -826.4]]",52692,3740,48952,0.214188,0.070979,0.139273,0.003919
13,B_DAYS_CREDIT_MEAN,"(-826.4, -369.333]]",52688,5090,47598,0.214172,0.096606,-0.196969,0.009025
18,B_DAYS_CREDIT_MEAN,"(-369.333, 0.0]]",21076,2909,18167,0.085672,0.138024,-0.600685,0.039731
20,B_DAYS_CREDIT_MEAN,SPECIAL_-99999,35244,3567,31677,0.143264,0.101209,-0.248616,0.009829
Totals,B_DAYS_CREDIT_MEAN,None,246008,19860,226148,1.000000,0.080729,NaN,0.121608


In [226]:
rows.append({
    'feature':'B_DAYS_CREDIT_MEAN',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.143264,
    'bad_rate_trend':'increasing',
    'iv':0.121608,
    'keep':'Yes',
    'comment':'Strong monotonic decreasing trend'
})

##### B_ACTIVE_CREDIT_RATIO

In [227]:
num_bins['B_ACTIVE_CREDIT_RATIO']


,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_ACTIVE_CREDIT_RATIO,"(-0.001, 0.143]",44965,2478,42487,0.182779,0.055110,0.409264,0.025824
1,B_ACTIVE_CREDIT_RATIO,"(0.143, 0.2]",13045,665,12380,0.053027,0.050977,0.491569,0.010450
2,B_ACTIVE_CREDIT_RATIO,"(0.2, 0.25]",14990,865,14125,0.060933,0.057705,0.360490,0.006815
3,B_ACTIVE_CREDIT_RATIO,"(0.25, 0.286]",6276,375,5901,0.025511,0.059751,0.323469,0.002333
4,B_ACTIVE_CREDIT_RATIO,"(0.286, 0.333]",21453,1400,20053,0.087204,0.065259,0.229425,0.004171
5,B_ACTIVE_CREDIT_RATIO,"(0.333, 0.4]",14132,1043,13089,0.057445,0.073804,0.097189,0.000521
6,B_ACTIVE_CREDIT_RATIO,"(0.4, 0.455]",7795,697,7098,0.031686,0.089416,-0.111699,0.000414
7,B_ACTIVE_CREDIT_RATIO,"(0.455, 0.5]",29122,2340,26782,0.118378,0.080352,0.005097,0.000003
8,B_ACTIVE_CREDIT_RATIO,"(0.5, 0.571]",4539,506,4033,0.018451,0.111478,-0.356753,0.002727
9,B_ACTIVE_CREDIT_RATIO,"(0.571, 0.667]",18079,1752,16327,0.073489,0.096908,-0.200420,0.003211


In [228]:
merge_plan = [
    [0, 1, 2, 3],      # Low active credit ratio, WoE positive, event rate increasing slightly
    [4, 5],
    [6, 7],      # Mid-range, WoE decreasing but still near zero
    [8, 9, 10],
    [11],    # High active credit ratio, WoE negative

]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_ACTIVE_CREDIT_RATIO",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [229]:
num_bins['B_ACTIVE_CREDIT_RATIO']


,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_ACTIVE_CREDIT_RATIO,"(-0.001, 0.286]]",79276,4383,74893,0.322250,0.055288,0.405845,0.044835
4,B_ACTIVE_CREDIT_RATIO,"(0.286, 0.4]]",35585,2443,33142,0.144650,0.068653,0.175093,0.004122
6,B_ACTIVE_CREDIT_RATIO,"(0.4, 0.5]]",36917,3037,33880,0.150064,0.082266,-0.020527,0.000064
8,B_ACTIVE_CREDIT_RATIO,"(0.5, 0.75]]",28818,2963,25855,0.117143,0.102818,-0.266180,0.009281
11,B_ACTIVE_CREDIT_RATIO,"(0.75, 1.0]]",30027,3441,26586,0.122057,0.114597,-0.387859,0.021605
12,B_ACTIVE_CREDIT_RATIO,SPECIAL_-99999,35385,3593,31792,0.143837,0.101540,-0.252255,0.010175
Totals,B_ACTIVE_CREDIT_RATIO,None,246008,19860,226148,1.000000,0.080729,NaN,0.092495


In [230]:
rows.append({
    'feature':'B_ACTIVE_CREDIT_RATIO',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.143837,
    'bad_rate_trend':'increasing',
    'iv':0.092495,
    'keep':'Yes',
    'comment':'Strong monotonic increasing trend'
})

##### B_CLOSED_CREDIT_RATIO

In [231]:
num_bins['B_CLOSED_CREDIT_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_CLOSED_CREDIT_RATIO,"(-0.001, 0.25]",34798,3952,30846,0.141451,0.113570,-0.377697,0.023642
1,B_CLOSED_CREDIT_RATIO,"(0.25, 0.333]",12543,1231,11312,0.050986,0.098142,-0.214445,0.002566
2,B_CLOSED_CREDIT_RATIO,"(0.333, 0.429]",9346,962,8384,0.037991,0.102932,-0.267416,0.003039
3,B_CLOSED_CREDIT_RATIO,"(0.429, 0.5]",30501,2502,27999,0.123984,0.082030,-0.017404,0.000038
4,B_CLOSED_CREDIT_RATIO,"(0.5, 0.545]",1648,190,1458,0.006699,0.115291,-0.394685,0.001231
5,B_CLOSED_CREDIT_RATIO,"(0.545, 0.6]",15019,1192,13827,0.061051,0.079366,0.018509,0.000021
6,B_CLOSED_CREDIT_RATIO,"(0.6, 0.667]",24234,1621,22613,0.098509,0.066889,0.203000,0.003729
7,B_CLOSED_CREDIT_RATIO,"(0.667, 0.714]",7494,470,7024,0.030462,0.062717,0.271873,0.002010
8,B_CLOSED_CREDIT_RATIO,"(0.714, 0.75]",13153,775,12378,0.053466,0.058922,0.338331,0.005315
9,B_CLOSED_CREDIT_RATIO,"(0.75, 0.8]",10576,594,9982,0.042990,0.056165,0.389177,0.005538


In [232]:
merge_plan = [
    [0, 1, 2, 3],      # Low active credit ratio, WoE positive, event rate increasing slightly
    [4, 5],
    [6, 7],      # Mid-range, WoE decreasing but still near zero
    [8, 9, 10,11],    # High active credit ratio, WoE negative

]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_CLOSED_CREDIT_RATIO",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [233]:
num_bins['B_CLOSED_CREDIT_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_CLOSED_CREDIT_RATIO,"(-0.001, 0.5]]",87188,8647,78541,0.354411,0.099176,-0.226074,0.019917
4,B_CLOSED_CREDIT_RATIO,"(0.5, 0.6]]",16667,1382,15285,0.067750,0.082918,-0.029142,0.000058
6,B_CLOSED_CREDIT_RATIO,"(0.6, 0.714]]",31728,2091,29637,0.128971,0.065904,0.218899,0.005640
8,B_CLOSED_CREDIT_RATIO,"(0.714, 1.0]]",75040,4147,70893,0.305031,0.055264,0.406305,0.042527
12,B_CLOSED_CREDIT_RATIO,SPECIAL_-99999,35385,3593,31792,0.143837,0.101540,-0.252255,0.010175
Totals,B_CLOSED_CREDIT_RATIO,None,246008,19860,226148,1.000000,0.080729,NaN,0.089606


In [234]:
rows.append({
    'feature':'B_CLOSED_CREDIT_RATIO',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.143837,
    'bad_rate_trend':'decreasing',
    'iv':0.089606,
    'keep':'Yes',
    'comment':'Strong monotonic decreasing trend'
})

##### B_DAYS_CREDIT_MAX

In [235]:
num_bins['B_DAYS_CREDIT_MAX']
#9

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_DAYS_CREDIT_MAX,"(-2922.001, -1647.0]",10547,625,9922,0.042873,0.059259,0.332276,0.004121
1,B_DAYS_CREDIT_MAX,"(-1647.0, -1173.0]",10536,642,9894,0.042828,0.060934,0.302613,0.003457
2,B_DAYS_CREDIT_MAX,"(-1173.0, -910.0]",10564,542,10022,0.042942,0.051306,0.484790,0.008254
3,B_DAYS_CREDIT_MAX,"(-910.0, -743.0]",10523,540,9983,0.042775,0.051316,0.484588,0.008215
4,B_DAYS_CREDIT_MAX,"(-743.0, -620.0]",10605,569,10036,0.043108,0.053654,0.437571,0.006882
5,B_DAYS_CREDIT_MAX,"(-620.0, -522.0]",10519,666,9853,0.042759,0.063314,0.261760,0.002627
6,B_DAYS_CREDIT_MAX,"(-522.0, -448.0]",10526,645,9881,0.042787,0.061277,0.296637,0.003327
7,B_DAYS_CREDIT_MAX,"(-448.0, -389.0]",10520,627,9893,0.042763,0.059601,0.326154,0.003971
8,B_DAYS_CREDIT_MAX,"(-389.0, -343.0]",10585,691,9894,0.043027,0.065281,0.229062,0.002052
9,B_DAYS_CREDIT_MAX,"(-343.0, -300.0]",10608,773,9835,0.043121,0.072870,0.110942,0.000507


In [236]:
merge_plan = [
    [0, 1, 2, 3,4],       # Early credit days, WoE positive
    [5, 6, 7],       # Mid-range, WoE positive but decreasing
    [8, 9, 10, 11],     # Higher days, WoE near zero
    [12, 13, 14, 15, 16, 17],
    [18, 19],  # Late credit days, WoE negative
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_DAYS_CREDIT_MAX",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [237]:
num_bins['B_DAYS_CREDIT_MAX']


,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_DAYS_CREDIT_MAX,"(-2922.001, -620.0]]",52775,2918,49857,0.214526,0.055291,0.405778,0.029838
5,B_DAYS_CREDIT_MAX,"(-620.0, -389.0]]",31565,1938,29627,0.128309,0.061397,0.294548,0.009845
8,B_DAYS_CREDIT_MAX,"(-389.0, -229.0]]",42280,3035,39245,0.171864,0.071783,0.127131,0.002634
12,B_DAYS_CREDIT_MAX,"(-229.0, -67.0]]",63202,5773,57429,0.256910,0.091342,-0.135124,0.004965
18,B_DAYS_CREDIT_MAX,"(-67.0, 0.0]]",20942,2629,18313,0.085127,0.125537,-0.491474,0.025261
20,B_DAYS_CREDIT_MAX,SPECIAL_-99999,35244,3567,31677,0.143264,0.101209,-0.248616,0.009829
Totals,B_DAYS_CREDIT_MAX,None,246008,19860,226148,1.000000,0.080729,NaN,0.086520


In [238]:
rows.append({
    'feature':'B_DAYS_CREDIT_MAX',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.143264,
    'bad_rate_trend':'increasing',
    'iv':0.086520,
    'keep':'Yes',
    'comment':'Strong monotonic increasing trend'
})

##### PA_RATIO_APPROVED_LOANS

In [239]:
num_bins['PA_RATIO_APPROVED_LOANS']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_APPROVED_LOANS,"(-0.001, 0.4]",12779,1972,10807,0.051945,0.154316,-0.731336,0.037669
1,PA_RATIO_APPROVED_LOANS,"(0.4, 0.5]",15739,1928,13811,0.063978,0.122498,-0.463500,0.016690
2,PA_RATIO_APPROVED_LOANS,"(0.5, 0.625]",8615,877,7738,0.035019,0.101799,-0.255090,0.002536
3,PA_RATIO_APPROVED_LOANS,"(0.625, 0.667]",12453,1212,11241,0.050620,0.097326,-0.205186,0.002323
4,PA_RATIO_APPROVED_LOANS,"(0.667, 0.75]",12491,1063,11428,0.050775,0.085101,-0.057511,0.000172
5,PA_RATIO_APPROVED_LOANS,"(0.75, 0.833]",12000,852,11148,0.048779,0.071000,0.138947,0.000889
6,PA_RATIO_APPROVED_LOANS,"(0.833, 1.0]",158429,11146,147283,0.643999,0.070353,0.148793,0.013397
7,PA_RATIO_APPROVED_LOANS,SPECIAL_-99999,13502,810,12692,0.054884,0.059991,0.319211,0.004896
Totals,PA_RATIO_APPROVED_LOANS,None,246008,19860,226148,1.000000,0.080729,NaN,0.078572


In [240]:
merge_plan = [
    [0],
    [1],
    [2, 3],      
    [4,5],
    [6],             
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_RATIO_APPROVED_LOANS",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [241]:
num_bins['PA_RATIO_APPROVED_LOANS']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_APPROVED_LOANS,"(-0.001, 0.4]]",12779,1972,10807,0.051945,0.154316,-0.731336,0.037669
1,PA_RATIO_APPROVED_LOANS,"(0.4, 0.5]]",15739,1928,13811,0.063978,0.122498,-0.463500,0.016690
2,PA_RATIO_APPROVED_LOANS,"(0.5, 0.667]]",21068,2089,18979,0.085639,0.099155,-0.225834,0.004802
4,PA_RATIO_APPROVED_LOANS,"(0.667, 0.833]]",24491,1915,22576,0.099554,0.078192,0.034688,0.000118
6,PA_RATIO_APPROVED_LOANS,"(0.833, 1.0]]",158429,11146,147283,0.643999,0.070353,0.148793,0.013397
7,PA_RATIO_APPROVED_LOANS,SPECIAL_-99999,13502,810,12692,0.054884,0.059991,0.319211,0.004896
Totals,PA_RATIO_APPROVED_LOANS,None,246008,19860,226148,1.000000,0.080729,NaN,0.078572


In [242]:
rows.append({
    'feature':'PA_RATIO_APPROVED_LOANS',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.054884,
    'bad_rate_trend':'decreassing',
    'iv':0.078572,
    'keep':'Yes',
    'comment':'Strong monotonic decreasing trend'
})

##### PA_AVG_RISK_WEIGHT_1080D

In [243]:
num_bins['PA_AVG_RISK_WEIGHT_1080D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_AVG_RISK_WEIGHT_1080D,"(0.999, 2.0]",60976,3571,57405,0.247862,0.058564,0.344804,0.025526
1,PA_AVG_RISK_WEIGHT_1080D,"(2.0, 2.25]",4058,279,3779,0.016495,0.068753,0.173521,0.000462
2,PA_AVG_RISK_WEIGHT_1080D,"(2.25, 2.5]",24147,1707,22440,0.098155,0.070692,0.143626,0.001907
3,PA_AVG_RISK_WEIGHT_1080D,"(2.5, 2.667]",8399,664,7735,0.034141,0.079057,0.022747,0.000017
4,PA_AVG_RISK_WEIGHT_1080D,"(2.667, 3.0]",63345,5351,57994,0.257492,0.084474,-0.049426,0.000642
5,PA_AVG_RISK_WEIGHT_1080D,"(3.0, 3.5]",16863,1980,14883,0.068547,0.117417,-0.415359,0.014075
6,PA_AVG_RISK_WEIGHT_1080D,"(3.5, 3.75]",3145,459,2686,0.012784,0.145946,-0.665724,0.007479
7,PA_AVG_RISK_WEIGHT_1080D,"(3.75, 4.0]",21063,2650,18413,0.085619,0.125813,-0.493985,0.025694
8,PA_AVG_RISK_WEIGHT_1080D,SPECIAL_-99999,44012,3199,40813,0.178905,0.072685,0.113680,0.002205
Totals,PA_AVG_RISK_WEIGHT_1080D,None,246008,19860,226148,1.000000,0.080729,NaN,0.078007


In [244]:
merge_plan = [
    [0,1],
    [2, 3],   
    [4],
    [5],
    [6,7],             
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_AVG_RISK_WEIGHT_1080D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [245]:
num_bins['PA_AVG_RISK_WEIGHT_1080D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_AVG_RISK_WEIGHT_1080D,"(0.999, 2.25]]",65034,3850,61184,0.264357,0.059200,0.333331,0.025564
2,PA_AVG_RISK_WEIGHT_1080D,"(2.25, 2.667]]",32546,2371,30175,0.132297,0.072851,0.111220,0.001562
4,PA_AVG_RISK_WEIGHT_1080D,"(2.667, 3.0]]",63345,5351,57994,0.257492,0.084474,-0.049426,0.000642
5,PA_AVG_RISK_WEIGHT_1080D,"(3.0, 3.5]]",16863,1980,14883,0.068547,0.117417,-0.415359,0.014075
6,PA_AVG_RISK_WEIGHT_1080D,"(3.5, 4.0]]",24208,3109,21099,0.098403,0.128429,-0.517557,0.032735
8,PA_AVG_RISK_WEIGHT_1080D,SPECIAL_-99999,44012,3199,40813,0.178905,0.072685,0.113680,0.002205
Totals,PA_AVG_RISK_WEIGHT_1080D,None,246008,19860,226148,1.000000,0.080729,NaN,0.078007


In [246]:
rows.append({
    'feature':'PA_AVG_RISK_WEIGHT_1080D',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.178905,
    'bad_rate_trend':'increasing',
    'iv':0.078007,
    'keep':'Yes',
    'comment':'Strong monotonic increasing trend'
})

##### PA_MAX_RISK_WEIGHT_1080D

In [247]:
num_bins['PA_MAX_RISK_WEIGHT_1080D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_MAX_RISK_WEIGHT_1080D,1.0,11342,645,10697,0.046104,0.056868,0.375986,0.005573
1,PA_MAX_RISK_WEIGHT_1080D,2.0,41185,2447,38738,0.167413,0.059415,0.329476,0.015842
2,PA_MAX_RISK_WEIGHT_1080D,3.0,87548,6349,81199,0.355875,0.072520,0.116124,0.004571
3,PA_MAX_RISK_WEIGHT_1080D,4.0,61921,7220,54701,0.251703,0.116600,-0.407455,0.049572
4,PA_MAX_RISK_WEIGHT_1080D,SPECIAL_-99999,44012,3199,40813,0.178905,0.072685,0.113680,0.002205
Totals,PA_MAX_RISK_WEIGHT_1080D,None,246008,19860,226148,1.000000,0.080729,NaN,0.077764


In [248]:
rows.append({
    'feature':'PA_MAX_RISK_WEIGHT_1080D',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.178905,
    'bad_rate_trend':'increasing',
    'iv':0.077764,
    'keep':'Yes',
    'comment':' monotonic increasing trend'
})

##### BUREAU_DAYS_CREDIT_ENDDATE_MEAN

In [249]:
num_bins['BUREAU_DAYS_CREDIT_ENDDATE_MEAN']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-41875.001, -1535.667]",10449,609,9840,0.042474,0.058283,0.349911,4.495190e-03
1,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-1535.667, -1199.667]",10447,575,9872,0.042466,0.055040,0.410606,6.035970e-03
2,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-1199.667, -992.0]",10455,531,9924,0.042499,0.050789,0.495467,8.495088e-03
3,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-992.0, -837.0]",10445,528,9917,0.042458,0.050551,0.500427,8.640236e-03
4,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-837.0, -702.571]",10443,604,9839,0.042450,0.057838,0.358053,4.688354e-03
5,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-702.571, -579.75]",10447,624,9823,0.042466,0.059730,0.323849,3.891446e-03
6,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-579.75, -465.896]",10447,612,9835,0.042466,0.058581,0.344488,4.365878e-03
7,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-465.896, -355.0]",10460,673,9787,0.042519,0.064340,0.244583,2.296575e-03
8,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-355.0, -245.199]",10435,702,9733,0.042417,0.067274,0.196862,1.514018e-03
9,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-245.199, -134.388]",10447,793,9654,0.042466,0.075907,0.066822,1.843865e-04


In [250]:
merge_plan = [
    [0, 1, 2, 3],                 # very old end dates → lowest risk
    [4, 5, 6],                    # still low risk
    [7, 8, 9, 10],                # transition zone
    [11, 12, 13, 14, 15],         # moderate risk
    [16, 17, 18, 19],             # high risk
    ]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="BUREAU_DAYS_CREDIT_ENDDATE_MEAN",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [251]:
num_bins['BUREAU_DAYS_CREDIT_ENDDATE_MEAN']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-41875.001, -837.0]]",41796,2243,39553,0.169897,0.053665,0.437345,0.027097
4,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-837.0, -465.896]]",31337,1840,29497,0.127382,0.058717,0.342041,0.012924
7,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-465.896, -28.0]]",41802,3009,38793,0.169921,0.071982,0.124150,0.002486
11,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(-28.0, 924.613]]",52225,4964,47261,0.212290,0.095050,-0.179008,0.007333
16,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,"(924.613, 31198.0]]",41790,4045,37745,0.169873,0.096793,-0.199111,0.007322
20,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,SPECIAL_-99999,37058,3759,33299,0.150637,0.101436,-0.251108,0.010554
Totals,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,None,246008,19860,226148,1.000000,0.080729,NaN,0.072750


In [252]:
rows.append({
    'feature':'BUREAU_DAYS_CREDIT_ENDDATE_MEAN',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.150637,
    'bad_rate_trend':'increasing',
    'iv':0.072750,
    'keep':'Yes',
    'comment':' monotonic increasing trend'
})

##### PA_MAX_RISK_WEIGHT_720D

In [253]:
num_bins['PA_MAX_RISK_WEIGHT_720D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_MAX_RISK_WEIGHT_720D,1.0,11286,630,10656,0.045877,0.055821,0.395677,0.006092
1,PA_MAX_RISK_WEIGHT_720D,2.0,43391,2682,40709,0.176380,0.061810,0.287404,0.012923
2,PA_MAX_RISK_WEIGHT_720D,3.0,75540,5821,69719,0.307063,0.077059,0.050519,0.000767
3,PA_MAX_RISK_WEIGHT_720D,4.0,44401,5464,38937,0.180486,0.123060,-0.468718,0.048255
4,PA_MAX_RISK_WEIGHT_720D,SPECIAL_-99999,71390,5263,66127,0.290194,0.073722,0.098394,0.002696
Totals,PA_MAX_RISK_WEIGHT_720D,None,246008,19860,226148,1.000000,0.080729,NaN,0.070734


In [254]:
rows.append({
    'feature':'PA_MAX_RISK_WEIGHT_720D',
    'n_bins':4,
    'n_special_bins':1,
    'special_bins_pct':0.290194,
    'bad_rate_trend':'increasing',
    'iv':0.070734,
    'keep':'Yes',
    'comment':' monotonic increasing trend'
})

##### PA_LOANS_REFUSED_RECENT_1080D

In [255]:
num_bins['PA_LOANS_REFUSED_RECENT_1080D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_LOANS_REFUSED_RECENT_1080D,"(-0.001, 1.0]",34345,3244,31101,0.139609,0.094453,-0.172049,0.004442
1,PA_LOANS_REFUSED_RECENT_1080D,"(1.0, 2.0]",13888,1583,12305,0.056453,0.113983,-0.381798,0.009658
2,PA_LOANS_REFUSED_RECENT_1080D,"(2.0, 3.0]",7097,924,6173,0.028849,0.130196,-0.533254,0.010254
3,PA_LOANS_REFUSED_RECENT_1080D,"(3.0, 4.0]",4106,528,3578,0.016691,0.128592,-0.519019,0.005587
4,PA_LOANS_REFUSED_RECENT_1080D,"(4.0, 5.0]",2344,334,2010,0.009528,0.142491,-0.637733,0.005057
5,PA_LOANS_REFUSED_RECENT_1080D,"(5.0, 6.0]",1520,213,1307,0.006179,0.140132,-0.618284,0.003058
6,PA_LOANS_REFUSED_RECENT_1080D,"(6.0, 58.0]",3183,547,2636,0.012939,0.171850,-0.859913,0.013661
7,PA_LOANS_REFUSED_RECENT_1080D,SPECIAL_-99999,179525,12487,167038,0.729753,0.069556,0.161051,0.017695
Totals,PA_LOANS_REFUSED_RECENT_1080D,None,246008,19860,226148,1.000000,0.080729,NaN,0.069412


In [256]:
merge_plan = [
    [2,3,4,5,6]
    ]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_LOANS_REFUSED_RECENT_1080D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [257]:
num_bins['PA_LOANS_REFUSED_RECENT_1080D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_LOANS_REFUSED_RECENT_1080D,"(-0.001, 1.0]",34345,3244,31101,0.139609,0.094453,-0.172049,0.004442
1,PA_LOANS_REFUSED_RECENT_1080D,"(1.0, 2.0]",13888,1583,12305,0.056453,0.113983,-0.381798,0.009658
2,PA_LOANS_REFUSED_RECENT_1080D,"(2.0, 58.0]]",18250,2546,15704,0.074185,0.139507,-0.613090,0.036023
7,PA_LOANS_REFUSED_RECENT_1080D,SPECIAL_-99999,179525,12487,167038,0.729753,0.069556,0.161051,0.017695
Totals,PA_LOANS_REFUSED_RECENT_1080D,None,246008,19860,226148,1.000000,0.080729,NaN,0.069412


In [258]:
rows.append({
    'feature':'PA_LOANS_REFUSED_RECENT_1080D',
    'n_bins':3,
    'n_special_bins':1,
    'special_bins_pct':0.729753,
    'bad_rate_trend':'increasing',
    'iv':0.069412,
    'keep':'Yes',
    'comment':' monotonic increasing trend'
})

##### PA_LOANS_REFUSED_RECENT_180D

In [259]:
num_bins['PA_LOANS_REFUSED_RECENT_180D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_LOANS_REFUSED_RECENT_180D,"(-0.001, 1.0]",60451,6631,53820,0.245728,0.109692,-0.338592,0.032472
1,PA_LOANS_REFUSED_RECENT_180D,"(1.0, 2.0]",3286,424,2862,0.013357,0.129032,-0.522939,0.004546
2,PA_LOANS_REFUSED_RECENT_180D,"(2.0, 26.0]",2746,318,2428,0.011162,0.115805,-0.399710,0.002109
3,PA_LOANS_REFUSED_RECENT_180D,SPECIAL_-99999,179525,12487,167038,0.729753,0.069556,0.161051,0.017695
Totals,PA_LOANS_REFUSED_RECENT_180D,None,246008,19860,226148,1.000000,0.080729,NaN,0.056822


In [260]:
rows.append({
    'feature':'PA_LOANS_REFUSED_RECENT_180D',
    'n_bins':3,
    'n_special_bins':1,
    'special_bins_pct':0.729753,
    'bad_rate_trend':'increasing',
    'iv':0.056822,
    'keep':'Drop',
    'comment':' monotonic increasing trend'
})

##### PA_LOANS_REFUSED_RECENT_90D

In [261]:
num_bins['PA_LOANS_REFUSED_RECENT_90D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_LOANS_REFUSED_RECENT_90D,"(-0.001, 1.0]",64493,7144,57349,0.262158,0.110772,-0.349599,0.037102
1,PA_LOANS_REFUSED_RECENT_90D,"(1.0, 26.0]",1990,229,1761,0.008089,0.115075,-0.392567,0.001470
2,PA_LOANS_REFUSED_RECENT_90D,SPECIAL_-99999,179525,12487,167038,0.729753,0.069556,0.161051,0.017695
Totals,PA_LOANS_REFUSED_RECENT_90D,None,246008,19860,226148,1.000000,0.080729,NaN,0.056267


In [262]:
rows.append({
    'feature':'PA_LOANS_REFUSED_RECENT_90D',
    'n_bins':2,
    'n_special_bins':1,
    'special_bins_pct':0.729753,
    'bad_rate_trend':'increasing',
    'iv':0.056267,
    'keep':'Drop',
    'comment':' monotonic increasing trend'
})

##### BUREAU_DAYS_CREDIT_ENDDATE_MAX

In [263]:
num_bins['BUREAU_DAYS_CREDIT_ENDDATE_MAX']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(-41875.001, -1074.0]",10450,666,9784,0.042478,0.063732,0.254732,2.478278e-03
1,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(-1074.0, -541.0]",10450,671,9779,0.042478,0.064211,0.246741,2.332962e-03
2,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(-541.0, -227.0]",10453,611,9842,0.042490,0.058452,0.346835,4.423820e-03
3,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(-227.0, -23.0]",10489,603,9886,0.042637,0.057489,0.364476,4.866552e-03
4,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(-23.0, 126.0]",10446,675,9771,0.042462,0.064618,0.239979,2.212203e-03
5,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(126.0, 270.0]",10428,703,9725,0.042389,0.067415,0.194616,1.480062e-03
6,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(270.0, 441.0]",10438,735,9703,0.042430,0.070416,0.147838,8.717214e-04
7,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(441.0, 615.0]",10474,700,9774,0.042576,0.066832,0.203919,1.625795e-03
8,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(615.0, 766.0]",10421,749,9672,0.042360,0.071874,0.125769,6.356958e-04
9,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(766.0, 910.0]",10476,759,9717,0.042584,0.072451,0.117148,5.564451e-04


In [264]:
merge_plan = [
    [0, 1, 2, 3],                 # very old end dates → lowest risk
    [4, 5, 6],                    # still low risk
    [7, 8, 9, 10],                # transition zone
    [11, 12, 13, 14,15,16],         # moderate risk
    [ 17, 18, 19],             # high risk
    ]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="BUREAU_DAYS_CREDIT_ENDDATE_MAX",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [265]:
num_bins['BUREAU_DAYS_CREDIT_ENDDATE_MAX']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(-41875.001, -23.0]]",41842,2551,39291,0.170084,0.060967,0.302028,0.013679
4,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(-23.0, 441.0]]",31312,2113,29199,0.127280,0.067482,0.193544,0.004397
7,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(441.0, 1029.0]]",41853,3067,38786,0.170129,0.073280,0.104877,0.001791
11,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(1029.0, 9592.3]]",62600,5354,57246,0.254463,0.085527,-0.062968,0.001036
17,BUREAU_DAYS_CREDIT_ENDDATE_MAX,"(9592.3, 31199.0]]",31343,3016,28327,0.127406,0.096226,-0.192598,0.005124
20,BUREAU_DAYS_CREDIT_ENDDATE_MAX,SPECIAL_-99999,37058,3759,33299,0.150637,0.101436,-0.251108,0.010554
Totals,BUREAU_DAYS_CREDIT_ENDDATE_MAX,None,246008,19860,226148,1.000000,0.080729,NaN,0.054836


In [266]:
rows.append({
    'feature':'BUREAU_DAYS_CREDIT_ENDDATE_MAX',
    'n_bins':5,
    'n_special_bins':1,
    'special_bins_pct':0.150637,
    'bad_rate_trend':'increasing',
    'iv':0.054836,
    'keep':'Yes',
    'comment':' monotonic increasing trend'
})

##### B_AMT_CREDIT_SUM_DEBT_SUM

In [267]:
num_bins['B_AMT_CREDIT_SUM_DEBT_SUM']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_AMT_CREDIT_SUM_DEBT_SUM,"(-6981558.001, 0.0]",62555,3517,59038,0.254280,0.056223,0.388091,0.032588
1,B_AMT_CREDIT_SUM_DEBT_SUM,"(0.0, 1394.55]",674,47,627,0.002740,0.069733,0.158317,0.000064
2,B_AMT_CREDIT_SUM_DEBT_SUM,"(1394.55, 36418.718]",10539,869,9670,0.042840,0.082456,-0.023042,0.000023
3,B_AMT_CREDIT_SUM_DEBT_SUM,"(36418.718, 73575.9]",10538,821,9717,0.042836,0.077909,0.038627,0.000063
4,B_AMT_CREDIT_SUM_DEBT_SUM,"(73575.9, 118705.5]",10539,821,9718,0.042840,0.077901,0.038730,0.000063
5,B_AMT_CREDIT_SUM_DEBT_SUM,"(118705.5, 169152.75]",10537,845,9692,0.042832,0.080194,0.007237,0.000002
6,B_AMT_CREDIT_SUM_DEBT_SUM,"(169152.75, 229127.445]",10538,891,9647,0.042836,0.084551,-0.050424,0.000111
7,B_AMT_CREDIT_SUM_DEBT_SUM,"(229127.445, 305264.7]",10538,873,9665,0.042836,0.082843,-0.028151,0.000034
8,B_AMT_CREDIT_SUM_DEBT_SUM,"(305264.7, 398690.55]",10538,937,9601,0.042836,0.088916,-0.105543,0.000499
9,B_AMT_CREDIT_SUM_DEBT_SUM,"(398690.55, 511015.977]",10539,921,9618,0.042840,0.087390,-0.086550,0.000333


In [268]:
merge_plan = [

    [0],                      
    [1,2, 3, 4],                
    [5,6],
    [7,8,9],                   
    [10, 11],                 
    [12, 13, 14, 15],            
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_AMT_CREDIT_SUM_DEBT_SUM",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [269]:
num_bins['B_AMT_CREDIT_SUM_DEBT_SUM']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_AMT_CREDIT_SUM_DEBT_SUM,"(-6981558.001, 0.0]]",62555,3517,59038,0.254280,0.056223,0.388091,0.032588
1,B_AMT_CREDIT_SUM_DEBT_SUM,"(0.0, 118705.5]]",32290,2558,29732,0.131256,0.079220,0.020516,0.000055
5,B_AMT_CREDIT_SUM_DEBT_SUM,"(118705.5, 229127.445]]",21075,1736,19339,0.085668,0.082372,-0.021942,0.000042
7,B_AMT_CREDIT_SUM_DEBT_SUM,"(229127.445, 511015.977]]",31615,2731,28884,0.128512,0.086383,-0.073862,0.000723
10,B_AMT_CREDIT_SUM_DEBT_SUM,"(511015.977, 856315.0]]",21076,1915,19161,0.085672,0.090862,-0.129323,0.001513
12,B_AMT_CREDIT_SUM_DEBT_SUM,"(856315.0, 334498340.0]]",42153,3836,38317,0.171348,0.091002,-0.131018,0.003108
16,B_AMT_CREDIT_SUM_DEBT_SUM,SPECIAL_-99999,35244,3567,31677,0.143264,0.101209,-0.248616,0.009829
Totals,B_AMT_CREDIT_SUM_DEBT_SUM,None,246008,19860,226148,1.000000,0.080729,NaN,0.050578


In [270]:

rows.append({
    'feature': 'B_AMT_CREDIT_SUM_DEBT_SUM',
    'n_bins': 6,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.143264,      
    'bad_rate_trend': 'increasing',    
    'iv': 0.050578,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend ',
})

##### PA_RATIO_HC_REFUSED_LOANS

In [271]:
num_bins['PA_RATIO_HC_REFUSED_LOANS']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_HC_REFUSED_LOANS,"(-0.001, 0.2]",28228,2849,25379,0.114744,0.100928,-0.245528,0.007668
1,PA_RATIO_HC_REFUSED_LOANS,"(0.2, 0.5]",9186,1166,8020,0.037340,0.126932,-0.504123,0.011720
2,PA_RATIO_HC_REFUSED_LOANS,"(0.5, 0.667]",2925,406,2519,0.011890,0.138803,-0.607218,0.005650
3,PA_RATIO_HC_REFUSED_LOANS,"(0.667, 1.0]",39995,3885,36110,0.162576,0.097137,-0.203035,0.007298
4,PA_RATIO_HC_REFUSED_LOANS,SPECIAL_-99999,165674,11554,154120,0.673450,0.069739,0.158218,0.015779
Totals,PA_RATIO_HC_REFUSED_LOANS,None,246008,19860,226148,1.000000,0.080729,NaN,0.048114


In [272]:
merge_plan = [
    [0],
    [1,2,3]         
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_RATIO_HC_REFUSED_LOANS",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [273]:
num_bins['PA_RATIO_HC_REFUSED_LOANS']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_HC_REFUSED_LOANS,"(-0.001, 0.2]]",28228,2849,25379,0.114744,0.100928,-0.245528,0.007668
1,PA_RATIO_HC_REFUSED_LOANS,"(0.2, 1.0]]",52106,5457,46649,0.211806,0.104729,-0.286730,0.019640
4,PA_RATIO_HC_REFUSED_LOANS,SPECIAL_-99999,165674,11554,154120,0.673450,0.069739,0.158218,0.015779
Totals,PA_RATIO_HC_REFUSED_LOANS,None,246008,19860,226148,1.000000,0.080729,NaN,0.048114


In [274]:

rows.append({
    'feature': 'PA_RATIO_HC_REFUSED_LOANS',
    'n_bins': 2,                       
    'n_special_bins': 2,               
    'special_bins_pct':0.67345,      
    'bad_rate_trend': 'increasing',    
    'iv': 0.049415,                    
    'keep': 'Yes',                     
    'comment': ' little diff monotonic trend  with 77777 and 99999 special values',
})

##### B_TOTAL_ANNUITY_TO_DEBT

In [275]:
'''merge_plan = [
    [0,1,2],
    [3,4,5,6],

]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_TOTAL_ANNUITY_TO_DEBT",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)
rows.append({
    'feature': 'B_TOTAL_ANNUITY_TO_DEBT',
    'n_bins': 2,                       
    'n_special_bins': 2,               
    'special_bins_pct':0.393378,      
    'bad_rate_trend': 'increasing',    
    'iv': 0.047317,                    
    'keep': 'Yes',                     
    'comment': ' monotonic trend with 77777 and 99999 special values',
})'''

'merge_plan = [\n    [0,1,2],\n    [3,4,5,6],\n\n]\n\nwoe_mappings = run_merge_plan_and_collect_mappings(\n    feature="B_TOTAL_ANNUITY_TO_DEBT",\n    merge_plan=merge_plan,\n    num_bins=num_bins\n)\n\nall_woe_mapping_num.update(woe_mappings)\nrows.append({\n    \'feature\': \'B_TOTAL_ANNUITY_TO_DEBT\',\n    \'n_bins\': 2,                       \n    \'n_special_bins\': 2,               \n    \'special_bins_pct\':0.393378,      \n    \'bad_rate_trend\': \'increasing\',    \n    \'iv\': 0.047317,                    \n    \'keep\': \'Yes\',                     \n    \'comment\': \' monotonic trend with 77777 and 99999 special values\',\n})'

##### PA_RATIO_HIGH_RISK_YIELD_LOANS_720D

In [276]:
num_bins['PA_RATIO_HIGH_RISK_YIELD_LOANS_720D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_HIGH_RISK_YIELD_LOANS_720D,"(-0.001, 0.167]",151334,11316,140018,0.615159,0.074775,0.083071,0.004100
1,PA_RATIO_HIGH_RISK_YIELD_LOANS_720D,"(0.167, 0.333]",14366,1765,12601,0.058396,0.122860,-0.466857,0.015477
2,PA_RATIO_HIGH_RISK_YIELD_LOANS_720D,"(0.333, 0.5]",9689,1187,8502,0.039385,0.122510,-0.463610,0.010280
3,PA_RATIO_HIGH_RISK_YIELD_LOANS_720D,"(0.5, 1.0]",12924,1541,11383,0.052535,0.119236,-0.432793,0.011797
4,PA_RATIO_HIGH_RISK_YIELD_LOANS_720D,SPECIAL_-99999,57695,4051,53644,0.234525,0.070214,0.150924,0.005015
Totals,PA_RATIO_HIGH_RISK_YIELD_LOANS_720D,None,246008,19860,226148,1.000000,0.080729,NaN,0.046670


In [277]:
merge_plan = [
    [0,1],
    [2,3]

]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_RATIO_HIGH_RISK_YIELD_LOANS_720D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [278]:
num_bins['PA_RATIO_HIGH_RISK_YIELD_LOANS_720D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_HIGH_RISK_YIELD_LOANS_720D,"(-0.001, 0.333]]",165700,13081,152619,0.673555,0.078944,0.024302,0.000394
2,PA_RATIO_HIGH_RISK_YIELD_LOANS_720D,"(0.333, 1.0]]",22613,2728,19885,0.091920,0.120639,-0.446085,0.022051
4,PA_RATIO_HIGH_RISK_YIELD_LOANS_720D,SPECIAL_-99999,57695,4051,53644,0.234525,0.070214,0.150924,0.005015
Totals,PA_RATIO_HIGH_RISK_YIELD_LOANS_720D,None,246008,19860,226148,1.000000,0.080729,NaN,0.046670


In [279]:
rows.append({
    'feature': 'PA_RATIO_HIGH_RISK_YIELD_LOANS_720D',
    'n_bins': 2,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.234525,      
    'bad_rate_trend': 'increasing',    
    'iv': 0.046670,                    
    'keep': 'Yes',                     
    'comment': ' monotonic trend with 99999 special values',
})

##### PA_RATIO_CREDIT_APPLICATION_Cash

In [280]:
num_bins['PA_RATIO_CREDIT_APPLICATION_Cash']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_CREDIT_APPLICATION_Cash,"(0.999, 1.007]",11933,880,11053,0.048507,0.073745,0.098053,4.476035e-04
1,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.007, 1.033]",5966,327,5639,0.024251,0.054811,0.415020,3.515111e-03
2,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.033, 1.053]",5967,334,5633,0.024255,0.055975,0.392774,3.177837e-03
3,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.053, 1.066]",10640,604,10036,0.043251,0.056767,0.377878,5.277111e-03
4,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.066, 1.069]",1292,79,1213,0.005252,0.061146,0.298922,4.142758e-04
5,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.069, 1.081]",5967,369,5598,0.024255,0.061840,0.286886,1.771131e-03
6,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.081, 1.092]",6483,436,6047,0.026353,0.067253,0.197193,9.436590e-04
7,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.092, 1.101]",5449,439,5010,0.022150,0.080565,0.002210,1.080603e-07
8,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.101, 1.111]",5967,458,5509,0.024255,0.076755,0.054787,7.115345e-05
9,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.111, 1.12]",5967,484,5483,0.024255,0.081113,-0.005159,6.470163e-07


In [281]:
merge_plan = [

    [0, 1, 2, 3, 4],
    [5,6, 7, 8, 9,],
    [10,11, 12],
    [13,14, 15],
    [16, 17, 18],
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_RATIO_CREDIT_APPLICATION_Cash",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [282]:
num_bins['PA_RATIO_CREDIT_APPLICATION_Cash']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_CREDIT_APPLICATION_Cash,"(0.999, 1.069]]",35798,2224,33574,0.145516,0.062126,0.281963,0.010285
5,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.069, 1.12]]",29833,2186,27647,0.121268,0.073275,0.104962,0.001279
10,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.12, 1.152]]",17898,1594,16304,0.072754,0.089060,-0.107318,0.000877
13,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.152, 1.205]]",17900,1851,16049,0.072762,0.103408,-0.272561,0.006061
16,PA_RATIO_CREDIT_APPLICATION_Cash,"(1.205, 1.66]]",17898,2220,15678,0.072754,0.124036,-0.477731,0.020283
19,PA_RATIO_CREDIT_APPLICATION_Cash,SPECIAL_-99999,126681,9785,116896,0.514947,0.077241,0.047952,0.001161
Totals,PA_RATIO_CREDIT_APPLICATION_Cash,None,246008,19860,226148,1.000000,0.080729,NaN,0.045698


In [283]:
rows.append({
    'feature': 'PA_RATIO_CREDIT_APPLICATION_Cash',
    'n_bins': 5,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.514947,      
    'bad_rate_trend': 'increasing',    
    'iv': 0.045698,                    
    'keep': 'Yes',                     
    'comment': ' monotonic trend with 99999 special values',
})

##### CREDIT_GOODS_DIFF

In [284]:
num_bins['CREDIT_GOODS_DIFF_AMT']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CREDIT_GOODS_DIFF_AMT,"(-765000.001, 0.0]",86802,5862,80940,0.352842,0.067533,0.192735,0.012092
1,CREDIT_GOODS_DIFF_AMT,"(0.0, 17820.0]",12272,643,11629,0.049885,0.052396,0.462631,0.008811
2,CREDIT_GOODS_DIFF_AMT,"(17820.0, 29403.0]",11531,820,10711,0.046872,0.071113,0.137240,0.000834
3,CREDIT_GOODS_DIFF_AMT,"(29403.0, 39204.0]",12558,977,11581,0.051047,0.077799,0.040152,0.000081
4,CREDIT_GOODS_DIFF_AMT,"(39204.0, 52033.5]",12051,1175,10876,0.048986,0.097502,-0.207192,0.002294
5,CREDIT_GOODS_DIFF_AMT,"(52033.5, 60588.0]",12403,1106,11297,0.050417,0.089172,-0.108695,0.000623
6,CREDIT_GOODS_DIFF_AMT,"(60588.0, 74844.0]",12661,1311,11350,0.051466,0.103546,-0.274054,0.004337
7,CREDIT_GOODS_DIFF_AMT,"(74844.0, 89100.0]",13409,1292,12117,0.054506,0.096353,-0.194064,0.002227
8,CREDIT_GOODS_DIFF_AMT,"(89100.0, 99972.0]",10673,1187,9486,0.043385,0.111215,-0.354094,0.006311
9,CREDIT_GOODS_DIFF_AMT,"(99972.0, 117612.0]",12745,1043,11702,0.051807,0.081836,-0.014823,0.000011


In [285]:
merge_plan = [

    [0, 1],
    [2, 3],
    [4, 5],
    [6, 7,8,9, 10, 11, 12, 13],
]



woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CREDIT_GOODS_DIFF_AMT",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [286]:
num_bins['CREDIT_GOODS_DIFF_AMT']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CREDIT_GOODS_DIFF_AMT,"(-765000.001, 17820.0]]",99074,6505,92569,0.402727,0.065658,0.222901,0.018230
2,CREDIT_GOODS_DIFF_AMT,"(17820.0, 39204.0]]",24089,1797,22292,0.097920,0.074598,0.085627,0.000693
4,CREDIT_GOODS_DIFF_AMT,"(39204.0, 60588.0]]",24454,2281,22173,0.099403,0.093277,-0.158221,0.002659
6,CREDIT_GOODS_DIFF_AMT,"(60588.0, 540000.0]]",98170,9260,88910,0.399052,0.094326,-0.170561,0.012470
14,CREDIT_GOODS_DIFF_AMT,SPECIAL_-99999,221,17,204,0.000898,0.076923,0.052425,0.000002
Totals,CREDIT_GOODS_DIFF_AMT,None,246008,19860,226148,1.000000,0.080729,NaN,0.043182


In [287]:
rows.append({
    'feature':'CREDIT_GOODS_DIFF_AMT',
    'n_bins': 4,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.000898,      
    'bad_rate_trend': 'increasing',    
    'iv': 0.043182,                    
    'keep': 'Yes',                     
    'comment': ' monotonic trend with 99999 special values',
})

##### PA_RATIO_CREDIT_APPLICATION_POS

In [288]:
num_bins['PA_RATIO_CREDIT_APPLICATION_POS']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_CREDIT_APPLICATION_POS,"(0.0906, 0.792]",11282,660,10622,0.045860,0.058500,0.345961,0.004752
1,PA_RATIO_CREDIT_APPLICATION_POS,"(0.792, 0.852]",11281,728,10553,0.045856,0.064533,0.241382,0.002416
2,PA_RATIO_CREDIT_APPLICATION_POS,"(0.852, 0.894]",11282,759,10523,0.045860,0.067275,0.196835,0.001636
3,PA_RATIO_CREDIT_APPLICATION_POS,"(0.894, 0.9]",13177,879,12298,0.053563,0.066707,0.205925,0.002084
4,PA_RATIO_CREDIT_APPLICATION_POS,"(0.9, 0.924]",9386,640,8746,0.038153,0.068187,0.182402,0.001176
5,PA_RATIO_CREDIT_APPLICATION_POS,"(0.924, 0.944]",11297,803,10494,0.045921,0.071081,0.137722,0.000822
6,PA_RATIO_CREDIT_APPLICATION_POS,"(0.944, 0.959]",11265,846,10419,0.045791,0.075100,0.078385,0.000272
7,PA_RATIO_CREDIT_APPLICATION_POS,"(0.959, 0.974]",11282,864,10418,0.045860,0.076582,0.057236,0.000147
8,PA_RATIO_CREDIT_APPLICATION_POS,"(0.974, 0.986]",11281,852,10429,0.045856,0.075525,0.072277,0.000232
9,PA_RATIO_CREDIT_APPLICATION_POS,"(0.986, 0.996]",11283,891,10392,0.045864,0.078968,0.023965,0.000026


In [289]:
merge_plan = [
    [0, 1, 2, 3],         # bin3 spikes UP vs bin2 → absorb into one positive block
    [4, 5],               # clean descent
    [6, 7, 8, 9, 10, 11, 12],  # bins 8, 10, 11 all spike → collapse entire mid-zone
    [13, 14],             # clean negative descent starts here
    [15,16],
    [17,18],
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_RATIO_CREDIT_APPLICATION_POS",   # ← fixed feature name
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [290]:
num_bins['PA_RATIO_CREDIT_APPLICATION_POS']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_CREDIT_APPLICATION_POS,"(0.0906, 0.9]]",47022,3026,43996,0.191140,0.064353,0.244375,0.010307
4,PA_RATIO_CREDIT_APPLICATION_POS,"(0.9, 0.944]]",20683,1443,19240,0.084075,0.069767,0.157785,0.001959
6,PA_RATIO_CREDIT_APPLICATION_POS,"(0.944, 1.022]]",78954,6024,72930,0.320941,0.076298,0.061267,0.001174
13,PA_RATIO_CREDIT_APPLICATION_POS,"(1.022, 1.059]]",22563,1886,20677,0.091717,0.083588,-0.037918,0.000134
15,PA_RATIO_CREDIT_APPLICATION_POS,"(1.059, 1.106]]",22567,2141,20426,0.091733,0.094873,-0.176946,0.003094
17,PA_RATIO_CREDIT_APPLICATION_POS,"(1.106, 1.552]]",22559,2776,19783,0.091700,0.123055,-0.468670,0.024512
19,PA_RATIO_CREDIT_APPLICATION_POS,SPECIAL_-99999,31660,2564,29096,0.128695,0.080985,-0.003450,0.000002
Totals,PA_RATIO_CREDIT_APPLICATION_POS,None,246008,19860,226148,1.000000,0.080729,NaN,0.043080


In [291]:

rows.append({
    'feature':'PA_RATIO_CREDIT_APPLICATION_POS',
    'n_bins': 6,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.128695,      
    'bad_rate_trend': 'increasing',    
    'iv': 0.043080,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend with 99999 special values',
})

##### PA_FLAG_NEW_CLIENT_1080D

In [292]:
num_bins['PA_FLAG_NEW_CLIENT_1080D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_FLAG_NEW_CLIENT_1080D,0.0,151830,10735,141095,0.617175,0.070704,0.143442,0.011959
1,PA_FLAG_NEW_CLIENT_1080D,1.0,81004,8334,72670,0.329274,0.102884,-0.266897,0.026236
2,PA_FLAG_NEW_CLIENT_1080D,SPECIAL_-99999,13174,791,12383,0.053551,0.060043,0.318300,0.004751
Totals,PA_FLAG_NEW_CLIENT_1080D,None,246008,19860,226148,1.000000,0.080729,NaN,0.042946


In [293]:

rows.append({
    'feature':'PA_FLAG_NEW_CLIENT_1080D',
    'n_bins': 2,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.053551,      
    'bad_rate_trend': 'increasing',    
    'iv': 0.042946,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend with 99999 special values',
})

##### B_MIN_REPAYMENT_DAYS_DIFF

In [294]:
num_bins['B_MIN_REPAYMENT_DAYS_DIFF']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_MIN_REPAYMENT_DAYS_DIFF,"(-40187.001, -9872.0]",9148,558,8590,0.037186,0.060997,0.301513,0.002981
1,B_MIN_REPAYMENT_DAYS_DIFF,"(-9872.0, -1645.0]",9181,684,8497,0.037320,0.074502,0.087029,0.000273
2,B_MIN_REPAYMENT_DAYS_DIFF,"(-1645.0, -1276.0]",9116,672,8444,0.037056,0.073717,0.098471,0.000345
3,B_MIN_REPAYMENT_DAYS_DIFF,"(-1276.0, -942.0]",9220,690,8530,0.037478,0.074837,0.082171,0.000244
4,B_MIN_REPAYMENT_DAYS_DIFF,"(-942.0, -715.0]",9081,618,8463,0.036913,0.068054,0.184489,0.001163
5,B_MIN_REPAYMENT_DAYS_DIFF,"(-715.0, -529.0]",9180,606,8574,0.037316,0.066013,0.217128,0.001607
6,B_MIN_REPAYMENT_DAYS_DIFF,"(-529.0, -365.0]",9229,611,8618,0.037515,0.066204,0.214029,0.001571
7,B_MIN_REPAYMENT_DAYS_DIFF,"(-365.0, -259.0]",9050,631,8419,0.036787,0.069724,0.158458,0.000864
8,B_MIN_REPAYMENT_DAYS_DIFF,"(-259.0, -178.0]",9223,633,8590,0.037491,0.068633,0.175402,0.001072
9,B_MIN_REPAYMENT_DAYS_DIFF,"(-178.0, -109.0]",9066,611,8455,0.036852,0.067395,0.194934,0.001291


In [295]:
merge_plan = [
    [0, 1, 2, 3,4, 5, 6, 7,8, 9, 10, 11, 12],  # near on-time
    [13, 14],
    [15,16],            # strong late
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_MIN_REPAYMENT_DAYS_DIFF",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)


In [296]:
num_bins['B_MIN_REPAYMENT_DAYS_DIFF']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_MIN_REPAYMENT_DAYS_DIFF,"(-40187.001, -28.0]]",121174,8283,112891,0.492561,0.068356,0.179736,0.014760
13,B_MIN_REPAYMENT_DAYS_DIFF,"(-28.0, 0.0]]",48814,3808,45006,0.198424,0.078010,0.037210,0.000270
15,B_MIN_REPAYMENT_DAYS_DIFF,"(0.0, 41448.0]]",12942,1134,11808,0.052608,0.087622,-0.089456,0.000437
17,B_MIN_REPAYMENT_DAYS_DIFF,SPECIAL_-99999,63078,6635,56443,0.256406,0.105187,-0.291609,0.024642
Totals,B_MIN_REPAYMENT_DAYS_DIFF,None,246008,19860,226148,1.000000,0.080729,NaN,0.042433


In [297]:
rows.append({
    'feature': 'B_MIN_REPAYMENT_DAYS_DIFF',
    'n_bins': 3,
    'n_special_bins': 1,
    'special_bins_pct': 0.256406,
    'bad_rate_trend': 'increasing',
    'iv':0.042433,
    'keep': 'Yes',
    'comment': 'clear monotonic trend'
})

##### EMPLOYMENT_GAP 

In [298]:
num_bins['EMPLOYMENT_GAP']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,EMPLOYMENT_GAP,"(17.929, 20.27]",10101,813,9288,0.041060,0.080487,0.003265,4.372187e-07
1,EMPLOYMENT_GAP,"(20.27, 21.83]",10117,896,9221,0.041125,0.088564,-0.101184,4.393016e-04
2,EMPLOYMENT_GAP,"(21.83, 23.17]",10070,907,9163,0.040934,0.090070,-0.119696,6.166682e-04
3,EMPLOYMENT_GAP,"(23.17, 24.5]",10118,927,9191,0.041129,0.091619,-0.138456,8.356083e-04
4,EMPLOYMENT_GAP,"(24.5, 25.81]",10069,955,9114,0.040930,0.094846,-0.176626,1.375136e-03
5,EMPLOYMENT_GAP,"(25.81, 27.09]",10108,1000,9108,0.041088,0.098932,-0.223329,2.250698e-03
6,EMPLOYMENT_GAP,"(27.09, 28.43]",10070,947,9123,0.040934,0.094042,-0.167227,1.227940e-03
7,EMPLOYMENT_GAP,"(28.43, 29.84]",10121,988,9133,0.041141,0.097619,-0.208515,1.952365e-03
8,EMPLOYMENT_GAP,"(29.84, 31.26]",10079,906,9173,0.040970,0.089890,-0.117502,5.942531e-04
9,EMPLOYMENT_GAP,"(31.26, 32.74]",10089,909,9180,0.041011,0.090098,-0.120045,6.215317e-04


In [299]:
merge_plan = [
    [0, 1, 2, 3, 4, 5, 6, 7],
    [ 8, 9, 10, 11, 12],
    [13, 14],  # collapse entire U-dip zone into one stable negative block
    [15, 16, 17],
    [18,19],               # strong positive tail — keep separate
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="EMPLOYMENT_GAP",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [300]:
num_bins['EMPLOYMENT_GAP']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,EMPLOYMENT_GAP,"(17.929, 29.84]]",80774,7433,73341,0.328339,0.092022,-0.143292,0.007160
8,EMPLOYMENT_GAP,"(29.84, 37.36]]",50490,4462,46028,0.205237,0.088374,-0.098829,0.002089
13,EMPLOYMENT_GAP,"(37.36, 40.96]]",20156,1714,18442,0.081932,0.085037,-0.056681,0.000270
15,EMPLOYMENT_GAP,"(40.96, 47.98]]",30266,2369,27897,0.123029,0.078273,0.033569,0.000137
18,EMPLOYMENT_GAP,"(47.98, 68.21]]",20179,1512,18667,0.082026,0.074929,0.080842,0.000518
20,EMPLOYMENT_GAP,SPECIAL_-99999,44143,2370,41773,0.179437,0.053689,0.436878,0.028563
Totals,EMPLOYMENT_GAP,None,246008,19860,226148,1.000000,0.080729,NaN,0.040608


In [301]:
rows.append({
    'feature': 'EMPLOYMENT_GAP',
    'n_bins': 5,
    'n_special_bins': 1,
    'special_bins_pct': 0.179437,
    'bad_rate_trend': 'decreasing',
    'iv':0.040608,
    'keep': 'Yes',
    'comment': 'clear monotonic trend'
})

##### CREDIT_PER_YEAR_AGE

In [302]:
num_bins['CREDIT_PER_YEAR_AGE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CREDIT_PER_YEAR_AGE,"(660.017, 3165.136]",12948,618,12330,0.052632,0.047729,0.560820,0.013125
1,CREDIT_PER_YEAR_AGE,"(3165.136, 4187.015]",12948,722,12226,0.052632,0.055762,0.396813,0.007027
2,CREDIT_PER_YEAR_AGE,"(4187.015, 5118.856]",12948,860,12088,0.052632,0.066420,0.210554,0.002137
3,CREDIT_PER_YEAR_AGE,"(5118.856, 6091.059]",12948,1028,11920,0.052632,0.079395,0.018121,0.000017
4,CREDIT_PER_YEAR_AGE,"(6091.059, 7123.46]",12947,1039,11908,0.052628,0.080250,0.006470,0.000002
5,CREDIT_PER_YEAR_AGE,"(7123.46, 8100.81]",12953,1126,11827,0.052653,0.086930,-0.080768,0.000355
6,CREDIT_PER_YEAR_AGE,"(8100.81, 9058.987]",12944,1141,11803,0.052616,0.088149,-0.096033,0.000505
7,CREDIT_PER_YEAR_AGE,"(9058.987, 10171.629]",12946,1173,11773,0.052624,0.090607,-0.126238,0.000884
8,CREDIT_PER_YEAR_AGE,"(10171.629, 11366.507]",12949,1189,11760,0.052636,0.091822,-0.140891,0.001108
9,CREDIT_PER_YEAR_AGE,"(11366.507, 12603.02]",12947,1246,11701,0.052628,0.096239,-0.192746,0.002120


In [303]:
merge_plan = [
    [0],
    [1, 2],
    [3, 4, 5],
    [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18],  # absorb entire tail
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CREDIT_PER_YEAR_AGE",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [304]:
num_bins['CREDIT_PER_YEAR_AGE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CREDIT_PER_YEAR_AGE,"(660.017, 3165.136]]",12948,618,12330,0.052632,0.047729,0.560820,0.013125
1,CREDIT_PER_YEAR_AGE,"(3165.136, 5118.856]]",25896,1582,24314,0.105265,0.061091,0.299880,0.008353
3,CREDIT_PER_YEAR_AGE,"(5118.856, 8100.81]]",38848,3193,35655,0.157914,0.082192,-0.019554,0.000061
6,CREDIT_PER_YEAR_AGE,"(8100.81, 130183.22]]",168316,14467,153849,0.684189,0.085951,-0.068381,0.003292
Totals,CREDIT_PER_YEAR_AGE,None,246008,19860,226148,1.000000,0.080729,NaN,0.040103


In [305]:
rows.append({
    'feature': 'CREDIT_PER_YEAR_AGE',
    'n_bins': 4,
    'n_special_bins': 0,
    'special_bins_pct': 0,
    'bad_rate_trend': 'increasing',
    'iv':0.040103,
    'keep': 'Yes',
    'comment': 'clear monotonic trend'
})

##### B_CREDIT_DURATION_MAX

In [306]:
num_bins['B_CREDIT_DURATION_MAX']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_CREDIT_DURATION_MAX,"(-40270.001, 242.0]",10534,849,9685,0.042820,0.080596,0.001792,1.374629e-07
1,B_CREDIT_DURATION_MAX,"(242.0, 364.0]",10475,691,9784,0.042580,0.065967,0.217882,1.845492e-03
2,B_CREDIT_DURATION_MAX,"(364.0, 368.0]",10592,820,9772,0.043056,0.077417,0.045490,8.741458e-05
3,B_CREDIT_DURATION_MAX,"(368.0, 702.0]",10190,694,9496,0.041421,0.068106,0.183672,1.294077e-03
4,B_CREDIT_DURATION_MAX,"(702.0, 741.0]",10458,720,9738,0.042511,0.068847,0.172058,1.171115e-03
5,B_CREDIT_DURATION_MAX,"(741.0, 1096.0]",18943,1426,17517,0.077002,0.075278,0.075817,4.287808e-04
6,B_CREDIT_DURATION_MAX,"(1096.0, 1098.0]",2640,203,2437,0.010731,0.076894,0.052835,2.930118e-05
7,B_CREDIT_DURATION_MAX,"(1098.0, 1259.0]",9782,672,9110,0.039763,0.068698,0.174388,1.124189e-03
8,B_CREDIT_DURATION_MAX,"(1259.0, 1461.0]",10445,753,9692,0.042458,0.072092,0.122509,6.053754e-04
9,B_CREDIT_DURATION_MAX,"(1461.0, 1826.0]",40231,3200,37031,0.163535,0.079541,0.016123,4.222274e-05


In [307]:
merge_plan = [
    [0, 1, 2, 3, 4],        # very short credit durations → low risk
    [5, 6, 7, 8,9, 10],
    [11,12,13, 14],
    [15,16],           # long durations → medium risk
                    # extreme/special durations → high risk
    ]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_CREDIT_DURATION_MAX",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)

In [308]:
num_bins['B_CREDIT_DURATION_MAX']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_CREDIT_DURATION_MAX,"(-40270.001, 741.0]]",52249,3774,48475,0.212387,0.072231,0.120431,0.002929
5,B_CREDIT_DURATION_MAX,"(741.0, 1827.0]]",94654,7098,87556,0.384760,0.074989,0.079984,0.002380
11,B_CREDIT_DURATION_MAX,"(1827.0, 27682.0]]",41153,3139,38014,0.167283,0.076276,0.061568,0.000618
15,B_CREDIT_DURATION_MAX,"(27682.0, 34105.0]]",20894,2090,18804,0.084932,0.100029,-0.235576,0.005203
17,B_CREDIT_DURATION_MAX,SPECIAL_-99999,37058,3759,33299,0.150637,0.101436,-0.251108,0.010554
Totals,B_CREDIT_DURATION_MAX,None,246008,19860,226148,1.000000,0.080729,NaN,0.037327


In [309]:


rows.append({
    'feature': 'B_CREDIT_DURATION_MAX',
    'n_bins': 4,                       
    'n_special_bins': 1,               
    'special_bins_pct': 0.150637,      
    'bad_rate_trend': 'increasing',    
    'iv': 0.037327,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend ',
})

##### PA_AVG_AMT_ANNUITY_POS

In [310]:
num_bins['PA_AVG_AMT_ANNUITY_POS']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_AVG_AMT_ANNUITY_POS,"(579.779, 3436.335]",11285,1259,10026,0.045872,0.111564,-0.357618,0.006816
1,PA_AVG_AMT_ANNUITY_POS,"(3436.335, 4243.017]",11278,1181,10097,0.045844,0.104717,-0.286605,0.004247
2,PA_AVG_AMT_ANNUITY_POS,"(4243.017, 4864.062]",11282,1079,10203,0.045860,0.095639,-0.185835,0.001712
3,PA_AVG_AMT_ANNUITY_POS,"(4864.062, 5434.404]",11281,1076,10205,0.045856,0.095382,-0.182855,0.001656
4,PA_AVG_AMT_ANNUITY_POS,"(5434.404, 5972.223]",11282,1006,10276,0.045860,0.089169,-0.108653,0.000567
5,PA_AVG_AMT_ANNUITY_POS,"(5972.223, 6507.22]",11281,951,10330,0.045856,0.084301,-0.047188,0.000104
6,PA_AVG_AMT_ANNUITY_POS,"(6507.22, 7030.708]",11281,989,10292,0.045856,0.087670,-0.090054,0.000386
7,PA_AVG_AMT_ANNUITY_POS,"(7030.708, 7588.189]",11282,958,10324,0.045860,0.084914,-0.055103,0.000143
8,PA_AVG_AMT_ANNUITY_POS,"(7588.189, 8186.064]",11281,946,10335,0.045856,0.083858,-0.041433,0.000080
9,PA_AVG_AMT_ANNUITY_POS,"(8186.064, 8817.594]",11282,952,10330,0.045860,0.084382,-0.048239,0.000109


In [311]:
merge_plan = [
    [0],                  # -0.3576 clean start
    [1, 2, 3],            # rising zigzag early negative zone
    [4, 5, 6, 7],
    [8, 9],   # messy mid-negative zone with dips, collapse fully
    [10, 11, 12, 13],     # positive zone zigzag, collapse
    [14, 15,16],                 # 0.2283 clean
    [17,18],                 # 0.5443 strong tail, keep separate
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_AVG_AMT_ANNUITY_POS",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [312]:
num_bins['PA_AVG_AMT_ANNUITY_POS']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_AVG_AMT_ANNUITY_POS,"(579.779, 3436.335]]",11285,1259,10026,0.045872,0.111564,-0.357618,0.006816
1,PA_AVG_AMT_ANNUITY_POS,"(3436.335, 5434.404]]",33841,3336,30505,0.137561,0.098579,-0.219364,0.007258
4,PA_AVG_AMT_ANNUITY_POS,"(5434.404, 7588.189]]",45126,3904,41222,0.183433,0.086513,-0.075512,0.001080
8,PA_AVG_AMT_ANNUITY_POS,"(7588.189, 8817.594]]",22563,1898,20665,0.091717,0.084120,-0.044841,0.000188
10,PA_AVG_AMT_ANNUITY_POS,"(8817.594, 12110.925]]",45125,3400,41725,0.183429,0.075346,0.074843,0.000996
14,PA_AVG_AMT_ANNUITY_POS,"(12110.925, 17434.324]]",33845,2283,31562,0.137577,0.067455,0.193982,0.004774
17,PA_AVG_AMT_ANNUITY_POS,"(17434.324, 393868.66]]",22563,1216,21347,0.091717,0.053894,0.432862,0.014356
19,PA_AVG_AMT_ANNUITY_POS,SPECIAL_-99999,31660,2564,29096,0.128695,0.080985,-0.003450,0.000002
Totals,PA_AVG_AMT_ANNUITY_POS,None,246008,19860,226148,1.000000,0.080729,NaN,0.036943


In [313]:

rows.append({
    'feature': 'PA_AVG_AMT_ANNUITY_POS',
    'n_bins': 7,                       
    'n_special_bins': 1,               
    'special_bins_pct': 0.128695,      
    'bad_rate_trend': 'decreasing',    
    'iv': 0.036943,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend ',
})

##### PA_MAX_RISK_WEIGHT_360D

In [314]:
num_bins['PA_MAX_RISK_WEIGHT_360D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_MAX_RISK_WEIGHT_360D,1.0,9239,491,8748,0.037556,0.053144,0.447654,0.006249
1,PA_MAX_RISK_WEIGHT_360D,2.0,38203,2736,35467,0.155292,0.071617,0.129624,0.002471
2,PA_MAX_RISK_WEIGHT_360D,3.0,41522,3723,37799,0.168783,0.089663,-0.114729,0.002331
3,PA_MAX_RISK_WEIGHT_360D,4.0,14978,1973,13005,0.060884,0.131727,-0.546703,0.022873
4,PA_MAX_RISK_WEIGHT_360D,SPECIAL_-99999,142066,10937,131129,0.577485,0.076985,0.051548,0.001502
Totals,PA_MAX_RISK_WEIGHT_360D,None,246008,19860,226148,1.000000,0.080729,NaN,0.035427


In [315]:

rows.append({
    'feature': 'PA_MAX_RISK_WEIGHT_360D',
    'n_bins': 4,                       
    'n_special_bins': 1,               
    'special_bins_pct': 0.577485,      
    'bad_rate_trend': 'increasing',    
    'iv':0.035427,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend ',
})

##### PA_AVG_RISK_WEIGHT_360D

In [316]:
num_bins['PA_AVG_RISK_WEIGHT_360D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_AVG_RISK_WEIGHT_360D,"(0.999, 1.5]",11279,604,10675,0.045848,0.053551,0.439604,0.007381
1,PA_AVG_RISK_WEIGHT_360D,"(1.5, 2.0]",38063,2758,35305,0.154723,0.072459,0.117037,0.002018
2,PA_AVG_RISK_WEIGHT_360D,"(2.0, 2.5]",7570,638,6932,0.030771,0.084280,-0.046917,0.000069
3,PA_AVG_RISK_WEIGHT_360D,"(2.5, 2.667]",1357,163,1194,0.005516,0.120118,-0.441168,0.001292
4,PA_AVG_RISK_WEIGHT_360D,"(2.667, 3.0]",33556,3155,30401,0.136402,0.094022,-0.166995,0.004080
5,PA_AVG_RISK_WEIGHT_360D,"(3.0, 3.5]",2360,320,2040,0.009593,0.135593,-0.580098,0.004114
6,PA_AVG_RISK_WEIGHT_360D,"(3.5, 4.0]",9757,1285,8472,0.039661,0.131700,-0.546474,0.014886
7,PA_AVG_RISK_WEIGHT_360D,SPECIAL_-99999,142066,10937,131129,0.577485,0.076985,0.051548,0.001502
Totals,PA_AVG_RISK_WEIGHT_360D,None,246008,19860,226148,1.000000,0.080729,NaN,0.035342


In [317]:
merge_plan = [
    [0,1],
    [2,3,4,5,6]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_AVG_RISK_WEIGHT_360D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [318]:
num_bins['PA_AVG_RISK_WEIGHT_360D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_AVG_RISK_WEIGHT_360D,"(0.999, 2.0]]",49342,3362,45980,0.200571,0.068137,0.183188,0.006234
2,PA_AVG_RISK_WEIGHT_360D,"(2.0, 4.0]]",54600,5561,49039,0.221944,0.101850,-0.255644,0.016148
7,PA_AVG_RISK_WEIGHT_360D,SPECIAL_-99999,142066,10937,131129,0.577485,0.076985,0.051548,0.001502
Totals,PA_AVG_RISK_WEIGHT_360D,None,246008,19860,226148,1.000000,0.080729,NaN,0.035342


In [319]:

rows.append({
    'feature': 'PA_AVG_RISK_WEIGHT_360D',
    'n_bins': 2,                       
    'n_special_bins': 1,               
    'special_bins_pct': 0.577485,      
    'bad_rate_trend': 'increasing',    
    'iv':0.035342,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend ',
})

##### B_CLOSED_CREDIT_COUNT

In [320]:
num_bins['B_CLOSED_CREDIT_COUNT']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_CLOSED_CREDIT_COUNT,"(-0.001, 1.0]",69554,6519,63035,0.282731,0.093726,-0.163513,0.008096
1,B_CLOSED_CREDIT_COUNT,"(1.0, 2.0]",34478,2548,31930,0.140150,0.073902,0.095755,0.001235
2,B_CLOSED_CREDIT_COUNT,"(2.0, 3.0]",27016,1834,25182,0.109818,0.067886,0.187148,0.003557
3,B_CLOSED_CREDIT_COUNT,"(3.0, 4.0]",20626,1361,19265,0.083843,0.065985,0.217588,0.003625
4,B_CLOSED_CREDIT_COUNT,"(4.0, 5.0]",15743,1073,14670,0.063994,0.068157,0.182864,0.001982
5,B_CLOSED_CREDIT_COUNT,"(5.0, 6.0]",11643,823,10820,0.047328,0.070686,0.143713,0.000920
6,B_CLOSED_CREDIT_COUNT,"(6.0, 7.0]",8701,609,8092,0.035369,0.069992,0.154331,0.000790
7,B_CLOSED_CREDIT_COUNT,"(7.0, 8.0]",6142,379,5763,0.024967,0.061706,0.289195,0.001851
8,B_CLOSED_CREDIT_COUNT,"(8.0, 10.0]",7798,511,7287,0.031698,0.065530,0.224996,0.001461
9,B_CLOSED_CREDIT_COUNT,"(10.0, 84.0]",9063,636,8427,0.036840,0.070175,0.151516,0.000794


In [321]:
merge_plan = [
    [0],              # -0.1635 anchor — lowest WoE, keep separate
    [1, 2],           # 0.096 → 0.187 clean rise, merge to stabilize
    [3, 4, 5, 6],     # 0.218 → dips → 0.154 zigzag, collapse to one mid-positive block
    [7, 8, 9],        # 0.289 → 0.225 → 0.152 falling tail, collapse to preserve high-end signal
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_CLOSED_CREDIT_COUNT",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [322]:
num_bins['B_CLOSED_CREDIT_COUNT']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_CLOSED_CREDIT_COUNT,"(-0.001, 1.0]]",69554,6519,63035,0.282731,0.093726,-0.163513,0.008096
1,B_CLOSED_CREDIT_COUNT,"(1.0, 3.0]]",61494,4382,57112,0.249967,0.071259,0.135027,0.004307
3,B_CLOSED_CREDIT_COUNT,"(3.0, 7.0]]",56713,3866,52847,0.230533,0.068168,0.182699,0.007129
7,B_CLOSED_CREDIT_COUNT,"(7.0, 84.0]]",23003,1526,21477,0.093505,0.066339,0.211851,0.003841
10,B_CLOSED_CREDIT_COUNT,SPECIAL_-99999,35244,3567,31677,0.143264,0.101209,-0.248616,0.009829
Totals,B_CLOSED_CREDIT_COUNT,None,246008,19860,226148,1.000000,0.080729,NaN,0.034139


In [323]:

rows.append({
    'feature': 'B_CLOSED_CREDIT_COUNT',
    'n_bins': 4,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.143264 ,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.034139,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend ',
})

##### B_CREDIT_DURATION_MIN

In [324]:
num_bins['B_CREDIT_DURATION_MIN']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_CREDIT_DURATION_MIN,"(-41790.001, 32.0]",10730,1223,9507,0.043616,0.113979,-0.381760,7.460449e-03
1,B_CREDIT_DURATION_MIN,"(32.0, 90.0]",11680,939,10741,0.047478,0.080394,0.004526,9.707307e-07
2,B_CREDIT_DURATION_MIN,"(90.0, 92.0]",9171,603,8568,0.037279,0.065751,0.221390,1.665776e-03
3,B_CREDIT_DURATION_MIN,"(92.0, 122.0]",14635,1013,13622,0.059490,0.069218,0.166288,1.534478e-03
4,B_CREDIT_DURATION_MIN,"(122.0, 141.0]",6032,438,5594,0.024520,0.072613,0.114749,3.077146e-04
5,B_CREDIT_DURATION_MIN,"(141.0, 173.0]",10495,629,9866,0.042661,0.059933,0.320237,3.828298e-03
6,B_CREDIT_DURATION_MIN,"(173.0, 181.0]",11472,767,10705,0.046633,0.066858,0.203497,1.773666e-03
7,B_CREDIT_DURATION_MIN,"(181.0, 183.0]",15120,1057,14063,0.061461,0.069907,0.155631,1.394820e-03
8,B_CREDIT_DURATION_MIN,"(183.0, 184.0]",8020,576,7444,0.032601,0.071820,0.126574,4.953442e-04
9,B_CREDIT_DURATION_MIN,"(184.0, 212.0]",10062,742,9320,0.040901,0.073743,0.098087,3.776748e-04


In [325]:
merge_plan = [
    [0, 1, 2, 3, 4, 5, 6,7, 8, 9, 10, 11, 12, 13],  # positive middle plateau, collapse zigzag
    [14, 15, 16, 17],           # turning negative, absorb bin16 uptick
    [18, 19],                   # deep negative tail, absorb bin19 uptick
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_CREDIT_DURATION_MIN",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [326]:
num_bins['B_CREDIT_DURATION_MIN']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_CREDIT_DURATION_MIN,"(-41790.001, 348.0]]",146279,10729,135550,0.594611,0.073346,0.103908,0.006147
14,B_CREDIT_DURATION_MIN,"(348.0, 1094.0]]",41786,3532,38254,0.169856,0.084526,-0.050098,0.000435
18,B_CREDIT_DURATION_MIN,"(1094.0, 33929.0]]",20885,1840,19045,0.084896,0.088102,-0.095443,0.000805
20,B_CREDIT_DURATION_MIN,SPECIAL_-99999,37058,3759,33299,0.150637,0.101436,-0.251108,0.010554
Totals,B_CREDIT_DURATION_MIN,None,246008,19860,226148,1.000000,0.080729,NaN,0.033947


In [327]:

rows.append({
    'feature': 'B_CREDIT_DURATION_MIN',
    'n_bins': 3,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.150637,      
    'bad_rate_trend': 'increasing',    
    'iv':0.033947,                    
    'keep': 'Yes',                     
    'comment': 'clear increasing monotonic trend ',
})

##### PA_FLAG_NEW_CLIENT_720D

In [328]:
num_bins['PA_FLAG_NEW_CLIENT_720D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_FLAG_NEW_CLIENT_720D,0.0,180274,13480,166794,0.732797,0.074775,0.083070,0.004884
1,PA_FLAG_NEW_CLIENT_720D,1.0,52560,5589,46971,0.213652,0.106336,-0.303752,0.022392
2,PA_FLAG_NEW_CLIENT_720D,SPECIAL_-99999,13174,791,12383,0.053551,0.060043,0.318300,0.004751
Totals,PA_FLAG_NEW_CLIENT_720D,None,246008,19860,226148,1.000000,0.080729,NaN,0.032028


In [329]:

rows.append({
    'feature': 'PA_FLAG_NEW_CLIENT_720D',
    'n_bins': 2,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.053551,      
    'bad_rate_trend': 'increasing',    
    'iv':0.032028,                    
    'keep': 'Yes',                     
    'comment': 'clear increasing monotonic trend ',
})

##### B_AMT_CREDIT_SUM_DEBT_MIN

In [330]:
num_bins['B_AMT_CREDIT_SUM_DEBT_MIN']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_AMT_CREDIT_SUM_DEBT_MIN,"(-4705600.501, 0.0]",182595,13349,169246,0.742232,0.073107,0.107430,0.008189
1,B_AMT_CREDIT_SUM_DEBT_MIN,"(0.0, 5365.421]",697,83,614,0.002833,0.119082,-0.431328,0.000632
2,B_AMT_CREDIT_SUM_DEBT_MIN,"(5365.421, 109199.615]",10782,1333,9449,0.043828,0.123632,-0.474005,0.012010
3,B_AMT_CREDIT_SUM_DEBT_MIN,"(109199.615, 43650000.0]",10782,1137,9645,0.043828,0.105454,-0.294436,0.004299
4,B_AMT_CREDIT_SUM_DEBT_MIN,SPECIAL_-99999,41152,3958,37194,0.167279,0.096180,-0.192073,0.006689
Totals,B_AMT_CREDIT_SUM_DEBT_MIN,None,246008,19860,226148,1.000000,0.080729,NaN,0.031820


In [331]:
merge_plan = [
  [0],
  [1,2,3]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_AMT_CREDIT_SUM_DEBT_MIN",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [332]:
num_bins['B_AMT_CREDIT_SUM_DEBT_MIN']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_AMT_CREDIT_SUM_DEBT_MIN,"(-4705600.501, 0.0]]",182595,13349,169246,0.742232,0.073107,0.107430,0.008189
1,B_AMT_CREDIT_SUM_DEBT_MIN,"(0.0, 43650000.0]]",22261,2553,19708,0.090489,0.114685,-0.388726,0.016095
4,B_AMT_CREDIT_SUM_DEBT_MIN,SPECIAL_-99999,41152,3958,37194,0.167279,0.096180,-0.192073,0.006689
Totals,B_AMT_CREDIT_SUM_DEBT_MIN,None,246008,19860,226148,1.000000,0.080729,NaN,0.031820


In [333]:

rows.append({
    'feature': 'B_AMT_CREDIT_SUM_DEBT_MIN',
    'n_bins': 2,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.167279,      
    'bad_rate_trend': 'increasing',    
    'iv':0.031820,                    
    'keep': 'Yes',                     
    'comment': 'clear increasing monotonic trend ',
})

##### ANNUITY_TO_AGE_RATIO

In [334]:
num_bins['ANNUITY_TO_AGE_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,ANNUITY_TO_AGE_RATIO,"(25.104, 196.077]",12948,658,12290,0.052632,0.050819,0.494854,0.010497
1,ANNUITY_TO_AGE_RATIO,"(196.077, 250.407]",12947,744,12203,0.052628,0.057465,0.364914,0.006020
2,ANNUITY_TO_AGE_RATIO,"(250.407, 298.4]",12947,843,12104,0.052628,0.065112,0.231842,0.002568
3,ANNUITY_TO_AGE_RATIO,"(298.4, 342.856]",12947,917,12030,0.052628,0.070827,0.141569,0.000994
4,ANNUITY_TO_AGE_RATIO,"(342.856, 382.637]",12948,896,12052,0.052632,0.069200,0.166563,0.001362
5,ANNUITY_TO_AGE_RATIO,"(382.637, 422.577]",12947,994,11953,0.052628,0.076775,0.054518,0.000153
6,ANNUITY_TO_AGE_RATIO,"(422.577, 466.362]",12947,1025,11922,0.052628,0.079169,0.021211,0.000023
7,ANNUITY_TO_AGE_RATIO,"(466.362, 512.339]",12947,1020,11927,0.052628,0.078783,0.026520,0.000037
8,ANNUITY_TO_AGE_RATIO,"(512.339, 558.648]",12947,1115,11832,0.052628,0.086120,-0.070529,0.000270
9,ANNUITY_TO_AGE_RATIO,"(558.648, 606.639]",12949,1148,11801,0.052636,0.088655,-0.102319,0.000575


In [335]:
merge_plan = [
    [0],              
    [1, 2],           
    [3, 4],           
    [5, 6, 7],        
    [8, 9, 10, 11, 12],   
    [13, 14, 15, 16, 17, 18],  
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="ANNUITY_TO_AGE_RATIO",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [336]:
num_bins['ANNUITY_TO_AGE_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,ANNUITY_TO_AGE_RATIO,"(25.104, 196.077]]",12948,658,12290,0.052632,0.050819,0.494854,0.010497
1,ANNUITY_TO_AGE_RATIO,"(196.077, 298.4]]",25894,1587,24307,0.105257,0.061288,0.296437,0.008174
3,ANNUITY_TO_AGE_RATIO,"(298.4, 382.637]]",25895,1813,24082,0.105261,0.070014,0.154000,0.002341
5,ANNUITY_TO_AGE_RATIO,"(382.637, 512.339]]",38841,3039,35802,0.157885,0.078242,0.033993,0.000180
8,ANNUITY_TO_AGE_RATIO,"(512.339, 774.453]]",64736,5732,59004,0.263146,0.088544,-0.100941,0.002797
13,ANNUITY_TO_AGE_RATIO,"(774.453, 8324.084]]",77684,7031,70653,0.315778,0.090508,-0.125030,0.005202
19,ANNUITY_TO_AGE_RATIO,SPECIAL_-88888,10,0,10,0.000041,0.000000,0.000000,0.000000
Totals,ANNUITY_TO_AGE_RATIO,None,246008,19860,226148,1.000000,0.080729,NaN,0.031518


In [337]:

rows.append({
    'feature': 'ANNUITY_TO_AGE_RATIO',
    'n_bins': 6,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.000041,      
    'bad_rate_trend': 'increasing',    
    'iv':0.031518,                    
    'keep': 'Yes',                     
    'comment': 'clear increasing monotonic trend ',
})

##### FLAG_HC_REJECT_REASON

In [338]:
num_bins['FLAG_HC_REJECT_REASON']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,FLAG_HC_REJECT_REASON,0.0,179871,13463,166408,0.731159,0.074848,0.082015,0.004752
1,FLAG_HC_REJECT_REASON,1.0,52963,5606,47357,0.215290,0.105847,-0.298605,0.021759
2,FLAG_HC_REJECT_REASON,SPECIAL_-99999,13174,791,12383,0.053551,0.060043,0.318300,0.004751
Totals,FLAG_HC_REJECT_REASON,None,246008,19860,226148,1.000000,0.080729,NaN,0.031262


In [339]:

rows.append({
    'feature': 'FLAG_HC_REJECT_REASON',
    'n_bins': 2,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.053551,      
    'bad_rate_trend': 'increasing',    
    'iv':0.031262,                    
    'keep': 'Yes',                     
    'comment': 'clear increasing monotonic trend ',
})

##### PCB_WORST_DPD_POS_CASH_24M

In [340]:
num_bins['PCB_WORST_DPD_POS_CASH_24M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PCB_WORST_DPD_POS_CASH_24M,"(-0.001, 3.0]",187699,14665,173034,0.762979,0.078130,0.035542,9.495927e-04
1,PCB_WORST_DPD_POS_CASH_24M,"(3.0, 4231.0]",8774,1403,7371,0.035666,0.159904,-0.773541,2.943388e-02
2,PCB_WORST_DPD_POS_CASH_24M,SPECIAL_-66666,35083,2826,32257,0.142609,0.080552,0.002391,8.143240e-07
3,PCB_WORST_DPD_POS_CASH_24M,SPECIAL_-99999,14452,966,13486,0.058746,0.066842,0.203762,2.239957e-03
Totals,PCB_WORST_DPD_POS_CASH_24M,None,246008,19860,226148,1.000000,0.080729,NaN,3.262424e-02


In [341]:

rows.append({
    'feature': 'PCB_WORST_DPD_POS_CASH_24M',
    'n_bins': 2,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.058746,      
    'bad_rate_trend': 'increasing',    
    'iv':0.036282,                    
    'keep': 'Drop',                     
    'comment': 'good feature ',
})

##### PA_MAX_DOWN_PAYMENT_RATE

In [342]:
num_bins['PA_MAX_DOWN_PAYMENT_RATE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_MAX_DOWN_PAYMENT_RATE,"(-1e-05, 0.08723]",64303,6054,58249,0.261386,0.094148,-0.168474,0.007963
1,PA_MAX_DOWN_PAYMENT_RATE,"(0.08723, 0.099502]",10717,1152,9565,0.043564,0.107493,-0.315871,0.004963
2,PA_MAX_DOWN_PAYMENT_RATE,"(0.099502, 0.10138]",10717,962,9755,0.043564,0.089764,-0.115961,0.000615
3,PA_MAX_DOWN_PAYMENT_RATE,"(0.10138, 0.10396]",10717,850,9867,0.043564,0.079313,0.019233,0.000016
4,PA_MAX_DOWN_PAYMENT_RATE,"(0.10396, 0.10706]",10717,859,9858,0.043564,0.080153,0.007788,0.000003
5,PA_MAX_DOWN_PAYMENT_RATE,"(0.10706, 0.10891]",13956,993,12963,0.056730,0.071152,0.136642,0.001000
6,PA_MAX_DOWN_PAYMENT_RATE,"(0.10891, 0.10894]",7480,457,7023,0.030406,0.061096,0.299780,0.002411
7,PA_MAX_DOWN_PAYMENT_RATE,"(0.10894, 0.11426]",10715,843,9872,0.043555,0.078675,0.028009,0.000034
8,PA_MAX_DOWN_PAYMENT_RATE,"(0.11426, 0.14878]",10717,853,9864,0.043564,0.079593,0.015406,0.000010
9,PA_MAX_DOWN_PAYMENT_RATE,"(0.14878, 0.19884]",10717,853,9864,0.043564,0.079593,0.015406,0.000010


In [343]:
merge_plan = [
    [0, 1, 2],        # bins 1→2 spike absorbed, net negative block
    [3, 4, 5],        # bin4 dip absorbed, near-zero to mild positive
    [6, 7, 8, 9, 10], # bin6 spike + bins 7,8,9 drop fully collapsed
    [11, 12],         # bin12 dip absorbed
    [13],             # 0.2994 clean
    [14],             # 0.3944 strong positive tail
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_MAX_DOWN_PAYMENT_RATE",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [344]:
num_bins['PA_MAX_DOWN_PAYMENT_RATE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_MAX_DOWN_PAYMENT_RATE,"(-1e-05, 0.10138]]",85737,8168,77569,0.348513,0.095268,-0.181538,0.012395
3,PA_MAX_DOWN_PAYMENT_RATE,"(0.10138, 0.10891]]",35390,2702,32688,0.143857,0.076349,0.060534,0.000514
6,PA_MAX_DOWN_PAYMENT_RATE,"(0.10891, 0.21227]]",50346,3758,46588,0.204652,0.074643,0.084974,0.001426
11,PA_MAX_DOWN_PAYMENT_RATE,"(0.21227, 0.30783]]",21434,1414,20020,0.087127,0.065970,0.217827,0.003774
13,PA_MAX_DOWN_PAYMENT_RATE,"(0.30783, 0.41489]]",10717,655,10062,0.043564,0.061118,0.299404,0.003447
14,PA_MAX_DOWN_PAYMENT_RATE,"(0.41489, 1.0]]",10718,599,10119,0.043568,0.055887,0.394427,0.005752
15,PA_MAX_DOWN_PAYMENT_RATE,SPECIAL_-99999,31666,2564,29102,0.128719,0.080970,-0.003244,0.000001
Totals,PA_MAX_DOWN_PAYMENT_RATE,None,246008,19860,226148,1.000000,0.080729,NaN,0.030941


In [345]:

rows.append({
    'feature': 'PA_MAX_DOWN_PAYMENT_RATE',
    'n_bins': 6,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.128719,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.030941,                    
    'keep': 'Yes',                     
    'comment': 'vcleae monotonic trend ',
})

##### WORST_DPD_DEF_POS_CASH_36M

In [346]:
num_bins['WORST_DPD_DEF_POS_CASH_36M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,WORST_DPD_DEF_POS_CASH_36M,"(-0.001, 2.0]",197202,15349,181853,0.801608,0.077834,0.039666,0.001240
1,WORST_DPD_DEF_POS_CASH_36M,"(2.0, 3373.0]",10363,1566,8797,0.042125,0.151115,-0.706596,0.028230
2,WORST_DPD_DEF_POS_CASH_36M,SPECIAL_-66666,23991,1979,22012,0.097521,0.082489,-0.023486,0.000054
3,WORST_DPD_DEF_POS_CASH_36M,SPECIAL_-99999,14452,966,13486,0.058746,0.066842,0.203762,0.002240
Totals,WORST_DPD_DEF_POS_CASH_36M,None,246008,19860,226148,1.000000,0.080729,NaN,0.031765


In [347]:

rows.append({
    'feature': 'WORST_DPD_DEF_POS_CASH_36M',
    'n_bins': 2,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.201355,      
    'bad_rate_trend': 'increasing',    
    'iv':0.030089,                    
    'keep': 'Drop',                     
    'comment': 'violates 5% bin per rule monotonic trend ',
})

##### FLOOR_RATIO

In [348]:
num_bins['FLOOR_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,FLOOR_RATIO,"(-0.001, 0.501]",4840,442,4398,0.019674,0.091322,-0.134887,0.000379
1,FLOOR_RATIO,"(0.501, 0.715]",2879,244,2635,0.011703,0.084752,-0.053012,0.000034
2,FLOOR_RATIO,"(0.715, 0.8]",28395,2066,26329,0.115423,0.072759,0.112575,0.001395
3,FLOOR_RATIO,"(0.8, 0.889]",15672,959,14713,0.063705,0.061192,0.298114,0.005000
4,FLOOR_RATIO,"(0.889, 0.9]",3138,174,2964,0.012756,0.055449,0.402758,0.001750
5,FLOOR_RATIO,"(0.9, 0.937]",3596,181,3415,0.014617,0.050334,0.504954,0.003023
6,FLOOR_RATIO,"(0.937, 1.0]",3276,187,3089,0.013317,0.057082,0.372012,0.001579
7,FLOOR_RATIO,"(1.0, 2.998]",4195,295,3900,0.017052,0.070322,0.149274,0.000357
8,FLOOR_RATIO,"(2.998, 3.998]",6207,462,5745,0.025231,0.074432,0.088038,0.000188
9,FLOOR_RATIO,"(3.998, 7.993]",3477,239,3238,0.014134,0.068737,0.173766,0.000397


In [349]:
merge_plan = [
    [0, 1],              # negative base
    [2, 3],              # rising mid
    [4, 5, 6, 7, 8, 9, 10],  # absorb entire peak + tail into one positive block
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="FLOOR_RATIO",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [350]:
num_bins['FLOOR_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,FLOOR_RATIO,"(-0.001, 0.715]]",7719,686,7033,0.031377,0.088872,-0.104991,0.000361
2,FLOOR_RATIO,"(0.715, 0.889]]",44067,3025,41042,0.179128,0.068645,0.175203,0.005110
4,FLOOR_RATIO,"(0.889, 33.149]]",25392,1618,23774,0.103216,0.063721,0.254920,0.006030
11,FLOOR_RATIO,SPECIAL_-77777,44703,3161,41542,0.181714,0.070711,0.143335,0.003516
12,FLOOR_RATIO,SPECIAL_-88888,79,11,68,0.000321,0.139241,-0.610870,0.000155
13,FLOOR_RATIO,SPECIAL_-99999,124048,11359,112689,0.504244,0.091569,-0.137861,0.010154
Totals,FLOOR_RATIO,None,246008,19860,226148,1.000000,0.080729,NaN,0.028936


In [351]:
0.189152 + 0.000321+0.496805

0.6862779999999999

In [352]:

rows.append({
    'feature': 'FLOOR_RATIO',
    'n_bins': 3,                       
    'n_special_bins': 3,               
    'special_bins_pct':0.6862779999999999,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.029108,                    
    'keep': 'Yes',                     
    'comment': 'special values contain all 3 types ',
})

##### B_CREDIT_DURATION_MEAN

In [353]:
num_bins['B_CREDIT_DURATION_MEAN']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_CREDIT_DURATION_MEAN,"(-40270.001, 191.333]",10452,891,9561,0.042486,0.085247,-0.059379,1.535788e-04
1,B_CREDIT_DURATION_MEAN,"(191.333, 274.0]",10655,670,9985,0.043312,0.062881,0.269080,2.802826e-03
2,B_CREDIT_DURATION_MEAN,"(274.0, 332.667]",10238,652,9586,0.041617,0.063684,0.255532,2.442471e-03
3,B_CREDIT_DURATION_MEAN,"(332.667, 371.393]",10445,791,9654,0.042458,0.075730,0.069348,1.983382e-04
4,B_CREDIT_DURATION_MEAN,"(371.393, 456.0]",10572,679,9893,0.042974,0.064226,0.246480,2.355449e-03
5,B_CREDIT_DURATION_MEAN,"(456.0, 531.297]",10323,703,9620,0.041962,0.068100,0.183761,1.312185e-03
6,B_CREDIT_DURATION_MEAN,"(531.297, 599.25]",10450,695,9755,0.042478,0.066507,0.209141,1.702517e-03
7,B_CREDIT_DURATION_MEAN,"(599.25, 672.5]",10453,746,9707,0.042490,0.071367,0.133395,7.150340e-04
8,B_CREDIT_DURATION_MEAN,"(672.5, 731.5]",10511,793,9718,0.042726,0.075445,0.073430,2.233996e-04
9,B_CREDIT_DURATION_MEAN,"(731.5, 815.785]",10376,772,9604,0.042177,0.074402,0.088468,3.181023e-04


In [354]:
merge_plan = [
    [0, 1, 2, 3, 4, 5, 6,7, 8, 9, 10, 11, 12, 13],  # positive middle plateau, collapse zigzag
    [14, 15, 16, 17],           # turning negative, absorb bin16 uptick
    [18, 19],                   # deep negative tail, absorb bin19 uptick
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_CREDIT_DURATION_MEAN",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [355]:
num_bins['B_CREDIT_DURATION_MEAN']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_CREDIT_DURATION_MEAN,"(-40270.001, 1250.5]]",146268,10734,135534,0.594566,0.073386,0.103324,0.006079
14,B_CREDIT_DURATION_MEAN,"(1250.5, 4147.91]]",41787,3291,38496,0.169860,0.078757,0.026881,0.000121
18,B_CREDIT_DURATION_MEAN,"(4147.91, 33929.0]]",20895,2076,18819,0.084936,0.099354,-0.228058,0.004861
20,B_CREDIT_DURATION_MEAN,SPECIAL_-99999,37058,3759,33299,0.150637,0.101436,-0.251108,0.010554
Totals,B_CREDIT_DURATION_MEAN,None,246008,19860,226148,1.000000,0.080729,NaN,0.028894


In [356]:

rows.append({
    'feature': 'B_CREDIT_DURATION_MEAN',
    'n_bins': 3,                       
    'n_special_bins': 1,               
    'special_bins_pct':0.150637,      
    'bad_rate_trend': 'increasing',    
    'iv':0.028894,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic increasing trend',
})

##### WORST_DPD_DEF_POS_CASH_72M

In [357]:
num_bins['WORST_DPD_DEF_POS_CASH_72M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,WORST_DPD_DEF_POS_CASH_72M,"(-0.001, 1.0]",202965,15712,187253,0.825034,0.077412,0.045554,0.001680
1,WORST_DPD_DEF_POS_CASH_72M,"(1.0, 6.0]",11542,1245,10297,0.046917,0.107867,-0.319765,0.005486
2,WORST_DPD_DEF_POS_CASH_72M,"(6.0, 3373.0]",10856,1453,9403,0.044129,0.133843,-0.565084,0.017847
3,WORST_DPD_DEF_POS_CASH_72M,SPECIAL_-66666,6193,484,5709,0.025174,0.078153,0.035232,0.000031
4,WORST_DPD_DEF_POS_CASH_72M,SPECIAL_-99999,14452,966,13486,0.058746,0.066842,0.203762,0.002240
Totals,WORST_DPD_DEF_POS_CASH_72M,None,246008,19860,226148,1.000000,0.080729,NaN,0.027284


In [358]:
merge_plan = [
    [1,2]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="WORST_DPD_DEF_POS_CASH_72M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [359]:
num_bins['WORST_DPD_DEF_POS_CASH_72M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,WORST_DPD_DEF_POS_CASH_72M,"(-0.001, 1.0]",202965,15712,187253,0.825034,0.077412,0.045554,0.001680
1,WORST_DPD_DEF_POS_CASH_72M,"(1.0, 3373.0]]",22398,2698,19700,0.091046,0.120457,-0.444374,0.021659
3,WORST_DPD_DEF_POS_CASH_72M,SPECIAL_-66666,6193,484,5709,0.025174,0.078153,0.035232,0.000031
4,WORST_DPD_DEF_POS_CASH_72M,SPECIAL_-99999,14452,966,13486,0.058746,0.066842,0.203762,0.002240
Totals,WORST_DPD_DEF_POS_CASH_72M,None,246008,19860,226148,1.000000,0.080729,NaN,0.027284


In [360]:

rows.append({
    'feature': 'WORST_DPD_DEF_POS_CASH_72M',
    'n_bins': 3,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.083920,      
    'bad_rate_trend': 'increasing',    
    'iv':0.026800,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic increasing trend',
})

##### LIVINGAREA_RATIO

In [361]:
num_bins['LIVINGAREA_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,LIVINGAREA_RATIO,"(-0.001, 0.674]",6111,429,5682,0.024841,0.070201,0.151120,0.000533
1,LIVINGAREA_RATIO,"(0.674, 0.757]",6114,436,5678,0.024853,0.071312,0.134230,0.000423
2,LIVINGAREA_RATIO,"(0.757, 0.814]",6106,471,5635,0.024820,0.077137,0.049412,0.000059
3,LIVINGAREA_RATIO,"(0.814, 0.864]",6122,447,5675,0.024885,0.073015,0.108785,0.000281
4,LIVINGAREA_RATIO,"(0.864, 0.912]",6099,401,5698,0.024792,0.065748,0.221427,0.001108
5,LIVINGAREA_RATIO,"(0.912, 0.952]",6113,383,5730,0.024849,0.062653,0.272954,0.001652
6,LIVINGAREA_RATIO,"(0.952, 0.989]",6109,398,5711,0.024833,0.065150,0.231215,0.001205
7,LIVINGAREA_RATIO,"(0.989, 1.033]",6109,464,5645,0.024833,0.075954,0.066159,0.000106
8,LIVINGAREA_RATIO,"(1.033, 1.075]",6109,407,5702,0.024833,0.066623,0.207277,0.000978
9,LIVINGAREA_RATIO,"(1.075, 1.106]",6111,476,5635,0.024841,0.077892,0.038853,0.000037


In [362]:

rows.append({
    'feature': 'LIVINGAREA_RATIO',
    'n_bins': 4,                      
    'n_special_bins': 3,               
    'special_bins_pct':0.503247,      
    'bad_rate_trend': 'non monotonic',    
    'iv':0.026800,                    
    'keep': 'Drop',                     
    'comment': 'all types of special values and very nosisy feature',
})

##### MONTHS_BALANCE_SUM

In [363]:
num_bins['MONTHS_BALANCE_SUM']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,MONTHS_BALANCE_SUM,"(-4656.001, -1128.0]",3792,236,3556,0.015414,0.062236,0.280078,0.001076
1,MONTHS_BALANCE_SUM,"(-1128.0, -630.0]",3611,205,3406,0.014678,0.056771,0.377802,0.001790
2,MONTHS_BALANCE_SUM,"(-630.0, -378.0]",4196,211,3985,0.017056,0.050286,0.505952,0.003540
3,MONTHS_BALANCE_SUM,"(-378.0, -260.0]",3140,185,2955,0.012764,0.058917,0.338416,0.001270
4,MONTHS_BALANCE_SUM,"(-260.0, -190.0]",3723,231,3492,0.015134,0.062047,0.283330,0.001079
5,MONTHS_BALANCE_SUM,"(-190.0, -136.0]",4178,286,3892,0.016983,0.068454,0.178205,0.000501
6,MONTHS_BALANCE_SUM,"(-136.0, -105.0]",3450,212,3238,0.014024,0.061449,0.293643,0.001070
7,MONTHS_BALANCE_SUM,"(-105.0, -78.0]",4184,314,3870,0.017008,0.075048,0.079135,0.000103
8,MONTHS_BALANCE_SUM,"(-78.0, -55.0]",5176,408,4768,0.021040,0.078825,0.025933,0.000014
9,MONTHS_BALANCE_SUM,"(-55.0, -45.0]",2878,195,2683,0.011699,0.067755,0.189209,0.000387


In [364]:
merge_plan = [
    [0, 1, 2, 3,4],        # spike at bin1, peak at bin2, drop at bin3 — collapse entire early positive hump
    [5, 6,7, 8, 9,10],           # bin9 uptick absorbed into near-zero zone
    [11,12, 13, 14],
    [15,16,17,18],                # -0.7422 strong negative tail
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="MONTHS_BALANCE_SUM",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [365]:
num_bins['MONTHS_BALANCE_SUM']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,MONTHS_BALANCE_SUM,"(-4656.001, -190.0]]",18462,1068,17394,0.075046,0.057849,0.357856,0.008280
5,MONTHS_BALANCE_SUM,"(-190.0, -36.0]]",23171,1688,21483,0.094188,0.072850,0.111236,0.001112
11,MONTHS_BALANCE_SUM,"(-36.0, -10.0]]",17388,1541,15847,0.070681,0.088624,-0.101933,0.000766
15,MONTHS_BALANCE_SUM,"(-10.0, 0.0]]",14645,1686,12959,0.059531,0.115125,-0.393050,0.010845
19,MONTHS_BALANCE_SUM,SPECIAL_-99999,172342,13877,158465,0.700554,0.080520,0.002819,0.000006
Totals,MONTHS_BALANCE_SUM,None,246008,19860,226148,1.000000,0.080729,NaN,0.025498


In [366]:

rows.append({
    'feature': 'MONTHS_BALANCE_SUM',
    'n_bins': 4,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.700554,      
    'bad_rate_trend': 'increasing',    
    'iv':0.025498,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend',
})

##### PA_MAX_AMT_GOODS_PRICE

In [367]:
num_bins['PA_MAX_AMT_GOODS_PRICE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_MAX_AMT_GOODS_PRICE,"(-0.001, 38724.525]",11602,1280,10322,0.047161,0.110326,-0.345065,6.490140e-03
1,PA_MAX_AMT_GOODS_PRICE,"(38724.525, 54000.0]",11637,1152,10485,0.047303,0.098995,-0.224036,2.608360e-03
2,PA_MAX_AMT_GOODS_PRICE,"(54000.0, 69489.225]",11566,999,10567,0.047015,0.086374,-0.073746,2.637198e-04
3,PA_MAX_AMT_GOODS_PRICE,"(69489.225, 85670.1]",11601,946,10655,0.047157,0.081545,-0.010940,5.669889e-06
4,PA_MAX_AMT_GOODS_PRICE,"(85670.1, 98982.0]",11603,966,10637,0.047165,0.083254,-0.033552,5.384816e-05
5,PA_MAX_AMT_GOODS_PRICE,"(98982.0, 114471.0]",11601,862,10739,0.047157,0.074304,0.089900,3.670414e-04
6,PA_MAX_AMT_GOODS_PRICE,"(114471.0, 134217.675]",11601,806,10795,0.047157,0.069477,0.162273,1.160272e-03
7,PA_MAX_AMT_GOODS_PRICE,"(134217.675, 146547.0]",11602,944,10658,0.047161,0.081365,-0.008542,3.453569e-06
8,PA_MAX_AMT_GOODS_PRICE,"(146547.0, 174793.5]",11602,827,10775,0.047161,0.071281,0.134697,8.087624e-04
9,PA_MAX_AMT_GOODS_PRICE,"(174793.5, 202500.0]",11681,921,10760,0.047482,0.078846,0.025649,3.090268e-05


In [368]:
merge_plan = [
    [0, 1],          # negative zone — bins 1,2,3 rise but net stays negative
    [2,3,4,5, 6, 7, 8, 9, 10],      # chaotic mid zone — bin7,9,10 all break, collapse fully
    [11, 12, 13, 14, 15],     # bin11 spikes, bin14 spikes — collapse entire mid-positive mess
    [16, 17,18, 19],         # tail zigzag absorbed into one positive block
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_MAX_AMT_GOODS_PRICE",   # ← fixed feature name
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [369]:
num_bins['PA_MAX_AMT_GOODS_PRICE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_MAX_AMT_GOODS_PRICE,"(-0.001, 54000.0]]",23239,2432,20807,0.094464,0.104652,-0.285906,8.706147e-03
2,PA_MAX_AMT_GOODS_PRICE,"(54000.0, 225000.0]]",104543,8437,96106,0.424958,0.080704,0.000343,4.998287e-08
11,PA_MAX_AMT_GOODS_PRICE,"(225000.0, 670500.0]]",57878,4608,53270,0.235269,0.079616,0.015097,5.328545e-05
16,PA_MAX_AMT_GOODS_PRICE,"(670500.0, 5850000.0]]",46370,3547,42823,0.188490,0.076493,0.058491,6.292517e-04
20,PA_MAX_AMT_GOODS_PRICE,SPECIAL_-99999,13978,836,13142,0.056819,0.059808,0.322458,5.165041e-03
Totals,PA_MAX_AMT_GOODS_PRICE,None,246008,19860,226148,1.000000,0.080729,NaN,2.542240e-02


In [370]:

rows.append({
    'feature': 'PA_MAX_AMT_GOODS_PRICE',
    'n_bins': 4,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.056819,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.025422,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend',
})

##### NONLIVING_RATIO

In [371]:
num_bins['NONLIVING_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,NONLIVING_RATIO,"(-0.001, 0.011]",49581,3547,46034,0.201542,0.071540,0.130796,0.003264
1,NONLIVING_RATIO,"(0.011, 0.0339]",5508,322,5186,0.022390,0.058460,0.346684,0.002329
2,NONLIVING_RATIO,"(0.0339, 0.0652]",5508,374,5134,0.022390,0.067901,0.186903,0.000723
3,NONLIVING_RATIO,"(0.0652, 0.113]",5509,366,5143,0.022394,0.066437,0.210277,0.000907
4,NONLIVING_RATIO,"(0.113, 0.18]",5509,337,5172,0.022394,0.061173,0.298450,0.001761
5,NONLIVING_RATIO,"(0.18, 0.27]",5509,380,5129,0.022394,0.068978,0.170013,0.000603
6,NONLIVING_RATIO,"(0.27, 0.38]",5509,351,5158,0.022394,0.063714,0.255036,0.001309
7,NONLIVING_RATIO,"(0.38, 0.512]",5508,372,5136,0.022390,0.067538,0.192654,0.000767
8,NONLIVING_RATIO,"(0.512, 0.708]",5509,405,5104,0.022394,0.073516,0.101411,0.000221
9,NONLIVING_RATIO,"(0.708, 0.979]",5509,389,5120,0.022394,0.070612,0.144848,0.000442


In [372]:
rows.append({
    'feature': 'NONLIVING_RATIO',
    'n_bins': 4,                      
    'n_special_bins': 3,               
    'special_bins_pct':0.056819,      
    'bad_rate_trend': 'non monotonic',    
    'iv':0.024725,                    
    'keep': 'Drop',                     
    'comment': 'non monotonic trend',
})

##### PA_AVG_AMT_ANNUITY_CARDS

In [373]:
num_bins['PA_AVG_AMT_ANNUITY_CARDS']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_AVG_AMT_ANNUITY_CARDS,"(-0.001, 2250.0]",17087,1872,15215,0.069457,0.109557,-0.337208,0.009098
1,PA_AVG_AMT_ANNUITY_CARDS,"(2250.0, 3375.0]",3116,261,2855,0.012666,0.083761,-0.040175,0.000021
2,PA_AVG_AMT_ANNUITY_CARDS,"(3375.0, 4500.0]",5365,544,4821,0.021808,0.101398,-0.250695,0.001523
3,PA_AVG_AMT_ANNUITY_CARDS,"(4500.0, 5625.0]",2632,273,2359,0.010699,0.103723,-0.275961,0.000915
4,PA_AVG_AMT_ANNUITY_CARDS,"(5625.0, 6750.0]",6979,647,6332,0.028369,0.092707,-0.151457,0.000693
5,PA_AVG_AMT_ANNUITY_CARDS,"(6750.0, 7125.0]",73,11,62,0.000297,0.150685,-0.703243,0.000197
6,PA_AVG_AMT_ANNUITY_CARDS,"(7125.0, 9000.0]",10262,997,9265,0.041714,0.097155,-0.203234,0.001876
7,PA_AVG_AMT_ANNUITY_CARDS,"(9000.0, 10125.0]",1905,178,1727,0.007744,0.093438,-0.160124,0.000212
8,PA_AVG_AMT_ANNUITY_CARDS,"(10125.0, 11250.0]",5698,561,5137,0.023162,0.098456,-0.217978,0.001206
9,PA_AVG_AMT_ANNUITY_CARDS,"(11250.0, 13500.0]",5234,448,4786,0.021276,0.085594,-0.063825,0.000089


In [374]:
merge_plan = [
    [0],                        
    [1, 2, 3, 4, 5, 6],       
    [7, 8, 9, 10, 11],
    [12, 13, 14],  
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_AVG_AMT_ANNUITY_CARDS",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [375]:
num_bins['PA_AVG_AMT_ANNUITY_CARDS']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_AVG_AMT_ANNUITY_CARDS,"(-0.001, 2250.0]]",17087,1872,15215,0.069457,0.109557,-0.337208,0.009098
1,PA_AVG_AMT_ANNUITY_CARDS,"(2250.0, 9000.0]]",28427,2733,25694,0.115553,0.096141,-0.191624,0.004598
7,PA_AVG_AMT_ANNUITY_CARDS,"(9000.0, 18750.0]]",17157,1608,15549,0.069742,0.093723,-0.163477,0.001996
12,PA_AVG_AMT_ANNUITY_CARDS,"(18750.0, 113625.0]]",15622,1276,14346,0.063502,0.081680,-0.012741,0.000010
15,PA_AVG_AMT_ANNUITY_CARDS,SPECIAL_-99999,167715,12371,155344,0.681746,0.073762,0.097805,0.006260
Totals,PA_AVG_AMT_ANNUITY_CARDS,None,246008,19860,226148,1.000000,0.080729,NaN,0.024323


In [376]:
rows.append({
    'feature': 'PA_AVG_AMT_ANNUITY_CARDS',
    'n_bins': 4,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.681746,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.024323,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend',
})

##### ID_TO_AGE_RATIO

In [377]:
num_bins['ID_TO_AGE_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,ID_TO_AGE_RATIO,"(-0.001, 0.026]",12948,1260,11688,0.052632,0.097312,-0.205031,0.002411
1,ID_TO_AGE_RATIO,"(0.026, 0.052]",12952,1232,11720,0.052649,0.095120,-0.179824,0.001836
2,ID_TO_AGE_RATIO,"(0.052, 0.077]",12944,1080,11864,0.052616,0.083436,-0.035934,0.000069
3,ID_TO_AGE_RATIO,"(0.077, 0.101]",12948,1213,11735,0.052632,0.093682,-0.163003,0.001497
4,ID_TO_AGE_RATIO,"(0.101, 0.123]",12948,1076,11872,0.052632,0.083102,-0.031550,0.000053
5,ID_TO_AGE_RATIO,"(0.123, 0.143]",12947,1081,11866,0.052628,0.083494,-0.036691,0.000072
6,ID_TO_AGE_RATIO,"(0.143, 0.161]",12950,1031,11919,0.052641,0.079614,0.015123,0.000012
7,ID_TO_AGE_RATIO,"(0.161, 0.175]",12945,934,12011,0.052620,0.072151,0.121620,0.000740
8,ID_TO_AGE_RATIO,"(0.175, 0.186]",12948,823,12125,0.052632,0.063562,0.257587,0.003136
9,ID_TO_AGE_RATIO,"(0.186, 0.197]",12948,820,12128,0.052632,0.063330,0.261486,0.003227


In [378]:
merge_plan = [
    [0, 1, 2, 3, 4, 5],                              # negative zone avg ~-0.12
    [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18],  # everything else avg ~0.08
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="ID_TO_AGE_RATIO",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [379]:
num_bins['ID_TO_AGE_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,ID_TO_AGE_RATIO,"(-0.001, 0.143]]",77687,6942,70745,0.315791,0.089359,-0.110990,0.004076
6,ID_TO_AGE_RATIO,"(0.143, 0.469]]",168321,12918,155403,0.684209,0.076746,0.054918,0.002017
Totals,ID_TO_AGE_RATIO,None,246008,19860,226148,1.000000,0.080729,NaN,0.024184


In [380]:
rows.append({
    'feature': 'ID_TO_AGE_RATIO',
    'n_bins': 4,                      
    'n_special_bins': 0,               
    'special_bins_pct':0,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.024184,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend',
})

##### PA_RATIO_HIGH_RISK_YIELD_LOANS_360D

In [381]:
num_bins['PA_RATIO_HIGH_RISK_YIELD_LOANS_360D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_HIGH_RISK_YIELD_LOANS_360D,"(-0.001, 0.167]",120496,10034,110462,0.489805,0.083272,-0.033790,0.000567
1,PA_RATIO_HIGH_RISK_YIELD_LOANS_360D,"(0.167, 0.5]",7688,1008,6680,0.031251,0.131113,-0.541332,0.011486
2,PA_RATIO_HIGH_RISK_YIELD_LOANS_360D,"(0.5, 1.0]",5406,662,4744,0.021975,0.122457,-0.463112,0.005722
3,PA_RATIO_HIGH_RISK_YIELD_LOANS_360D,SPECIAL_-99999,112418,8156,104262,0.456969,0.072551,0.115671,0.005825
Totals,PA_RATIO_HIGH_RISK_YIELD_LOANS_360D,None,246008,19860,226148,1.000000,0.080729,NaN,0.023600


In [382]:
merge_plan = [
    [1,2]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_RATIO_HIGH_RISK_YIELD_LOANS_360D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [383]:
num_bins['PA_RATIO_HIGH_RISK_YIELD_LOANS_360D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_HIGH_RISK_YIELD_LOANS_360D,"(-0.001, 0.167]",120496,10034,110462,0.489805,0.083272,-0.033790,0.000567
1,PA_RATIO_HIGH_RISK_YIELD_LOANS_360D,"(0.167, 1.0]]",13094,1670,11424,0.053226,0.127539,-0.509589,0.017108
3,PA_RATIO_HIGH_RISK_YIELD_LOANS_360D,SPECIAL_-99999,112418,8156,104262,0.456969,0.072551,0.115671,0.005825
Totals,PA_RATIO_HIGH_RISK_YIELD_LOANS_360D,None,246008,19860,226148,1.000000,0.080729,NaN,0.023600


In [384]:
rows.append({
    'feature': 'PA_RATIO_HIGH_RISK_YIELD_LOANS_360D',
    'n_bins': 2,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.456969,      
    'bad_rate_trend': 'increasing',    
    'iv':0.023600,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend',
})

##### CREDIT_PER_PERSON

In [385]:
num_bins['CREDIT_PER_PERSON']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CREDIT_PER_PERSON,"(6749.999, 60000.0]",12672,877,11795,0.051511,0.069208,0.166442,1.331036e-03
1,CREDIT_PER_PERSON,"(60000.0, 82500.0]",12064,901,11163,0.049039,0.074685,0.084373,3.369770e-04
2,CREDIT_PER_PERSON,"(82500.0, 101880.0]",12379,1020,11359,0.050320,0.082398,-0.022274,2.519990e-05
3,CREDIT_PER_PERSON,"(101880.0, 123637.5]",12401,1075,11326,0.050409,0.086687,-0.077702,3.144319e-04
4,CREDIT_PER_PERSON,"(123637.5, 135988.5]",11989,952,11037,0.048734,0.079406,0.017962,1.560450e-05
5,CREDIT_PER_PERSON,"(135988.5, 157500.0]",13440,1297,12143,0.054632,0.096503,-0.195783,2.273476e-03
6,CREDIT_PER_PERSON,"(157500.0, 180000.0]",12672,1194,11478,0.051511,0.094223,-0.169359,1.586295e-03
7,CREDIT_PER_PERSON,"(180000.0, 210000.0]",10787,1075,9712,0.043848,0.099657,-0.231440,2.588331e-03
8,CREDIT_PER_PERSON,"(210000.0, 227250.0]",12615,1195,11420,0.051279,0.094728,-0.175262,1.695361e-03
9,CREDIT_PER_PERSON,"(227250.0, 256032.0]",12061,970,11091,0.049027,0.080425,0.004111,8.272162e-07


In [386]:
merge_plan = [
    [0,1,2,3,4],
    [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CREDIT_PER_PERSON",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [387]:
num_bins['CREDIT_PER_PERSON']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CREDIT_PER_PERSON,"(6749.999, 135988.5]]",61505,4825,56680,0.250012,0.078449,0.031129,0.000239
5,CREDIT_PER_PERSON,"(135988.5, 4031032.5]]",184501,15035,169466,0.749980,0.081490,-0.010210,0.000079
20,CREDIT_PER_PERSON,SPECIAL_-77777,2,0,2,0.000008,0.000000,0.000000,0.000000
Totals,CREDIT_PER_PERSON,None,246008,19860,226148,1.000000,0.080729,NaN,0.022446


In [388]:
rows.append({
    'feature': 'CREDIT_PER_PERSON',
    'n_bins': 2,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.000008,      
    'bad_rate_trend': 'increasing',    
    'iv':0.022446,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend with specical values -77777',
})

##### PA_FLAG_HAD_HIGH_RISK_LOAN

In [389]:
num_bins['PA_FLAG_HAD_HIGH_RISK_LOAN']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_FLAG_HAD_HIGH_RISK_LOAN,0.0,101824,7189,94635,0.413905,0.070602,0.144993,0.008189
1,PA_FLAG_HAD_HIGH_RISK_LOAN,1.0,131010,11880,119130,0.532544,0.090680,-0.127123,0.009078
2,PA_FLAG_HAD_HIGH_RISK_LOAN,SPECIAL_-99999,13174,791,12383,0.053551,0.060043,0.318300,0.004751
Totals,PA_FLAG_HAD_HIGH_RISK_LOAN,None,246008,19860,226148,1.000000,0.080729,NaN,0.022018


In [390]:
rows.append({
    'feature': 'PA_FLAG_HAD_HIGH_RISK_LOAN',
    'n_bins': 2,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.053551,      
    'bad_rate_trend': 'increasing',    
    'iv':0.022018,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend with specical values -99999',
})

##### PA_MIN_AMT_GOODS_PRICE

In [391]:
num_bins['PA_MIN_AMT_GOODS_PRICE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_MIN_AMT_GOODS_PRICE,"(-0.001, 16649.876]",11602,1093,10509,0.047161,0.094208,-0.169176,1.449109e-03
1,PA_MIN_AMT_GOODS_PRICE,"(16649.876, 19993.05]",11601,1084,10517,0.047157,0.093440,-0.160147,1.293527e-03
2,PA_MIN_AMT_GOODS_PRICE,"(19993.05, 22810.5]",11628,1071,10557,0.047267,0.092105,-0.144286,1.045447e-03
3,PA_MIN_AMT_GOODS_PRICE,"(22810.5, 25946.1]",11575,1037,10538,0.047051,0.089590,-0.113826,6.394418e-04
4,PA_MIN_AMT_GOODS_PRICE,"(25946.1, 29205.0]",11896,1021,10875,0.048356,0.085827,-0.066798,2.218955e-04
5,PA_MIN_AMT_GOODS_PRICE,"(29205.0, 32611.5]",11308,955,10353,0.045966,0.084453,-0.049162,1.134088e-04
6,PA_MIN_AMT_GOODS_PRICE,"(32611.5, 36517.5]",11604,959,10645,0.047169,0.082644,-0.025527,3.106878e-05
7,PA_MIN_AMT_GOODS_PRICE,"(36517.5, 41145.202]",11598,1016,10582,0.047145,0.087601,-0.089201,3.894280e-04
8,PA_MIN_AMT_GOODS_PRICE,"(41145.202, 45000.0]",22303,2145,20158,0.090660,0.096175,-0.192020,3.623370e-03
9,PA_MIN_AMT_GOODS_PRICE,"(45000.0, 45708.75]",900,75,825,0.003658,0.083333,-0.034587,4.440297e-06


In [392]:
merge_plan = [

    [0,1,2,3,4,5,6,7,8],
    [9,10],
    [11,12,13],
    [14,15,16,17],
    [18,19]

]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_MIN_AMT_GOODS_PRICE",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [393]:
num_bins['PA_MIN_AMT_GOODS_PRICE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_MIN_AMT_GOODS_PRICE,"(-0.001, 45000.0]]",115115,10381,104734,0.467932,0.090179,-0.121035,0.007212
9,PA_MIN_AMT_GOODS_PRICE,"(45000.0, 53010.0]]",12520,993,11527,0.050893,0.079313,0.019235,0.000019
11,PA_MIN_AMT_GOODS_PRICE,"(53010.0, 79290.0]]",34791,2693,32098,0.141422,0.077405,0.045656,0.000289
14,PA_MIN_AMT_GOODS_PRICE,"(79290.0, 155470.95]]",46401,3380,43021,0.188616,0.072843,0.111331,0.002231
18,PA_MIN_AMT_GOODS_PRICE,"(155470.95, 4050000.0]]",23203,1577,21626,0.094318,0.067965,0.185890,0.003015
20,PA_MIN_AMT_GOODS_PRICE,SPECIAL_-99999,13978,836,13142,0.056819,0.059808,0.322458,0.005165
Totals,PA_MIN_AMT_GOODS_PRICE,None,246008,19860,226148,1.000000,0.080729,NaN,0.021774


In [394]:
rows.append({
    'feature': 'PA_MIN_AMT_GOODS_PRICE',
    'n_bins': 5,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.056819,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.021774,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend with specical values -99999',
})

##### PA_RATIO_HIGH_RISK_CHANNEL

In [395]:
num_bins['PA_RATIO_HIGH_RISK_CHANNEL']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_HIGH_RISK_CHANNEL,"(-0.001, 0.0455]",186284,14417,171867,0.757227,0.077393,0.045831,0.001560
1,PA_RATIO_HIGH_RISK_CHANNEL,"(0.0455, 0.167]",13890,1342,12548,0.056462,0.096616,-0.197082,0.002382
2,PA_RATIO_HIGH_RISK_CHANNEL,"(0.167, 0.25]",10063,940,9123,0.040905,0.093412,-0.159808,0.001117
3,PA_RATIO_HIGH_RISK_CHANNEL,"(0.25, 0.5]",16339,1555,14784,0.066417,0.095171,-0.180412,0.002332
4,PA_RATIO_HIGH_RISK_CHANNEL,"(0.5, 1.0]",6258,815,5443,0.025438,0.130233,-0.533584,0.009054
5,PA_RATIO_HIGH_RISK_CHANNEL,SPECIAL_-99999,13174,791,12383,0.053551,0.060043,0.318300,0.004751
Totals,PA_RATIO_HIGH_RISK_CHANNEL,None,246008,19860,226148,1.000000,0.080729,NaN,0.021197


In [396]:
merge_plan = [
    [1,2],
    [3,4]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PA_RATIO_HIGH_RISK_CHANNEL",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [397]:
num_bins['PA_RATIO_HIGH_RISK_CHANNEL']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PA_RATIO_HIGH_RISK_CHANNEL,"(-0.001, 0.0455]",186284,14417,171867,0.757227,0.077393,0.045831,0.001560
1,PA_RATIO_HIGH_RISK_CHANNEL,"(0.0455, 0.25]]",23953,2282,21671,0.097367,0.095270,-0.181559,0.003464
3,PA_RATIO_HIGH_RISK_CHANNEL,"(0.25, 1.0]]",22597,2370,20227,0.091855,0.104881,-0.288354,0.008620
5,PA_RATIO_HIGH_RISK_CHANNEL,SPECIAL_-99999,13174,791,12383,0.053551,0.060043,0.318300,0.004751
Totals,PA_RATIO_HIGH_RISK_CHANNEL,None,246008,19860,226148,1.000000,0.080729,NaN,0.021197


In [398]:
rows.append({
    'feature': 'PA_RATIO_HIGH_RISK_CHANNEL',
    'n_bins': 3,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.053551,      
    'bad_rate_trend': 'increasing',    
    'iv':0.021197,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend with specical values -99999',
})

## MANUAL BINNING CATEGORICAL


In [399]:
# MISSING SPECIAL VALUES GROUP MISSING VALUES
# special_bins_pct is 0 here for all because not filled 
# categorical feature dont have trend like increasing or decreasing they are risk seprable

##### OCCUPATION_GROUP

In [11]:
iv_df[iv_df['feature']=='OCCUPATION_GROUP']

,feature,IV
502,OCCUPATION_GROUP,0.074611


In [14]:
temp = cat_df[cat_df['feature']=='OCCUPATION_GROUP']
temp

,feature,Bin,Count,Event,Non-event,Count(%),Event rate,WoE,IV,is_rare_bin,rare_categories
69,OCCUPATION_GROUP,LOW_SKILL,75825,8199,67626,0.308222,0.108131,-0.322502,0.036703,False,NaN
70,OCCUPATION_GROUP,MANAGERS,17096,1064,16032,0.069494,0.062237,0.280069,0.004850,False,NaN
71,OCCUPATION_GROUP,MISSING,76940,4995,71945,0.312754,0.064921,0.234983,0.015655,False,NaN
72,OCCUPATION_GROUP,SERVICE,28294,2644,25650,0.115013,0.093447,-0.160231,0.003158,False,NaN
73,OCCUPATION_GROUP,SKILLED_PRO,47853,2958,44895,0.194518,0.061814,0.287331,0.014245,False,NaN
74,OCCUPATION_GROUP,RARE,0,0,0,0.000000,0.000000,0.000000,0.000000,True,NaN


In [402]:
rows.append({
    'feature':'OCCUPATION_GROUP',
    'n_bins':4,
    'n_special_bins':1, # missing
    'special_bins_pct':0.312754,
    'bad_rate_trend':'Risk-separated',
    'iv':0.074611,
    'keep':'Yes',
    'comment':'have high iv and clear risk separation' ,
})

##### ORG_GROUP

In [403]:
iv_df[iv_df['feature']=='ORG_GROUP']

,feature,IV
503,ORG_GROUP,0.057884


In [404]:
cat_df[cat_df['feature']=='ORG_GROUP']

,feature,Bin,Count,Event,Non-event,Count(%),Event rate,WoE,IV,is_rare_bin,rare_categories
64,ORG_GROUP,ORG_OTHER,65415,3914,61501,0.265906,0.059833,0.322012,0.024109,False,NaN
65,ORG_GROUP,ORG_PRIVATE,103202,9283,93919,0.419507,0.089950,-0.118234,0.006163,False,NaN
66,ORG_GROUP,ORG_STABLE,36102,2263,33839,0.146751,0.062684,0.272441,0.009722,False,NaN
67,ORG_GROUP,ORG_UNSTABLE,41289,4400,36889,0.167836,0.106566,-0.306173,0.017890,False,NaN
68,ORG_GROUP,RARE,0,0,0,0.000000,0.000000,0.000000,0.000000,True,NaN


In [405]:
rows.append({
    'feature':'ORG_GROUP',
    'n_bins':4,
    'n_special_bins':0, # missing
    'special_bins_pct':0,
    'bad_rate_trend':'Risk-separated',
    'iv':0.057884,
    'keep':'Yes',
    'comment':'have high iv and clear risk separation' ,
})

##### NAME_INCOME_TYPE

In [406]:
iv_df[iv_df['feature']=='NAME_INCOME_TYPE']

,feature,IV
504,NAME_INCOME_TYPE,0.055916


In [407]:
cat_df[cat_df['feature']=='NAME_INCOME_TYPE']

,feature,Bin,Count,Event,Non-event,Count(%),Event rate,WoE,IV,is_rare_bin,rare_categories
17,NAME_INCOME_TYPE,Commercial associate,57273,4348,52925,0.232810,0.075917,0.066678,0.001007,False,NaN
18,NAME_INCOME_TYPE,Pensioner,44134,2365,41769,0.179401,0.053587,0.438894,0.028798,False,NaN
19,NAME_INCOME_TYPE,RARE,40,5,35,0.000163,0.125000,-0.486572,0.000047,True,"Unemployed, Student, Businessman, Maternity leave"
20,NAME_INCOME_TYPE,State servant,17518,1024,16494,0.071209,0.058454,0.346798,0.007412,False,NaN
21,NAME_INCOME_TYPE,Working,127043,12118,114925,0.516418,0.095385,-0.182894,0.018653,False,NaN


In [408]:
rows.append({
    'feature':'NAME_INCOME_TYPE',
    'n_bins':4,
    'n_special_bins':1, # RARE
    'special_bins_pct':0.000163,
    'bad_rate_trend':'Risk-separated',
    'iv':0.055916,
    'keep':'Yes',
    'comment':'have high iv and clear risk separation with Rare category' ,
})

##### NAME_EDUCATION_TYPE

In [409]:
iv_df[iv_df['feature']=='NAME_EDUCATION_TYPE']

,feature,IV
505,NAME_EDUCATION_TYPE,0.04889


In [410]:
cat_df[cat_df['feature']=='NAME_EDUCATION_TYPE']

,feature,Bin,Count,Event,Non-event,Count(%),Event rate,WoE,IV,is_rare_bin,rare_categories
22,NAME_EDUCATION_TYPE,Higher education,59926,3228,56698,0.243594,0.053866,0.433394,0.038214,False,NaN
23,NAME_EDUCATION_TYPE,Lower secondary,3025,335,2690,0.012296,0.110744,-0.349316,0.001737,False,NaN
24,NAME_EDUCATION_TYPE,MISSING,174768,15582,159186,0.710416,0.089158,-0.108525,0.008757,False,NaN
25,NAME_EDUCATION_TYPE,Secondary,8289,715,7574,0.033694,0.086259,-0.072288,0.000181,False,NaN
26,NAME_EDUCATION_TYPE,RARE,0,0,0,0.000000,0.000000,0.000000,0.000000,True,NaN


In [411]:
0.710416

0.710416

In [412]:
rows.append({
    'feature':'NAME_EDUCATION_TYPE',
    'n_bins':3,
    'n_special_bins':1, # MISSING 
    'special_bins_pct':0,
    'bad_rate_trend':'Risk-separated',
    'iv':0.04889,
    'keep':'Yes',
    'comment':'clear risk separation with 0.710416 missing values',
})

##### CODE_GENDER

In [413]:
iv_df[iv_df['feature']=='CODE_GENDER']

,feature,IV
506,CODE_GENDER,0.038421


In [414]:
cat_df[cat_df['feature']=='CODE_GENDER']

,feature,Bin,Count,Event,Non-event,Count(%),Event rate,WoE,IV,is_rare_bin,rare_categories
3,CODE_GENDER,F,161887,11334,150553,0.658056,0.070012,0.154026,0.014638,False,NaN
4,CODE_GENDER,M,84119,8526,75593,0.341936,0.101356,-0.250239,0.023783,False,NaN
5,CODE_GENDER,RARE,2,0,2,0.000008,0.000000,0.000000,0.000000,True,Missing


In [415]:
rows.append({
    'feature':'CODE_GENDER',
    'n_bins':2,
    'n_special_bins':1, # MISSING i.e RARE
    'special_bins_pct':0.000008,
    'bad_rate_trend':'Risk-separated',
    'iv':0.038421,
    'keep':'Yes',
    'comment':'have high iv and clear risk separation' ,
})

##### NAME_FAMILY_STATUS

In [416]:
iv_df[iv_df['feature']=='NAME_FAMILY_STATUS']

,feature,IV
510,NAME_FAMILY_STATUS,0.023334


In [417]:
cat_df[cat_df['feature']=='NAME_FAMILY_STATUS']

,feature,Bin,Count,Event,Non-event,Count(%),Event rate,WoE,IV,is_rare_bin,rare_categories
27,NAME_FAMILY_STATUS,Civil marriage,23812,2385,21427,0.096794,0.100160,-0.237029,0.006007,False,NaN
28,NAME_FAMILY_STATUS,Married,157153,11840,145313,0.638813,0.075341,0.074924,0.003475,False,NaN
29,NAME_FAMILY_STATUS,RARE,2,0,2,0.000008,0.000000,0.000000,0.000000,True,Missing
30,NAME_FAMILY_STATUS,Separated,15840,1265,14575,0.064388,0.079861,0.011754,0.000009,False,NaN
31,NAME_FAMILY_STATUS,Single,36359,3616,32743,0.147796,0.099453,-0.229161,0.008545,False,NaN
32,NAME_FAMILY_STATUS,Widow,12842,754,12088,0.052202,0.058714,0.342094,0.005298,False,NaN


In [418]:
rows.append({
    'feature':'NAME_FAMILY_STATUS',
    'n_bins':5,
    'n_special_bins':1, # RARE i.e missing
    'special_bins_pct':0.000008,
    'bad_rate_trend':'Risk-separated',
    'iv':0.023334,
    'keep':'Yes',
    'comment':'have high iv and clear risk separation' ,
})

## 3rd UPDATE FEATURES

##### EXT_SOURCE_RANGE

In [419]:
num_bins['EXT_SOURCE_RANGE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,EXT_SOURCE_RANGE,"(-0.001, 0.0175]",36881,3467,33414,0.149918,0.094005,-0.166797,0.004473
1,EXT_SOURCE_RANGE,"(0.0175, 0.0442]",12293,852,11441,0.049970,0.069308,0.164890,0.001268
2,EXT_SOURCE_RANGE,"(0.0442, 0.0695]",12294,794,11500,0.049974,0.064584,0.240537,0.002615
3,EXT_SOURCE_RANGE,"(0.0695, 0.0941]",12293,861,11432,0.049970,0.070040,0.153595,0.001106
4,EXT_SOURCE_RANGE,"(0.0941, 0.118]",12293,779,11514,0.049970,0.063369,0.260826,0.003049
5,EXT_SOURCE_RANGE,"(0.118, 0.142]",12294,812,11482,0.049974,0.066048,0.216554,0.002141
6,EXT_SOURCE_RANGE,"(0.142, 0.166]",12293,885,11408,0.049970,0.071992,0.124001,0.000729
7,EXT_SOURCE_RANGE,"(0.166, 0.191]",12294,811,11483,0.049974,0.065967,0.217873,0.002166
8,EXT_SOURCE_RANGE,"(0.191, 0.216]",12293,920,11373,0.049970,0.074839,0.082142,0.000326
9,EXT_SOURCE_RANGE,"(0.216, 0.243]",12293,889,11404,0.049970,0.072318,0.119140,0.000675


In [420]:
merge_plan = [
    [0,1,2,3, 4,5,6,7],  
    [8,9, 10, 11],
    [12,13, 14,],
    [15, 16,17]
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="EXT_SOURCE_RANGE",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)


In [421]:
num_bins['EXT_SOURCE_RANGE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,EXT_SOURCE_RANGE,"(-0.001, 0.191]]",122935,9261,113674,0.499720,0.075332,0.075041,0.002727
8,EXT_SOURCE_RANGE,"(0.191, 0.302]]",49173,3736,45437,0.199884,0.075977,0.065829,0.000843
12,EXT_SOURCE_RANGE,"(0.302, 0.412]]",36880,3192,33688,0.149914,0.086551,-0.075988,0.000894
15,EXT_SOURCE_RANGE,"(0.412, 0.922]]",36881,3659,33222,0.149918,0.099211,-0.226460,0.008455
18,EXT_SOURCE_RANGE,SPECIAL_-99999,139,12,127,0.000565,0.086331,-0.073202,0.000003
Totals,EXT_SOURCE_RANGE,None,246008,19860,226148,1.000000,0.080729,NaN,0.028551


In [422]:
rows.append({
    'feature': 'EXT_SOURCE_RANGE',
    'n_bins': 4,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.000565,      
    'bad_rate_trend': 'increasing',    
    'iv':0.028551,                    
    'keep': 'Yes',                     
    'comment': 'forced slight trend clear monotonic trend ',
})


##### BB_RECENT_MONTH_OF_DPD

In [423]:
num_bins['BB_RECENT_MONTH_OF_DPD']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,BB_RECENT_MONTH_OF_DPD,"(-93.001, -66.0]",1343,95,1248,0.005459,0.070737,0.142939,0.000105
1,BB_RECENT_MONTH_OF_DPD,"(-66.0, -51.0]",1360,91,1269,0.005528,0.066912,0.202643,0.000209
2,BB_RECENT_MONTH_OF_DPD,"(-51.0, -42.0]",1311,104,1207,0.005329,0.079329,0.019020,0.000002
3,BB_RECENT_MONTH_OF_DPD,"(-42.0, -35.0]",1369,102,1267,0.005565,0.074507,0.086952,0.000041
4,BB_RECENT_MONTH_OF_DPD,"(-35.0, -29.0]",1368,115,1253,0.005561,0.084064,-0.044118,0.000011
5,BB_RECENT_MONTH_OF_DPD,"(-29.0, -24.0]",1337,110,1227,0.005435,0.082274,-0.020635,0.000002
6,BB_RECENT_MONTH_OF_DPD,"(-24.0, -20.0]",1344,111,1233,0.005463,0.082589,-0.024807,0.000003
7,BB_RECENT_MONTH_OF_DPD,"(-20.0, -17.0]",1079,102,977,0.004386,0.094532,-0.172968,0.000141
8,BB_RECENT_MONTH_OF_DPD,"(-17.0, -13.0]",1627,156,1471,0.006614,0.095882,-0.188640,0.000255
9,BB_RECENT_MONTH_OF_DPD,"(-13.0, -11.0]",943,99,844,0.003833,0.104984,-0.289449,0.000363


In [424]:
merge_plan = [
    [1,2],  
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="BB_RECENT_MONTH_OF_DPD",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)


In [425]:
num_bins['BB_RECENT_MONTH_OF_DPD']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,BB_RECENT_MONTH_OF_DPD,"(-93.001, -66.0]",1343,95,1248,0.005459,0.070737,0.142939,0.000105
1,BB_RECENT_MONTH_OF_DPD,"(-66.0, -42.0]]",2671,195,2476,0.010857,0.073006,0.108918,0.000123
3,BB_RECENT_MONTH_OF_DPD,"(-42.0, -35.0]",1369,102,1267,0.005565,0.074507,0.086952,0.000041
4,BB_RECENT_MONTH_OF_DPD,"(-35.0, -29.0]",1368,115,1253,0.005561,0.084064,-0.044118,0.000011
5,BB_RECENT_MONTH_OF_DPD,"(-29.0, -24.0]",1337,110,1227,0.005435,0.082274,-0.020635,0.000002
6,BB_RECENT_MONTH_OF_DPD,"(-24.0, -20.0]",1344,111,1233,0.005463,0.082589,-0.024807,0.000003
7,BB_RECENT_MONTH_OF_DPD,"(-20.0, -17.0]",1079,102,977,0.004386,0.094532,-0.172968,0.000141
8,BB_RECENT_MONTH_OF_DPD,"(-17.0, -13.0]",1627,156,1471,0.006614,0.095882,-0.188640,0.000255
9,BB_RECENT_MONTH_OF_DPD,"(-13.0, -11.0]",943,99,844,0.003833,0.104984,-0.289449,0.000363
10,BB_RECENT_MONTH_OF_DPD,"(-11.0, -8.0]",1792,182,1610,0.007284,0.101562,-0.252499,0.000516


In [426]:
rows.append({
    'feature': 'BB_RECENT_MONTH_OF_DPD',
    'n_bins': 2,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.143264,      
    'bad_rate_trend': 'increasing',    
    'iv':0.027100,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend ',
})


##### B_OVERDUE_TO_CREDIT_RATIO

In [427]:
num_bins['B_OVERDUE_TO_CREDIT_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_OVERDUE_TO_CREDIT_RATIO,"(-0.001, 0.000278]",157426,11141,146285,0.639922,0.070770,0.142443,0.012233
1,B_OVERDUE_TO_CREDIT_RATIO,"(0.000278, 0.00272]",10495,883,9612,0.042661,0.084135,-0.045040,0.000088
2,B_OVERDUE_TO_CREDIT_RATIO,"(0.00272, 0.00616]",10495,981,9514,0.042661,0.093473,-0.160535,0.001176
3,B_OVERDUE_TO_CREDIT_RATIO,"(0.00616, 0.0121]",10495,996,9499,0.042661,0.094902,-0.177287,0.001444
4,B_OVERDUE_TO_CREDIT_RATIO,"(0.0121, 0.0257]",10495,1021,9474,0.042661,0.097284,-0.204713,0.001948
5,B_OVERDUE_TO_CREDIT_RATIO,"(0.0257, 377.0]",10496,1201,9295,0.042665,0.114425,-0.386160,0.007481
6,B_OVERDUE_TO_CREDIT_RATIO,SPECIAL_-99999,36106,3637,32469,0.146768,0.100731,-0.243355,0.009627
Totals,B_OVERDUE_TO_CREDIT_RATIO,None,246008,19860,226148,1.000000,0.080729,NaN,0.033997


In [428]:
merge_plan = [
    [1,2],  
    [3,4,5,]
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="B_OVERDUE_TO_CREDIT_RATIO",
    merge_plan=merge_plan,
    num_bins=num_bins
)
all_woe_mapping_num.update(woe_mappings)


In [429]:
num_bins['B_OVERDUE_TO_CREDIT_RATIO']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,B_OVERDUE_TO_CREDIT_RATIO,"(-0.001, 0.000278]",157426,11141,146285,0.639922,0.070770,0.142443,0.012233
1,B_OVERDUE_TO_CREDIT_RATIO,"(0.000278, 0.00616]]",20990,1864,19126,0.085322,0.088804,-0.104158,0.000967
3,B_OVERDUE_TO_CREDIT_RATIO,"(0.00616, 377.0]]",31486,3218,28268,0.127988,0.102204,-0.259512,0.009611
6,B_OVERDUE_TO_CREDIT_RATIO,SPECIAL_-99999,36106,3637,32469,0.146768,0.100731,-0.243355,0.009627
Totals,B_OVERDUE_TO_CREDIT_RATIO,None,246008,19860,226148,1.000000,0.080729,NaN,0.033997


In [430]:
rows.append({
    'feature': 'B_OVERDUE_TO_CREDIT_RATIO',
    'n_bins': 3,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.146768,      
    'bad_rate_trend': 'increasing',    
    'iv':0.033997,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend ',
})


##### IP_MIN_RATIO_PAYMENT_INSTALMENT

In [431]:
num_bins['IP_MIN_RATIO_PAYMENT_INSTALMENT']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_MIN_RATIO_PAYMENT_INSTALMENT,"(-0.001, 0.000193]",11623,983,10640,0.047246,0.084574,-0.050715,1.241329e-04
1,IP_MIN_RATIO_PAYMENT_INSTALMENT,"(0.000193, 0.000743]",11623,1176,10447,0.047246,0.101179,-0.248286,3.232458e-03
2,IP_MIN_RATIO_PAYMENT_INSTALMENT,"(0.000743, 0.00231]",11623,1076,10547,0.047246,0.092575,-0.149891,1.130426e-03
3,IP_MIN_RATIO_PAYMENT_INSTALMENT,"(0.00231, 0.00633]",11623,1160,10463,0.047246,0.099802,-0.233057,2.829939e-03
4,IP_MIN_RATIO_PAYMENT_INSTALMENT,"(0.00633, 0.0147]",11623,1122,10501,0.047246,0.096533,-0.196124,1.973261e-03
5,IP_MIN_RATIO_PAYMENT_INSTALMENT,"(0.0147, 0.0351]",11623,1220,10403,0.047246,0.104964,-0.289239,4.462709e-03
6,IP_MIN_RATIO_PAYMENT_INSTALMENT,"(0.0351, 0.0937]",11623,1134,10489,0.047246,0.097565,-0.207906,2.228456e-03
7,IP_MIN_RATIO_PAYMENT_INSTALMENT,"(0.0937, 0.281]",11623,1082,10541,0.047246,0.093091,-0.156021,1.227930e-03
8,IP_MIN_RATIO_PAYMENT_INSTALMENT,"(0.281, 1.0]",139463,9988,129475,0.566904,0.071618,0.129621,9.022023e-03
9,IP_MIN_RATIO_PAYMENT_INSTALMENT,"(1.0, 1.6]",12,1,11,0.000049,0.083333,-0.034587,5.920396e-08


In [432]:
merge_plan = [
    [0, 1, 2, 3, 4, 5, 6],   
    [7,8, 9],                      

]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_MIN_RATIO_PAYMENT_INSTALMENT",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [433]:
num_bins['IP_MIN_RATIO_PAYMENT_INSTALMENT']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_MIN_RATIO_PAYMENT_INSTALMENT,"(-0.001, 0.0937]]",81361,7871,73490,0.330725,0.096742,-0.198518,0.014166
7,IP_MIN_RATIO_PAYMENT_INSTALMENT,"(0.0937, 1.6]]",151098,11071,140027,0.614200,0.073270,0.105024,0.006483
10,IP_MIN_RATIO_PAYMENT_INSTALMENT,SPECIAL_-88888,861,159,702,0.003500,0.184669,-0.947453,0.004644
11,IP_MIN_RATIO_PAYMENT_INSTALMENT,SPECIAL_-99999,12688,759,11929,0.051576,0.059820,0.322244,0.004683
Totals,IP_MIN_RATIO_PAYMENT_INSTALMENT,None,246008,19860,226148,1.000000,0.080729,NaN,0.035558


In [434]:
rows.append({
    'feature': 'IP_MIN_RATIO_PAYMENT_INSTALMENT',
    'n_bins': 2,                      
    'n_special_bins': 2 ,               
    'special_bins_pct':0.055076,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.035558,                    
    'keep': 'Yes',                     
    'comment': 'enforced clear monotonic trend contain -88888 and -99999 ',
})


##### IP_RATIO_LATE_PAYMENTS_90D

In [435]:
num_bins['IP_WORST_DPD_90D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_WORST_DPD_90D,"(-0.001, 1.0]",46586,4196,42390,0.189368,0.090070,-0.119701,0.002853
1,IP_WORST_DPD_90D,"(1.0, 2.0]",3938,451,3487,0.016008,0.114525,-0.387152,0.002822
2,IP_WORST_DPD_90D,"(2.0, 3.0]",2810,345,2465,0.011422,0.122776,-0.466079,0.003016
3,IP_WORST_DPD_90D,"(3.0, 4.0]",2194,298,1896,0.008918,0.135825,-0.582074,0.003854
4,IP_WORST_DPD_90D,"(4.0, 6.0]",2634,387,2247,0.010707,0.146925,-0.673555,0.006433
5,IP_WORST_DPD_90D,"(6.0, 86.0]",2691,551,2140,0.010939,0.204757,-1.075656,0.019664
6,IP_WORST_DPD_90D,SPECIAL_-66666,98168,7907,90261,0.399044,0.080546,0.002475,0.000002
7,IP_WORST_DPD_90D,SPECIAL_-99999,86987,5725,81262,0.353594,0.065814,0.220354,0.015659
Totals,IP_WORST_DPD_90D,None,246008,19860,226148,1.000000,0.080729,NaN,0.054304


In [436]:
merge_plan = [
    [1, 2, 3,4,5],   
                  

]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_WORST_DPD_90D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [437]:
num_bins['IP_WORST_DPD_90D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_WORST_DPD_90D,"(-0.001, 1.0]",46586,4196,42390,0.189368,0.090070,-0.119701,0.002853
1,IP_WORST_DPD_90D,"(1.0, 86.0]]",14267,2032,12235,0.057994,0.142427,-0.637202,0.030722
6,IP_WORST_DPD_90D,SPECIAL_-66666,98168,7907,90261,0.399044,0.080546,0.002475,0.000002
7,IP_WORST_DPD_90D,SPECIAL_-99999,86987,5725,81262,0.353594,0.065814,0.220354,0.015659
Totals,IP_WORST_DPD_90D,None,246008,19860,226148,1.000000,0.080729,NaN,0.054304


In [438]:
0.399044 + 0.353594

0.752638

In [439]:
rows.append({
    'feature':'IP_WORST_DPD_90D',
    'n_bins':2,
    'n_special_bins':1,
    'special_bins_pct':0.752638,
    'bad_rate_trend':'increasing',
    'iv':0.054304,
    'keep':'Yes',
    'comment':'clear trend with  high SC_-99999 values ' ,
})

##### IP_RATIO_LATE_PAYMENTS_2160D

In [440]:
num_bins['IP_RATIO_LATE_PAYMENTS_2160D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_RATIO_LATE_PAYMENTS_2160D,"(-0.001, 0.0241]",125447,8274,117173,0.509931,0.065956,0.218051,0.022134
1,IP_RATIO_LATE_PAYMENTS_2160D,"(0.0241, 0.04]",11797,896,10901,0.047954,0.075952,0.066187,0.000204
2,IP_RATIO_LATE_PAYMENTS_2160D,"(0.04, 0.0571]",11033,877,10156,0.044848,0.079489,0.016831,0.000013
3,IP_RATIO_LATE_PAYMENTS_2160D,"(0.0571, 0.0778]",11382,986,10396,0.046267,0.086628,-0.076962,0.000283
4,IP_RATIO_LATE_PAYMENTS_2160D,"(0.0778, 0.1]",11532,1140,10392,0.046877,0.098855,-0.222474,0.002547
5,IP_RATIO_LATE_PAYMENTS_2160D,"(0.1, 0.135]",11306,1103,10203,0.045958,0.097559,-0.207834,0.002166
6,IP_RATIO_LATE_PAYMENTS_2160D,"(0.135, 0.173]",11382,1212,10170,0.046267,0.106484,-0.305312,0.004902
7,IP_RATIO_LATE_PAYMENTS_2160D,"(0.173, 0.231]",11660,1271,10389,0.047397,0.109005,-0.331538,0.005987
8,IP_RATIO_LATE_PAYMENTS_2160D,"(0.231, 0.321]",11163,1283,9880,0.045377,0.114933,-0.391171,0.008181
9,IP_RATIO_LATE_PAYMENTS_2160D,"(0.321, 1.0]",11357,1605,9752,0.046165,0.141323,-0.628133,0.023677


In [441]:
merge_plan = [
    [1,2,3],
    [4,5],
    [6,7,],
    [8,9]

]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_RATIO_LATE_PAYMENTS_2160D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [442]:
num_bins['IP_RATIO_LATE_PAYMENTS_2160D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_RATIO_LATE_PAYMENTS_2160D,"(-0.001, 0.0241]",125447,8274,117173,0.509931,0.065956,0.218051,2.213418e-02
1,IP_RATIO_LATE_PAYMENTS_2160D,"(0.0241, 0.0778]]",34212,2759,31453,0.139069,0.080644,0.001144,1.819295e-07
4,IP_RATIO_LATE_PAYMENTS_2160D,"(0.0778, 0.135]]",22838,2243,20595,0.092834,0.098214,-0.215248,4.707883e-03
6,IP_RATIO_LATE_PAYMENTS_2160D,"(0.135, 0.231]]",23042,2483,20559,0.093664,0.107760,-0.318651,1.087100e-02
8,IP_RATIO_LATE_PAYMENTS_2160D,"(0.231, 1.0]]",22520,2888,19632,0.091542,0.128242,-0.515885,3.023474e-02
10,IP_RATIO_LATE_PAYMENTS_2160D,SPECIAL_-99999,17949,1213,16736,0.072961,0.067580,0.191983,2.481788e-03
Totals,IP_RATIO_LATE_PAYMENTS_2160D,None,246008,19860,226148,1.000000,0.080729,NaN,7.257634e-02


In [443]:
rows.append({
    'feature': 'IP_RATIO_LATE_PAYMENTS_2160D',
    'n_bins': 5,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.072961,      
    'bad_rate_trend': 'increasing',    
    'iv':0.072576,                    
    'keep': 'Yes',                     
    'comment': 'enforced clear monotonic trend contain -99999 ',
})


##### IP_NUM_COMPLETED_LOANS

In [444]:
num_bins['IP_NUM_COMPLETED_LOANS']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_NUM_COMPLETED_LOANS,"(-0.001, 1.0]",96060,9491,86569,0.390475,0.098803,-0.221884,0.021101
1,IP_NUM_COMPLETED_LOANS,"(1.0, 2.0]",57593,4451,53142,0.234110,0.077284,0.047357,0.000515
2,IP_NUM_COMPLETED_LOANS,"(2.0, 3.0]",35463,2481,32982,0.144154,0.069960,0.154818,0.003238
3,IP_NUM_COMPLETED_LOANS,"(3.0, 4.0]",20299,1274,19025,0.082514,0.062762,0.271110,0.005416
4,IP_NUM_COMPLETED_LOANS,"(4.0, 5.0]",11135,649,10486,0.045263,0.058285,0.349882,0.004790
5,IP_NUM_COMPLETED_LOANS,"(5.0, 6.0]",5891,346,5545,0.023946,0.058734,0.341731,0.002425
6,IP_NUM_COMPLETED_LOANS,"(6.0, 26.0]",6881,409,6472,0.027971,0.059439,0.329043,0.002640
7,IP_NUM_COMPLETED_LOANS,SPECIAL_-99999,12686,759,11927,0.051567,0.059830,0.322076,0.004677
Totals,IP_NUM_COMPLETED_LOANS,None,246008,19860,226148,1.000000,0.080729,NaN,0.044802


In [445]:
merge_plan = [
    [4,5,6]
   
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_NUM_COMPLETED_LOANS",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [446]:
num_bins['IP_NUM_COMPLETED_LOANS']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_NUM_COMPLETED_LOANS,"(-0.001, 1.0]",96060,9491,86569,0.390475,0.098803,-0.221884,0.021101
1,IP_NUM_COMPLETED_LOANS,"(1.0, 2.0]",57593,4451,53142,0.234110,0.077284,0.047357,0.000515
2,IP_NUM_COMPLETED_LOANS,"(2.0, 3.0]",35463,2481,32982,0.144154,0.069960,0.154818,0.003238
3,IP_NUM_COMPLETED_LOANS,"(3.0, 4.0]",20299,1274,19025,0.082514,0.062762,0.271110,0.005416
4,IP_NUM_COMPLETED_LOANS,"(4.0, 26.0]]",23907,1404,22503,0.097180,0.058728,0.341841,0.009849
7,IP_NUM_COMPLETED_LOANS,SPECIAL_-99999,12686,759,11927,0.051567,0.059830,0.322076,0.004677
Totals,IP_NUM_COMPLETED_LOANS,None,246008,19860,226148,1.000000,0.080729,NaN,0.044802


In [447]:
rows.append({
    'feature': 'IP_NUM_COMPLETED_LOANS',
    'n_bins': 5,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.051567,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.044802,                    
    'keep': 'Yes',                     
    'comment': 'enforced clear monotonic trend contain -99999 ',
})


##### IP_RATIO_AMT_PAID_OWED_360D

In [448]:
num_bins['IP_RATIO_AMT_PAID_OWED_360D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_RATIO_AMT_PAID_OWED_360D,"(-0.0001, 0.8]",8790,1198,7592,0.035731,0.136291,-0.586040,0.015677
1,IP_RATIO_AMT_PAID_OWED_360D,"(0.8, 0.9]",8537,1082,7455,0.034702,0.126742,-0.502408,0.010810
2,IP_RATIO_AMT_PAID_OWED_360D,"(0.9, 0.9452]",8663,1033,7630,0.035214,0.119243,-0.432861,0.007911
3,IP_RATIO_AMT_PAID_OWED_360D,"(0.9452, 0.9998]",8663,1051,7612,0.035214,0.121321,-0.452498,0.008716
4,IP_RATIO_AMT_PAID_OWED_360D,"(0.9998, 1.0]",122003,8859,113144,0.495931,0.072613,0.114745,0.006223
5,IP_RATIO_AMT_PAID_OWED_360D,"(1.0, 1.3306]",7943,488,7455,0.032288,0.061438,0.293843,0.002466
6,IP_RATIO_AMT_PAID_OWED_360D,"(1.3306, 2.0]",8664,451,8213,0.035218,0.052054,0.469524,0.006389
7,IP_RATIO_AMT_PAID_OWED_360D,SPECIAL_-99999,72745,5698,67047,0.295702,0.078328,0.032797,0.000314
Totals,IP_RATIO_AMT_PAID_OWED_360D,None,246008,19860,226148,1.000000,0.080729,NaN,0.058506


In [449]:
merge_plan = [
    [0,1],
    [2,3],
    [4],
    [5,6]
   
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_RATIO_AMT_PAID_OWED_360D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [450]:
num_bins['IP_RATIO_AMT_PAID_OWED_360D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_RATIO_AMT_PAID_OWED_360D,"(-0.0001, 0.9]]",17327,2280,15047,0.070433,0.131587,-0.545479,0.026329
2,IP_RATIO_AMT_PAID_OWED_360D,"(0.9, 0.9998]]",17326,2084,15242,0.070429,0.120282,-0.442716,0.016618
4,IP_RATIO_AMT_PAID_OWED_360D,"(0.9998, 1.0]]",122003,8859,113144,0.495931,0.072613,0.114745,0.006223
5,IP_RATIO_AMT_PAID_OWED_360D,"(1.0, 2.0]]",16607,939,15668,0.067506,0.056542,0.382078,0.008406
7,IP_RATIO_AMT_PAID_OWED_360D,SPECIAL_-99999,72745,5698,67047,0.295702,0.078328,0.032797,0.000314
Totals,IP_RATIO_AMT_PAID_OWED_360D,None,246008,19860,226148,1.000000,0.080729,NaN,0.058506


In [451]:
rows.append({
    'feature': 'IP_RATIO_AMT_PAID_OWED_360D',
    'n_bins': 4,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.295702,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.058506,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend contain -99999 ',
})


##### IP_RATIO_AMT_PAID_OWED_1080D

In [452]:
num_bins['IP_RATIO_AMT_PAID_OWED_1080D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_RATIO_AMT_PAID_OWED_1080D,"(-0.001, 0.817]",11152,1590,9562,0.045332,0.142575,-0.638419,0.024118
1,IP_RATIO_AMT_PAID_OWED_1080D,"(0.817, 0.896]",11152,1353,9799,0.045332,0.121324,-0.452526,0.011221
2,IP_RATIO_AMT_PAID_OWED_1080D,"(0.896, 0.936]",11152,1263,9889,0.045332,0.113253,-0.374549,0.007441
3,IP_RATIO_AMT_PAID_OWED_1080D,"(0.936, 0.965]",11152,1140,10012,0.045332,0.102224,-0.259726,0.003410
4,IP_RATIO_AMT_PAID_OWED_1080D,"(0.965, 0.988]",11152,1107,10045,0.045332,0.099265,-0.227061,0.002571
5,IP_RATIO_AMT_PAID_OWED_1080D,"(0.988, 1.0]",128869,9207,119662,0.523841,0.071445,0.132225,0.008666
6,IP_RATIO_AMT_PAID_OWED_1080D,"(1.0, 1.072]",4953,358,4595,0.020133,0.072279,0.119709,0.000274
7,IP_RATIO_AMT_PAID_OWED_1080D,"(1.072, 1.285]",11152,688,10464,0.045332,0.061693,0.289425,0.003365
8,IP_RATIO_AMT_PAID_OWED_1080D,"(1.285, 2.0]",11152,594,10558,0.045332,0.053264,0.445278,0.007470
9,IP_RATIO_AMT_PAID_OWED_1080D,SPECIAL_-99999,34122,2560,31562,0.138703,0.075025,0.079465,0.000847


In [453]:
merge_plan = [
    [0,1],
    [2,3,4],
    [5],
    [6,7,8]
   
]
woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_RATIO_AMT_PAID_OWED_1080D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [454]:
num_bins['IP_RATIO_AMT_PAID_OWED_1080D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_RATIO_AMT_PAID_OWED_1080D,"(-0.001, 0.896]]",22304,2943,19361,0.090664,0.131949,-0.548651,0.034332
2,IP_RATIO_AMT_PAID_OWED_1080D,"(0.896, 0.988]]",33456,3510,29946,0.135996,0.104914,-0.288702,0.012795
5,IP_RATIO_AMT_PAID_OWED_1080D,"(0.988, 1.0]]",128869,9207,119662,0.523841,0.071445,0.132225,0.008666
6,IP_RATIO_AMT_PAID_OWED_1080D,"(1.0, 2.0]]",27257,1640,25617,0.110797,0.060168,0.316078,0.009703
9,IP_RATIO_AMT_PAID_OWED_1080D,SPECIAL_-99999,34122,2560,31562,0.138703,0.075025,0.079465,0.000847
Totals,IP_RATIO_AMT_PAID_OWED_1080D,None,246008,19860,226148,1.000000,0.080729,NaN,0.069385


In [455]:
rows.append({
    'feature': 'IP_RATIO_AMT_PAID_OWED_1080D',
    'n_bins': 4,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.138703,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.069385,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend contain -99999 ',
})


##### PCB_HISTORY_LENGTH 

In [456]:
num_bins['PCB_HISTORY_LENGTH']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PCB_HISTORY_LENGTH,"(0.999, 11.0]",14996,1566,13430,0.060957,0.104428,-0.283516,0.005519
1,PCB_HISTORY_LENGTH,"(11.0, 14.0]",9466,898,8568,0.038478,0.094866,-0.176862,0.001296
2,PCB_HISTORY_LENGTH,"(14.0, 18.0]",13854,1459,12395,0.056315,0.105313,-0.292940,0.005465
3,PCB_HISTORY_LENGTH,"(18.0, 22.0]",12652,1293,11359,0.051429,0.102197,-0.259437,0.003860
4,PCB_HISTORY_LENGTH,"(22.0, 26.0]",12172,1185,10987,0.049478,0.097355,-0.205512,0.002278
5,PCB_HISTORY_LENGTH,"(26.0, 30.0]",10548,982,9566,0.042877,0.093098,-0.156103,0.001116
6,PCB_HISTORY_LENGTH,"(30.0, 36.0]",13756,1283,12473,0.055917,0.093268,-0.158117,0.001494
7,PCB_HISTORY_LENGTH,"(36.0, 42.0]",11164,880,10284,0.045381,0.078825,0.025941,0.000030
8,PCB_HISTORY_LENGTH,"(42.0, 48.0]",11205,876,10329,0.045547,0.078179,0.034863,0.000055
9,PCB_HISTORY_LENGTH,"(48.0, 54.0]",12185,959,11226,0.049531,0.078703,0.027615,0.000037


In [457]:
merge_plan = [
    [0,1,2,3],
    [4,5,6,7],
    [8,9,10],
    [11,12,13],
    [14,15],
    [16,17],
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PCB_HISTORY_LENGTH",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [458]:
num_bins['PCB_HISTORY_LENGTH']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PCB_HISTORY_LENGTH,"(0.999, 22.0]]",50968,5216,45752,0.207180,0.102339,-0.260977,0.015744
4,PCB_HISTORY_LENGTH,"(22.0, 42.0]]",47640,4330,43310,0.193652,0.090890,-0.129666,0.003438
8,PCB_HISTORY_LENGTH,"(42.0, 61.0]]",35498,2817,32681,0.144296,0.079357,0.018639,0.000050
11,PCB_HISTORY_LENGTH,"(61.0, 80.0]]",36993,2662,34331,0.150373,0.071960,0.124489,0.002212
14,PCB_HISTORY_LENGTH,"(80.0, 89.0]]",25208,1711,23497,0.102468,0.067875,0.187313,0.003324
16,PCB_HISTORY_LENGTH,"(89.0, 96.0]]",35249,2158,33091,0.143284,0.061222,0.297597,0.011209
18,PCB_HISTORY_LENGTH,SPECIAL_-99999,14452,966,13486,0.058746,0.066842,0.203762,0.002240
Totals,PCB_HISTORY_LENGTH,None,246008,19860,226148,1.000000,0.080729,NaN,0.040383


In [459]:
rows.append({
    'feature': 'PCB_HISTORY_LENGTH',
    'n_bins': 6,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.058746,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.040383,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend contain -99999 ',
})


##### CB_WT_CREDIT_UTIL_TREND_3M_12M

In [460]:
num_bins['CB_WT_CREDIT_UTIL_TREND_3M_12M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_WT_CREDIT_UTIL_TREND_3M_12M,"(-1.0119999999999998, -0.148]",2763,280,2483,0.011231,0.101339,-0.250049,0.000780
1,CB_WT_CREDIT_UTIL_TREND_3M_12M,"(-0.148, -0.051]",2762,255,2507,0.011227,0.092324,-0.146903,0.000258
2,CB_WT_CREDIT_UTIL_TREND_3M_12M,"(-0.051, -0.0106]",2762,271,2491,0.011227,0.098117,-0.214161,0.000563
3,CB_WT_CREDIT_UTIL_TREND_3M_12M,"(-0.0106, 0.0]",26423,1569,24854,0.107407,0.059380,0.330098,0.010200
4,CB_WT_CREDIT_UTIL_TREND_3M_12M,"(0.0, 0.00914]",1196,136,1060,0.004862,0.113712,-0.379113,0.000819
5,CB_WT_CREDIT_UTIL_TREND_3M_12M,"(0.00914, 0.0423]",2763,369,2394,0.011231,0.133550,-0.562558,0.004497
6,CB_WT_CREDIT_UTIL_TREND_3M_12M,"(0.0423, 0.0904]",2762,349,2413,0.011227,0.126358,-0.498928,0.003444
7,CB_WT_CREDIT_UTIL_TREND_3M_12M,"(0.0904, 0.149]",2762,408,2354,0.011227,0.147719,-0.679878,0.006890
8,CB_WT_CREDIT_UTIL_TREND_3M_12M,"(0.149, 0.222]",2761,399,2362,0.011223,0.144513,-0.654179,0.006310
9,CB_WT_CREDIT_UTIL_TREND_3M_12M,"(0.222, 0.327]",2763,393,2370,0.011231,0.142237,-0.635646,0.005917


In [461]:
merge_plan = [
    [0,1,2,3],
    [4,5,6,7,8,9,10]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_WT_CREDIT_UTIL_TREND_3M_12M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [462]:
num_bins['CB_WT_CREDIT_UTIL_TREND_3M_12M']


,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_WT_CREDIT_UTIL_TREND_3M_12M,"(-1.0119999999999998, 0.0]]",34710,2375,32335,0.141093,0.068424,0.178671,0.004180
4,CB_WT_CREDIT_UTIL_TREND_3M_12M,"(0.0, 1.679]]",17769,2446,15323,0.072229,0.137655,-0.597581,0.033109
11,CB_WT_CREDIT_UTIL_TREND_3M_12M,SPECIAL_-99999,193529,15039,178490,0.786678,0.077709,0.041404,0.001325
Totals,CB_WT_CREDIT_UTIL_TREND_3M_12M,None,246008,19860,226148,1.000000,0.080729,NaN,0.046865


In [463]:
rows.append({
    'feature': 'CB_WT_CREDIT_UTIL_TREND_3M_12M',
    'n_bins': 2,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.786678,      
    'bad_rate_trend': 'increasing',    
    'iv':0.046865,                    
    'keep': 'Yes',                     
    'comment': 'clear monotonic trend contain -99999 ',
})


##### CB_MAX_RATIO_POS_TO_TOTAL_DRAW_6M

In [464]:
num_bins['CB_MAX_RATIO_POS_TO_TOTAL_DRAW_6M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_6M,"(-0.001, 0.0702]",8968,1051,7917,0.036454,0.117194,-0.413212,0.007402
1,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_6M,"(0.0702, 0.364]",1281,236,1045,0.005207,0.184231,-0.944542,0.006860
2,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_6M,"(0.364, 0.786]",1281,188,1093,0.005207,0.146760,-0.672242,0.003115
3,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_6M,"(0.786, 1.0]",14093,1768,12325,0.057287,0.125452,-0.490701,0.016941
4,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_6M,SPECIAL_-99999,220385,16617,203768,0.895845,0.075400,0.074074,0.004765
Totals,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_6M,None,246008,19860,226148,1.000000,0.080729,NaN,0.039082


In [465]:
rows.append({
    'feature': 'CB_MAX_RATIO_POS_TO_TOTAL_DRAW_6M',
    'n_bins': 2,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.786678,      
    'bad_rate_trend': 'non monotonic',    
    'iv':0.046865,                    
    'keep': 'Drop',                     
    'comment': 'each bin cant have 5% data and dont have clear seapration of default with specoal values -99999 ',
})


##### CB_RATIO_MAX_POS_SPEND_12M

In [466]:
num_bins['CB_RATIO_MAX_POS_SPEND_12M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_RATIO_MAX_POS_SPEND_12M,"(-0.001, 0.0275]",39146,2920,36226,0.159125,0.074593,0.085711,0.001128
1,CB_RATIO_MAX_POS_SPEND_12M,"(0.0275, 0.0714]",3011,366,2645,0.012239,0.121554,-0.454689,0.003061
2,CB_RATIO_MAX_POS_SPEND_12M,"(0.0714, 0.141]",3012,373,2639,0.012244,0.123838,-0.475905,0.003385
3,CB_RATIO_MAX_POS_SPEND_12M,"(0.141, 0.248]",3011,343,2668,0.012239,0.113916,-0.381128,0.002086
4,CB_RATIO_MAX_POS_SPEND_12M,"(0.248, 0.413]",3011,393,2618,0.012239,0.130521,-0.536126,0.004403
5,CB_RATIO_MAX_POS_SPEND_12M,"(0.413, 0.682]",3011,387,2624,0.012239,0.128529,-0.518452,0.004087
6,CB_RATIO_MAX_POS_SPEND_12M,"(0.682, 8.521]",3012,409,2603,0.012244,0.135790,-0.581777,0.005285
7,CB_RATIO_MAX_POS_SPEND_12M,SPECIAL_-99999,188794,14669,174125,0.767430,0.077698,0.041555,0.001302
Totals,CB_RATIO_MAX_POS_SPEND_12M,None,246008,19860,226148,1.000000,0.080729,NaN,0.024737


In [467]:
merge_plan = [
    [1,2,3,4,5,6],
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_RATIO_MAX_POS_SPEND_12M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [468]:
num_bins['CB_RATIO_MAX_POS_SPEND_12M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_RATIO_MAX_POS_SPEND_12M,"(-0.001, 0.0275]",39146,2920,36226,0.159125,0.074593,0.085711,0.001128
1,CB_RATIO_MAX_POS_SPEND_12M,"(0.0275, 8.521]]",18068,2271,15797,0.073445,0.125692,-0.492882,0.021932
7,CB_RATIO_MAX_POS_SPEND_12M,SPECIAL_-99999,188794,14669,174125,0.767430,0.077698,0.041555,0.001302
Totals,CB_RATIO_MAX_POS_SPEND_12M,None,246008,19860,226148,1.000000,0.080729,NaN,0.024737


In [469]:
rows.append({
    'feature': 'CB_RATIO_MAX_POS_SPEND_12M',
    'n_bins': 2,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.767430,      
    'bad_rate_trend': 'increasing',    
    'iv':0.024737,                    
    'keep': 'Yes',                     
    'comment': 'clear trend increasing',
})


##### CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M

In [470]:
num_bins['CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M,"(-0.001, 0.113]",11636,1271,10365,0.047299,0.109230,-0.333851,0.006064
1,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M,"(0.113, 0.456]",1662,269,1393,0.006756,0.161853,-0.787978,0.005819
2,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M,"(0.456, 1.0]",19946,2393,17553,0.081079,0.119974,-0.439805,0.018857
3,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M,SPECIAL_-99999,212764,15927,196837,0.864866,0.074858,0.081878,0.005603
Totals,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M,None,246008,19860,226148,1.000000,0.080729,NaN,0.036344


In [471]:
merge_plan = [
    [0,1]
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [472]:
num_bins['CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M,"(-0.001, 0.456]]",13298,1540,11758,0.054055,0.115807,-0.399731,0.010213
2,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M,"(0.456, 1.0]",19946,2393,17553,0.081079,0.119974,-0.439805,0.018857
3,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M,SPECIAL_-99999,212764,15927,196837,0.864866,0.074858,0.081878,0.005603
Totals,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M,None,246008,19860,226148,1.000000,0.080729,NaN,0.036344


In [473]:
rows.append({
    'feature': 'CB_MAX_RATIO_POS_TO_TOTAL_DRAW_18M',
    'n_bins': 2,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.864866,      
    'bad_rate_trend': 'non monotonic',    
    'iv':0.036344,                    
    'keep': 'Drop',                     
    'comment': 'not clear sepration ',
})


##### CB_STD_PAYMENT_VOLATILITY_3M

In [474]:
rows.append({
    'feature': 'CB_STD_PAYMENT_VOLATILITY_3M',
    'n_bins': 2,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.864866,      
    'bad_rate_trend': 'non monotonic',    
    'iv':0.036344,                    
    'keep': 'Drop',                     
    'comment': 'not clear sepration ',
})


##### CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M

In [475]:
num_bins['CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,"(-0.001, 1.017]",1443,148,1295,0.005866,0.102564,-0.263428,0.000455
1,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,"(1.017, 1.07]",1443,245,1198,0.005866,0.169785,-0.845331,0.005950
2,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,"(1.07, 1.166]",1443,217,1226,0.005866,0.150381,-0.700867,0.003858
3,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,"(1.166, 1.331]",1443,217,1226,0.005866,0.150381,-0.700867,0.003858
4,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,"(1.331, 1.642]",1443,195,1248,0.005866,0.135135,-0.576184,0.002478
5,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,"(1.642, 2.0]",1592,202,1390,0.006471,0.126884,-0.503691,0.002027
6,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,"(2.0, 2.099]",1294,214,1080,0.005260,0.165379,-0.813742,0.004882
7,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,"(2.099, 2.336]",1443,199,1244,0.005866,0.137907,-0.599700,0.002710
8,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,"(2.336, 2.886]",1443,179,1264,0.005866,0.124047,-0.477831,0.001636
9,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,"(2.886, 3.75]",1443,187,1256,0.005866,0.129591,-0.527903,0.002039


In [476]:
merge_plan = [

    [0,1,2,3,4,5,6,7,8,9,10],           
    [11,12,13,14,15,16,17,18,19]             

]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [477]:
num_bins['CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,"(-0.001, 5.055]]",15873,2181,13692,0.064522,0.137403,-0.595454,0.029341
11,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,"(5.055, 2975900.0]]",12987,1375,11612,0.052791,0.105875,-0.298897,0.005347
20,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,SPECIAL_-99999,217148,16304,200844,0.882687,0.075082,0.078636,0.005281
Totals,CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M,None,246008,19860,226148,1.000000,0.080729,NaN,0.044067


In [478]:
rows.append({
    'feature': 'CB_MAX_RATIO_AMT_PAYMENT_MIN_INST_9M',
    'n_bins': 2,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.864866,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.044067,                    
    'keep': 'Yes',                     
    'comment': 'slight enforced clear sepration ',
})


##### CB_STD_PAYMENT_VOLATILITY_9M

In [479]:
num_bins['CB_STD_PAYMENT_VOLATILITY_9M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_STD_PAYMENT_VOLATILITY_9M,"(-0.001, 0.0115]",1385,177,1208,0.005630,0.127798,-0.511910,0.001828
1,CB_STD_PAYMENT_VOLATILITY_9M,"(0.0115, 0.0332]",1385,210,1175,0.005630,0.151625,-0.710566,0.003822
2,CB_STD_PAYMENT_VOLATILITY_9M,"(0.0332, 0.0753]",1384,212,1172,0.005626,0.153179,-0.722601,0.003969
3,CB_STD_PAYMENT_VOLATILITY_9M,"(0.0753, 0.158]",1385,176,1209,0.005630,0.127076,-0.505417,0.001777
4,CB_STD_PAYMENT_VOLATILITY_9M,"(0.158, 0.333]",1386,180,1206,0.005634,0.129870,-0.530374,0.001979
5,CB_STD_PAYMENT_VOLATILITY_9M,"(0.333, 0.474]",1383,176,1207,0.005622,0.127260,-0.507073,0.001787
6,CB_STD_PAYMENT_VOLATILITY_9M,"(0.474, 0.6]",1385,189,1196,0.005630,0.136462,-0.587491,0.002484
7,CB_STD_PAYMENT_VOLATILITY_9M,"(0.6, 0.737]",1384,197,1187,0.005626,0.142341,-0.636501,0.002973
8,CB_STD_PAYMENT_VOLATILITY_9M,"(0.737, 0.904]",1385,217,1168,0.005630,0.156679,-0.749331,0.004317
9,CB_STD_PAYMENT_VOLATILITY_9M,"(0.904, 1.176]",1385,195,1190,0.005630,0.140794,-0.623773,0.002842


In [480]:
merge_plan = [

    [0,1,2,3,4,5,6,7,8,9,10],           
    [11,12,13,14,15,16,17,18,19]             

]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_STD_PAYMENT_VOLATILITY_9M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [481]:
num_bins['CB_STD_PAYMENT_VOLATILITY_9M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_STD_PAYMENT_VOLATILITY_9M,"(-0.001, 1.718]]",15231,2087,13144,0.061913,0.137023,-0.592244,0.027814
11,CB_STD_PAYMENT_VOLATILITY_9M,"(1.718, 1214852.4]]",12462,1357,11105,0.050657,0.108891,-0.330363,0.006351
20,CB_STD_PAYMENT_VOLATILITY_9M,SPECIAL_-99999,218315,16416,201899,0.887430,0.075194,0.077029,0.005098
Totals,CB_STD_PAYMENT_VOLATILITY_9M,None,246008,19860,226148,1.000000,0.080729,NaN,0.041716


In [482]:
rows.append({
    'feature': 'CB_STD_PAYMENT_VOLATILITY_9M',
    'n_bins': 2,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.887430,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.041716,                    
    'keep': 'Yes',                     
    'comment': 'slight enforced clear sepration ',
})


##### CB_MAX_RATIO_PAYMENT_BALANCE_24M

In [483]:
num_bins['CB_MAX_RATIO_PAYMENT_BALANCE_24M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_PAYMENT_BALANCE_24M,"(-0.001, 0.0506]",1721,165,1556,0.006996,0.095874,-0.188554,0.000269
1,CB_MAX_RATIO_PAYMENT_BALANCE_24M,"(0.0506, 0.0575]",1720,331,1389,0.006992,0.192442,-0.998261,0.010506
2,CB_MAX_RATIO_PAYMENT_BALANCE_24M,"(0.0575, 0.0697]",1721,279,1442,0.006996,0.162115,-0.789907,0.006060
3,CB_MAX_RATIO_PAYMENT_BALANCE_24M,"(0.0697, 0.1]",1720,236,1484,0.006992,0.137209,-0.593817,0.003160
4,CB_MAX_RATIO_PAYMENT_BALANCE_24M,"(0.1, 0.111]",1721,291,1430,0.006996,0.169088,-0.840376,0.007000
5,CB_MAX_RATIO_PAYMENT_BALANCE_24M,"(0.111, 0.13]",1720,239,1481,0.006992,0.138953,-0.608473,0.003338
6,CB_MAX_RATIO_PAYMENT_BALANCE_24M,"(0.13, 0.177]",1720,212,1508,0.006992,0.123256,-0.470529,0.001885
7,CB_MAX_RATIO_PAYMENT_BALANCE_24M,"(0.177, 0.273]",1721,202,1519,0.006996,0.117374,-0.414942,0.001433
8,CB_MAX_RATIO_PAYMENT_BALANCE_24M,"(0.273, 0.488]",1720,195,1525,0.006992,0.113372,-0.375732,0.001156
9,CB_MAX_RATIO_PAYMENT_BALANCE_24M,"(0.488, 0.926]",1721,182,1539,0.006996,0.105752,-0.297601,0.000702


In [484]:
merge_plan = [

    [0,1,2,3,4,5,6,7,8,9,10],           
    [11,12,13,14,15,16,17,18,19]             

]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="CB_MAX_RATIO_PAYMENT_BALANCE_24M",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [485]:
num_bins['CB_MAX_RATIO_PAYMENT_BALANCE_24M']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CB_MAX_RATIO_PAYMENT_BALANCE_24M,"(-0.001, 1.686]]",18925,2527,16398,0.076928,0.133527,-0.562355,0.030778
11,CB_MAX_RATIO_PAYMENT_BALANCE_24M,"(1.686, 2538301.0]]",15484,1496,13988,0.062941,0.096616,-0.197077,0.002655
20,CB_MAX_RATIO_PAYMENT_BALANCE_24M,SPECIAL_-99999,211599,15837,195762,0.860131,0.074844,0.082069,0.005597
Totals,CB_MAX_RATIO_PAYMENT_BALANCE_24M,None,246008,19860,226148,1.000000,0.080729,NaN,0.045499


In [486]:
rows.append({
    'feature': 'CB_MAX_RATIO_PAYMENT_BALANCE_24M',
    'n_bins': 2,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.860131,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.045499,                    
    'keep': 'Yes',                     
    'comment': 'slight enforced clear sepration ',
})


##### IP_DPD_TREND

In [487]:
num_bins['IP_DPD_TREND']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_DPD_TREND,"(-273.001, -7.0]",3794,587,3207,0.015422,0.154718,-0.734416,0.011292
1,IP_DPD_TREND,"(-7.0, -5.0]",2394,314,2080,0.009731,0.131161,-0.541752,0.003583
2,IP_DPD_TREND,"(-5.0, -3.0]",4057,472,3585,0.016491,0.116342,-0.404947,0.003205
3,IP_DPD_TREND,"(-3.0, -2.0]",2917,310,2607,0.011857,0.106274,-0.303099,0.001237
4,IP_DPD_TREND,"(-2.0, -1.0]",4341,457,3884,0.017646,0.105275,-0.292545,0.001707
5,IP_DPD_TREND,"(-1.0, 0.0]",43350,4088,39262,0.176214,0.094302,-0.170281,0.005488
6,IP_DPD_TREND,SPECIAL_-66666,98168,7907,90261,0.399044,0.080546,0.002475,0.000002
7,IP_DPD_TREND,SPECIAL_-99999,86987,5725,81262,0.353594,0.065814,0.220354,0.015659
Totals,IP_DPD_TREND,None,246008,19860,226148,1.000000,0.080729,NaN,0.042174


In [488]:
0.399044+0.353594


0.752638

In [489]:
merge_plan = [
    [1,2,3,4,5]           
          
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_DPD_TREND",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [490]:
rows.append({
    'feature': 'IP_DPD_TREND',
    'n_bins': 2,                      
    'n_special_bins': 1 ,               
    'special_bins_pct':0.752638,      
    'bad_rate_trend': 'decreasing',    
    'iv':0.042174,                    
    'keep': 'Yes',                     
    'comment': ' clear sepration ',
})


In [491]:
num_bins['IP_DPD_TREND']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_DPD_TREND,"(-273.001, -7.0]",3794,587,3207,0.015422,0.154718,-0.734416,0.011292
1,IP_DPD_TREND,"(-7.0, 0.0]]",57059,5641,51418,0.231940,0.098863,-0.222555,0.012613
6,IP_DPD_TREND,SPECIAL_-66666,98168,7907,90261,0.399044,0.080546,0.002475,0.000002
7,IP_DPD_TREND,SPECIAL_-99999,86987,5725,81262,0.353594,0.065814,0.220354,0.015659
Totals,IP_DPD_TREND,None,246008,19860,226148,1.000000,0.080729,NaN,0.042174


##### PCB_LOAN_COMPLETION_RATE

In [492]:
num_bins['PCB_LOAN_COMPLETION_RATE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PCB_LOAN_COMPLETION_RATE,"(0.142, 0.5]",41494,3145,38349,0.168669,0.075794,0.068433,0.000768
1,PCB_LOAN_COMPLETION_RATE,"(0.5, 0.6]",4803,301,4502,0.019524,0.062669,0.272685,0.001296
2,PCB_LOAN_COMPLETION_RATE,"(0.6, 0.667]",22526,1595,20931,0.091566,0.070807,0.141876,0.001737
3,PCB_LOAN_COMPLETION_RATE,"(0.667, 0.75]",14708,943,13765,0.059787,0.064115,0.248336,0.003324
4,PCB_LOAN_COMPLETION_RATE,"(0.75, 0.8]",7674,495,7179,0.031194,0.064504,0.241876,0.001650
5,PCB_LOAN_COMPLETION_RATE,"(0.8, 0.875]",7113,464,6649,0.028914,0.065233,0.229855,0.001388
6,PCB_LOAN_COMPLETION_RATE,"(0.875, 1.0]",107543,9488,98055,0.437152,0.088225,-0.096981,0.004282
7,PCB_LOAN_COMPLETION_RATE,SPECIAL_-88888,25695,2463,23232,0.104448,0.095855,-0.188331,0.004009
8,PCB_LOAN_COMPLETION_RATE,SPECIAL_-99999,14452,966,13486,0.058746,0.066842,0.203762,0.002240
Totals,PCB_LOAN_COMPLETION_RATE,None,246008,19860,226148,1.000000,0.080729,NaN,0.020693


In [493]:
merge_plan = [
    [1,2],
    [3,4,5,6]
               
          
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="PCB_LOAN_COMPLETION_RATE",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [494]:
num_bins['PCB_LOAN_COMPLETION_RATE']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,PCB_LOAN_COMPLETION_RATE,"(0.142, 0.5]",41494,3145,38349,0.168669,0.075794,0.068433,0.000768
1,PCB_LOAN_COMPLETION_RATE,"(0.5, 0.667]]",27329,1896,25433,0.111090,0.069377,0.163819,0.002784
3,PCB_LOAN_COMPLETION_RATE,"(0.667, 1.0]]",137038,11390,125648,0.557047,0.083116,-0.031733,0.000568
7,PCB_LOAN_COMPLETION_RATE,SPECIAL_-88888,25695,2463,23232,0.104448,0.095855,-0.188331,0.004009
8,PCB_LOAN_COMPLETION_RATE,SPECIAL_-99999,14452,966,13486,0.058746,0.066842,0.203762,0.002240
Totals,PCB_LOAN_COMPLETION_RATE,None,246008,19860,226148,1.000000,0.080729,NaN,0.020693


In [495]:
rows.append({
    'feature': 'PCB_LOAN_COMPLETION_RATE',
    'n_bins': 3,                      
    'n_special_bins': 2 ,               
    'special_bins_pct':0.154448,      
    'bad_rate_trend': 'non monotonic feature',    
    'iv':0.020693,                    
    'keep': 'Drop',                     
    'comment': ' non monotonic features ',
})


##### IP_RATIO_LATE_PAYMENTS_360D

In [496]:
num_bins['IP_RATIO_LATE_PAYMENTS_360D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_RATIO_LATE_PAYMENTS_360D,"(-0.001, 0.0476]",130037,8914,121123,0.528589,0.068550,0.176701,0.015329
1,IP_RATIO_LATE_PAYMENTS_360D,"(0.0476, 0.0833]",9090,890,8200,0.036950,0.097910,-0.211814,0.001812
2,IP_RATIO_LATE_PAYMENTS_360D,"(0.0833, 0.133]",8374,1037,7337,0.034040,0.123836,-0.475884,0.009409
3,IP_RATIO_LATE_PAYMENTS_360D,"(0.133, 0.2]",9229,1099,8130,0.037515,0.119081,-0.431322,0.008362
4,IP_RATIO_LATE_PAYMENTS_360D,"(0.2, 0.324]",7874,988,6886,0.032007,0.125476,-0.490919,0.009474
5,IP_RATIO_LATE_PAYMENTS_360D,"(0.324, 1.0]",8659,1234,7425,0.035198,0.142511,-0.637890,0.018692
6,IP_RATIO_LATE_PAYMENTS_360D,SPECIAL_-99999,72745,5698,67047,0.295702,0.078328,0.032797,0.000314
Totals,IP_RATIO_LATE_PAYMENTS_360D,None,246008,19860,226148,1.000000,0.080729,NaN,0.063392


In [497]:
merge_plan = [
    [1,2,3],
    [4,5]
               
]

woe_mappings = run_merge_plan_and_collect_mappings(
    feature="IP_RATIO_LATE_PAYMENTS_360D",
    merge_plan=merge_plan,
    num_bins=num_bins
)

all_woe_mapping_num.update(woe_mappings)

In [498]:
num_bins['IP_RATIO_LATE_PAYMENTS_360D']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,IP_RATIO_LATE_PAYMENTS_360D,"(-0.001, 0.0476]",130037,8914,121123,0.528589,0.068550,0.176701,0.015329
1,IP_RATIO_LATE_PAYMENTS_360D,"(0.0476, 0.2]]",26693,3026,23667,0.108505,0.113363,-0.375642,0.017923
4,IP_RATIO_LATE_PAYMENTS_360D,"(0.2, 1.0]]",16533,2222,14311,0.067205,0.134398,-0.569861,0.027696
6,IP_RATIO_LATE_PAYMENTS_360D,SPECIAL_-99999,72745,5698,67047,0.295702,0.078328,0.032797,0.000314
Totals,IP_RATIO_LATE_PAYMENTS_360D,None,246008,19860,226148,1.000000,0.080729,NaN,0.063392


In [499]:
rows.append({
    'feature': 'IP_RATIO_LATE_PAYMENTS_360D',
    'n_bins': 3,                      
    'n_special_bins': 1,               
    'special_bins_pct':0.295702,      
    'bad_rate_trend': 'increasing',    
    'iv':0.063392,                    
    'keep': 'Yes',                     
    'comment': ' clear sepration ',
})

# LATER HANDLE IT

In [500]:
rows_df = pd.DataFrame(rows)

In [503]:
final_selected_features = ['EXT_SOURCE_WEIGHTED',
 'CREDIT_EXT_SOURCE_PRODUCT','ANNUITY_CREDIT_RATIO','YEARS_EMPLOYED','AMT_GOODS_PRICE','IP_WORST_DPD_720D','GOODS_CREDIT_RATIO','IP_RATIO_LATE_PAYMENTS_2160D',
 'B_NUM_ACTIVE_CREDIT_720D','ORG_GROUP','REGION_RATING_CLIENT_W_CITY','NAME_EDUCATION_TYPE','PA_RATIO_CREDIT_APPLICATION_Cash','PA_RATIO_CREDIT_APPLICATION_POS','CODE_GENDER',
 'PA_AVG_AMT_ANNUITY_POS','EXT_SOURCE_3','EXT_SOURCE_1','B_DEBT_TO_CREDIT_RATIO','PA_RATIO_APPROVED_LOANS','IP_NUM_COMPLETED_LOANS','PA_AVG_RISK_WEIGHT_1080D','OCCUPATION_GROUP','B_CREDIT_DURATION_MIN','CB_AVG_ATM_WITHDRAWAL_FREQ_6M','CB_WT_CREDIT_UTIL_TREND_3M_12M']

In [509]:
num_bins['CREDIT_EXT_SOURCE_PRODUCT']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,CREDIT_EXT_SOURCE_PRODUCT,"(4.25, 74921.044]]",24587,3564,21023,0.099944,0.144955,-0.657748,0.056892
2,CREDIT_EXT_SOURCE_PRODUCT,"(74921.044, 148179.082]]",49174,4881,44293,0.199888,0.099260,-0.227005,0.011330
6,CREDIT_EXT_SOURCE_PRODUCT,"(148179.082, 274041.892]]",61467,5516,55951,0.249858,0.089739,-0.115659,0.003509
11,CREDIT_EXT_SOURCE_PRODUCT,"(274041.892, 381604.556]]",36880,2526,34354,0.149914,0.068492,0.177599,0.004390
14,CREDIT_EXT_SOURCE_PRODUCT,"(381604.556, 556108.02]]",36880,1979,34901,0.149914,0.053661,0.437442,0.023920
17,CREDIT_EXT_SOURCE_PRODUCT,"(556108.02, 2995346.0]]",36881,1382,35499,0.149918,0.037472,0.813491,0.071087
20,CREDIT_EXT_SOURCE_PRODUCT,SPECIAL_-99999,139,12,127,0.000565,0.086331,-0.073202,0.000003
Totals,CREDIT_EXT_SOURCE_PRODUCT,None,246008,19860,226148,1.000000,0.080729,NaN,0.183983


In [508]:
rows_df[rows_df['feature'].isin(final_selected_features)][['feature','bad_rate_trend']]

,feature,bad_rate_trend
0,EXT_SOURCE_3,decreasing
1,EXT_SOURCE_1,decreasing
2,ANNUITY_CREDIT_RATIO,increasing
3,YEARS_EMPLOYED,decreasing
4,AMT_GOODS_PRICE,decreasing
5,IP_WORST_DPD_720D,increasing
8,GOODS_CREDIT_RATIO,decreasing
12,B_DEBT_TO_CREDIT_RATIO,increasing
13,B_NUM_ACTIVE_CREDIT_720D,increasing
16,REGION_RATING_CLIENT_W_CITY,increasing


In [ ]:
done = rows_df['feature'].unique().tolist()
rem = list(num_bins.keys())

In [ ]:
remaining_feature =[]
for feature in rem:
    if feature in done:
        pass
    else:
        remaining_feature.append(feature)

# SELECT FINAL FEATURES AND SAVE

In [ ]:
final_features = rows_df[rows_df['keep']=='Yes']['feature'].tolist()

In [ ]:
from  src.entity.artifact_entity import FeatureBinMergingArtifact
from src.constants.artifacts_paths import *
# SAVE FINAL FEATURE'S
ob = FeatureBinMergingArtifact()
feature_path = ob.selected_features_path 
feature_path.parent.mkdir(parents=True,exist_ok=True)
with open(feature_path, "w") as f:
    json.dump(final_features, f, indent=4)
print(feature_path)

D:\home loan credit risk\artifact\binning\postbin_manual\selected_features.json


In [ ]:
12ds

# APPLY NUMERICAL WOE MAPPING TRASFORMATION

In [ ]:
import pandas as pd


def apply_woe_remap(
    X_woe_df,
    woe_mapping,
    selected_features
) -> pd.DataFrame:
    """
    Remap old WoE values to merged WoE values.

    Parameters
    ----------
    X_woe_df : pd.DataFrame
        WoE-encoded dataframe (train / test / val)

    woe_mapping : dict
        {
            feature_name: {old_woe_value: new_woe_value}
        }
        
    selected_features : list
        Final selected features after feature selection
    Returns
    -------
    pd.DataFrame
        WoE-remapped dataframe
    """

    X_out = X_woe_df[selected_features].copy()
    
    # Only features that actually have WoE mappings
    woe_features = set(selected_features).intersection(woe_mapping.keys())

    for feature in woe_features:
        if feature in X_out.columns:
            X_out[feature] = X_out[feature].round(6)
            X_out[feature] = (
                X_out[feature]
                .replace(woe_mapping[feature])
                .astype(float)
            )
    return X_out

##### APPLY THE WOE MAPING TO TRAIN AND TEST

In [ ]:
X_woe_train = pd.read_csv(r'D:\home loan credit risk\artifact\binning\prebin\X_train_selected_woe.csv')
X_woe_test = pd.read_csv(r'D:\home loan credit risk\artifact\binning\prebin\X_test_selected_woe.csv')

In [ ]:
X_woe_train = apply_woe_remap(X_woe_train,all_woe_mapping_num,final_features)
X_woe_test = apply_woe_remap(X_woe_test,all_woe_mapping_num,final_features)

# CORR FILTRING

In [ ]:
temp_df = iv_df[iv_df['feature'].isin(final_features)].copy()
pd.set_option('display.max_columns',None)
from src.components.module_05_feature_engineering import FeatureSelector
fe = FeatureSelector()
rr = fe.correlation_filter(X_woe_train,final_features,temp_df)
fe.removed_corr_

2026-04-03 06:31:41,986 - module_05_feature_engineering.py - INFO - Total features before : 91
2026-04-03 06:31:42,029 - module_05_feature_engineering.py - INFO - Total features after  : 84
2026-04-03 06:31:42,030 - module_05_feature_engineering.py - INFO - Dropped features      : 7
2026-04-03 06:31:42,032 - module_05_feature_engineering.py - INFO - Removed 7 features due to high correlation (> 0.7)
2026-04-03 06:31:42,035 - module_05_feature_engineering.py - INFO - Remaining feature list final shape is  84


['CB_AVG_ATM_WITHDRAWAL_FREQ_6M',
 'CREDIT_GOODS_DIFF',
 'B_CREDIT_DURATION_MEAN',
 'BUREAU_DAYS_CREDIT_ENDDATE_MAX',
 'ANNUITY_TO_AGE_RATIO',
 'AMT_GOODS_PRICE',
 'B_AMT_CREDIT_SUM_DEBT_SUM']

# MULTICOLLINEARITY HANDLING

In [ ]:
'''import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor


def remove_multicollinearity_vif(
        X_woe,
        threshold=5.0
):
    """
    Remove multicollinear features using VIF.

    Parameters:
    -----------
    X_woe : DataFrame
        Dataset with WOE-transformed features

    threshold : float
        VIF threshold (default = 5)

    Returns:
    --------
    selected_features : list
    dropped_features : list
    """

    # Copy dataset
    X = X_woe.copy()

    # Remove constant columns
    X = X.loc[:, X.nunique() > 1]

    # Fill NaN (VIF cannot handle NaN)
    X = X.fillna(0)

    dropped_features = []

    while True:

        # Calculate VIF
        vif_data = pd.DataFrame()

        vif_data["feature"] = X.columns

        vif_data["VIF"] = [
            variance_inflation_factor(
                X.values,
                i
            )
            for i in range(X.shape[1])
        ]

        max_vif = vif_data["VIF"].max()

        if max_vif > threshold:

            feature_to_drop = vif_data.sort_values(
                "VIF",
                ascending=False
            )["feature"].iloc[0]

            print(
                f"Dropping {feature_to_drop} "
                f"(VIF={max_vif:.2f})"
            )

            X = X.drop(columns=[feature_to_drop])

            dropped_features.append(
                feature_to_drop
            )

        else:
            break

    selected_features = list(X.columns)

    return selected_features, dropped_features'''

'import pandas as pd\nimport numpy as np\nfrom statsmodels.stats.outliers_influence import variance_inflation_factor\n\n\ndef remove_multicollinearity_vif(\n        X_woe,\n        threshold=5.0\n):\n    """\n    Remove multicollinear features using VIF.\n\n    Parameters:\n    -----------\n    X_woe : DataFrame\n        Dataset with WOE-transformed features\n\n    threshold : float\n        VIF threshold (default = 5)\n\n    Returns:\n    --------\n    selected_features : list\n    dropped_features : list\n    """\n\n    # Copy dataset\n    X = X_woe.copy()\n\n    # Remove constant columns\n    X = X.loc[:, X.nunique() > 1]\n\n    # Fill NaN (VIF cannot handle NaN)\n    X = X.fillna(0)\n\n    dropped_features = []\n\n    while True:\n\n        # Calculate VIF\n        vif_data = pd.DataFrame()\n\n        vif_data["feature"] = X.columns\n\n        vif_data["VIF"] = [\n            variance_inflation_factor(\n                X.values,\n                i\n            )\n            for i 

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor


def remove_multicollinearity_vif(
    X_woe: pd.DataFrame,
    iv_df: pd.DataFrame,
    threshold: float = 5.0,
) -> tuple[list[str], list[str]]:
    """
    Remove multicollinear features using VIF, dropping lowest-IV violator each round.

    Parameters
    ----------
    X_woe     : WOE-transformed feature matrix
    iv_df     : DataFrame with columns ['feature', 'IV']
    threshold : VIF threshold (default 5.0)

    Returns
    -------
    selected_features : list[str]
    dropped_features  : list[str]
    """
    X = X_woe.loc[:, X_woe.nunique() > 1].fillna(0).copy()

    iv = iv_df.set_index("feature")["IV"].to_dict()
    dropped = []

    while X.shape[1] > 1:
        mat = X.to_numpy()

        vif = np.array([variance_inflation_factor(mat, i) for i in range(mat.shape[1])])

        if vif.max() <= threshold:
            break

        # Among violators → drop the one with lowest IV
        mask      = vif > threshold
        violators = X.columns[mask]
        drop_col  = min(violators, key=lambda c: iv.get(c, 0.0))

        print(f"Dropping '{drop_col}' (VIF={vif[X.columns.get_loc(drop_col)]:.2f}, IV={iv.get(drop_col, 0.0):.4f})")

        X = X.drop(columns=drop_col)
        dropped.append(drop_col)

    return X.columns.tolist(), dropped

In [ ]:
final_features, dropped_features = remove_multicollinearity_vif(
    X_woe_train,
    iv_df,
    threshold=5
)

print("Selected Features:", len(final_features))
print("Dropped Features:", dropped_features)

Dropping 'BASEMENTAREA_AVG' (VIF=7.27, IV=0.0221)
Dropping 'NONLIVINGAREA_AVG' (VIF=28.09, IV=0.0226)
Dropping 'PA_AVG_AMT_ANNUITY_CARDS' (VIF=14.16, IV=0.0243)
Dropping 'B_MAX_REPAYMENT_DAYS_DIFF' (VIF=9.61, IV=0.0395)
Dropping 'B_TOTAL_ANNUITY_TO_DEBT' (VIF=5.81, IV=0.0473)
Dropping 'PA_RATIO_HC_REFUSED_LOANS' (VIF=110.06, IV=0.0494)


In [ ]:
# after multicollinearaty featutres df
X_woe_train = X_woe_train[final_features]
X_woe_test = X_woe_test[final_features]

# PSI

In [ ]:
import numpy as np
import pandas as pd

def psi_check(X_train_woe, X_test_woe, psi_threshold=0.25, eps=1e-6):
    """
    Perform Population Stability Index (PSI) check between
    training and testing datasets for WoE-transformed features.

    The Home Credit Kaggle dataset does not contain time stamps
    so out-of-time (OOT) sample cannot be constructed,
    and population stability analysis (PSI) is not meaningful in a temporal sense.
    there for the testing data is used as the to check psi
    (it will not remove any feature because them have the same distributions)
    but it is implemented for credit risk standard pipline purpose

    """

    kept_features = []
    removed_features = []

    for feature in X_train_woe.columns:
        # Distribution
        train_dist = X_train_woe[feature].value_counts(normalize=True)
        test_dist  = X_test_woe[feature].value_counts(normalize=True)

        # Align bins
        all_bins = train_dist.index.union(test_dist.index)
        train_dist = train_dist.reindex(all_bins, fill_value=0) + eps
        test_dist  = test_dist.reindex(all_bins, fill_value=0) + eps

        # PSI formula
        psi = np.sum((train_dist - test_dist) * np.log(train_dist / test_dist))

        # Decision
        if psi > psi_threshold:
            print(f" REMOVE | {feature} | PSI = {psi:.4f}")
            removed_features.append(feature)
        else:
            print(f" KEEP   | {feature} | PSI = {psi:.4f}")
            kept_features.append(feature)

    return kept_features, removed_features

In [ ]:
'''
“The Home Credit Kaggle dataset does not contain temporal information.
As a result, a true out-of-time (OOT) sample could not be constructed, and population stability analysis (PSI) is not meaningful in a temporal sense.
Therefore, PSI results are reported for completeness only and were not used for feature elimination or stability claims.”
'''

'\n“The Home Credit Kaggle dataset does not contain temporal information.\nAs a result, a true out-of-time (OOT) sample could not be constructed, and population stability analysis (PSI) is not meaningful in a temporal sense.\nTherefore, PSI results are reported for completeness only and were not used for feature elimination or stability claims.”\n'

In [ ]:
kept_features, removed_features = psi_check_print_only(
    X_woe_train,
    X_woe_test,
    psi_threshold=0.25
)


📊 PSI STABILITY CHECK
--------------------------------------------------
✅ KEEP   | EXT_SOURCE_3 | PSI = 0.0001
✅ KEEP   | EXT_SOURCE_2 | PSI = 0.0000
✅ KEEP   | EXT_SOURCE_1 | PSI = 0.0002
✅ KEEP   | ANNUITY_CREDIT_RATIO | PSI = 0.0001
✅ KEEP   | YEARS_EMPLOYED | PSI = 0.0001
✅ KEEP   | AMT_GOODS_PRICE | PSI = 0.0001
✅ KEEP   | IP_WORST_DPD_720D | PSI = 0.0001
✅ KEEP   | YEARS_AGE | PSI = 0.0002
✅ KEEP   | GOODS_CREDIT_RATIO | PSI = 0.0000
✅ KEEP   | IP_RATIO_EARLY_PAYMENTS_1080D | PSI = 0.0000
✅ KEEP   | IP_WORST_DPD_270D | PSI = 0.0001
✅ KEEP   | B_RATIO_DEBT_TO_LOAN | PSI = 0.0001
✅ KEEP   | B_NUM_ACTIVE_CREDIT_720D | PSI = 0.0001
✅ KEEP   | IP_AVG_RATIO_PAYMENT_INSTALMENT | PSI = 0.0000
✅ KEEP   | AMT_CREDIT | PSI = 0.0000
✅ KEEP   | IP_WORST_DPD_2160D | PSI = 0.0000
✅ KEEP   | REGION_RATING_CLIENT_W_CITY | PSI = 0.0000
✅ KEEP   | IP_NUM_UNDERPAID_INSTALLMENTS_720D | PSI = 0.0000
✅ KEEP   | IP_MEAN_INSTALMENT_PAYMENT_DIFF | PSI = 0.0001
✅ KEEP   | B_NUM_ACTIVE_CREDIT_270D | PSI =